# Main comparison on GTSRB (natural imbalance), 10 clients, Dirichlet 0.5 (part 3 of 3)

**What this notebook does:** Main comparison on GTSRB (natural imbalance), 10 clients, Dirichlet 0.5: FedECS-LT against 11 published methods and the Local lower bound, seeds 3-5. Part 3 of 3: FedGH, LG-FedAvg, FedMRL; seeds 3, 4, 5.

**Runs:** 9. **Expected time:** about 7.4 h (Kaggle's limit is 12 h; no new run starts after 11 h, and finished runs are always saved).

## Before you start (once)
1. Right panel → **Settings** → **Accelerator** → **GPU T4 x2**.
2. **Settings** → **Internet** → **On**.
3. Do not edit any cell (above all the run cell's `--gpus 2 --per_gpu 2`: one edited copy ran on one worker of four and took 9.9 h).

## How to run
**Save Version** → **Save & Run All (Commit)** → **Save**. You can close the tab.

## What to send back
When the version shows a green tick: open it → **Output** tab → download the files:
- `NB10_main_gtsrb_part3of3__summary.zip` (small, a few MB: every metric and log; **send this one first**)
- `NB10_main_gtsrb_part3of3__report.pdf` (everything in one readable document)
- `NB10_main_gtsrb_part3of3__results.zip` (every image, metric, log and prediction; keep it)
- `NB10_main_gtsrb_part3of3__models.zip` (trained weights; keep it)

If the run stops with a red **STOP** message, it tells you which setting to change; change it and run again.

In [ ]:
# Environment check
import os, sys, shutil, urllib.request
import torch
if sys.version_info < (3, 11):
    raise RuntimeError(f'STOP: Kaggle Python {sys.version} is older than 3.11, which Flower 1.42 needs. '
                       'Settings -> Environment -> Always use latest environment, then run again.')
n = torch.cuda.device_count(); names = [torch.cuda.get_device_name(i) for i in range(n)]
print('GPUs:', n, names)
if n < 2 or not all('T4' in x for x in names):
    raise RuntimeError("STOP: set Settings -> Accelerator -> 'GPU T4 x2', then run again.")
try:
    urllib.request.urlopen('https://huggingface.co', timeout=20); print('Internet: on')
except Exception as e:
    raise RuntimeError('STOP: set Settings -> Internet -> On, then run again. (' + str(e) + ')')
free = shutil.disk_usage('/tmp').free / 2**30
print(f'Free disk in /tmp: {free:.0f} GB')
print('OK - environment is correct.')

In [ ]:
# Packages (fixed versions)
!pip install -q "flwr[simulation]==1.42.0" "reportlab==5.0.1" 2>&1 | tail -2
import flwr, ray; print('Flower', flwr.__version__, 'Ray', ray.__version__)
# UMAP (umap-learn, BSD-3-Clause, downloaded at run time) for the embedding maps: the release that
# matches this image's scikit-learn (0.5.9.post2 needs >= 1.6; 0.5.7 needs < 1.8), which is kept
import subprocess, sys, sklearn
_v = tuple(int(x) for x in sklearn.__version__.split('.')[:2])
_u = 'umap-learn==0.5.9.post2' if _v >= (1, 6) else 'umap-learn==0.5.7'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _u, f'scikit-learn=={sklearn.__version__}'], check=False)
try:
    import umap; print('UMAP', umap.__version__, 'scikit-learn', sklearn.__version__)
except Exception as e:
    print('UMAP not available (its figure will be listed as an error):', e)

## Helper scripts
The cells below write the suite's code to disk (hidden; nothing to change).

In [ ]:
import os
os.makedirs('/kaggle/working/fedecs_lt', exist_ok=True)

In [ ]:
%%writefile /kaggle/working/fedecs_lt/acceptance.py
"""Acceptance check of the Flower implementation against the earlier experiments (first notebook, NB00).

The new code cannot reproduce the old runs bit for bit: the old pipeline ran all clients in one process with one random
stream, the Flower simulation seeds every client action separately. It must reproduce what does not depend on that
stream exactly, and everything else within seed-to-seed variation:
  1. client splits: every client's class counts equal those logged by the old runs (exact);
  2. balanced accuracy as trained and with the exact readout: within max(2 points, 3 x the old seed SD) of the old
     run with the same configuration and seed 3;
  3. class separation F of the frozen features: within 25% of the old value (it depends on the trained extractor);
  4. the gate chooses the same generator branch as before (MNIST and CIFAR-10: the mean anchor).
Reference values: experiments/results (K17 GeFL-F and FedECS-LT; K14 for the final configuration without kernel
herding; K21 for the client counts). The CIFAR-10 FedECS-LT reference still had kernel herding (removing it costs
0.13-0.37 points with the readout), so its tolerance is 3 points.

    python acceptance.py RESULTS_DIR   ->  prints the verdict, writes RESULTS_DIR/acceptance.json
"""
import json
import os
import sys

import numpy as np

REF = {
    "mnist|GeFL-F": dict(final_bal={3: 0.7495, 4: 0.6941, 5: 0.7185}, er_bal={3: 0.9036, 4: 0.9137, 5: 0.9137}, F={3: 0.3255, 4: 0.3142, 5: 0.2911}, time=dict(fe=27, gen=483, head=33)),
    "mnist|-KH": dict(final_bal={3: 0.9451, 4: 0.9453}, er_bal={3: 0.9382, 4: 0.9427}, F={3: 0.3135, 4: 0.3108}, time=dict(fe=25, gen=487, head=100)),
    "mnist|Ours Ts=10": dict(final_bal={3: 0.9512, 4: 0.9488, 5: 0.9546}, er_bal={3: 0.9423, 4: 0.9452, 5: 0.9503}, F={3: 0.3165, 4: 0.3198, 5: 0.29}, time=dict(fe=25, gen=480, head=529)),
    "cifar10|GeFL-F": dict(final_bal={3: 0.3296, 4: 0.3452, 5: 0.3343}, er_bal={3: 0.5027, 4: 0.517, 5: 0.5018}, F={3: 0.0769, 4: 0.0596, 5: 0.0703}, time=dict(fe=128, gen=2970, head=173)),
    "cifar10|Ours Ts=10": dict(final_bal={3: 0.4234, 4: 0.4414, 5: 0.4486}, er_bal={3: 0.5235, 4: 0.5327, 5: 0.5349}, F={3: 0.0764, 4: 0.0584, 5: 0.0714}, time=dict(fe=132, gen=2974, head=2360)),
}
COUNTS = {"mnist": [[134, 604, 6, 0, 30, 101, 5, 13, 2, 0], [349, 5, 433, 15, 9, 7, 3, 0, 9, 2], [1257, 4, 34, 0, 109, 36, 18, 10, 16, 0], [114, 5, 6, 3, 1, 0, 9, 8, 0, 1], [143, 33, 42, 17, 80, 8, 58, 21, 1, 0], [85, 85, 191, 54, 6, 7, 1, 1, 0, 3], [141, 365, 129, 93, 10, 17, 1, 3, 7, 2], [170, 65, 7, 1, 14, 0, 5, 3, 3, 2], [22, 228, 3, 11, 0, 5, 6, 2, 0, 2], [2, 55, 17, 326, 53, 6, 6, 6, 2, 12]], "cifar10": [[18, 398, 7, 91, 75, 137, 70, 72, 41, 4], [148, 26, 1077, 1405, 1, 47, 0, 5, 7, 0], [385, 23, 81, 42, 250, 72, 17, 0, 1, 1], [67, 1, 0, 199, 379, 197, 21, 14, 3, 6], [1151, 147, 20, 21, 155, 3, 121, 17, 2, 0], [123, 236, 323, 30, 45, 126, 11, 13, 91, 2], [172, 2803, 1107, 71, 256, 5, 1, 57, 3, 63], [1740, 54, 106, 42, 136, 5, 22, 76, 1, 9], [765, 23, 567, 78, 0, 22, 165, 8, 0, 14], [231, 1089, 332, 191, 4, 165, 39, 18, 19, 1]]}
CHECKS = [  # (dataset, new label, reference key, minimum tolerance)
    ("mnist", "GeFL-F", "mnist|GeFL-F", 0.02),
    ("mnist", "FedECS-LT", "mnist|-KH", 0.02),
    ("cifar10", "GeFL-F", "cifar10|GeFL-F", 0.02),
    ("cifar10", "FedECS-LT", "cifar10|Ours Ts=10", 0.03),
]


def check(res_dir):
    runs = [json.loads(line) for line in open(os.path.join(res_dir, "runs.jsonl")) if line.strip()]
    rows, ok_all = [], True
    for ds, lab, ref_key, tol_min in CHECKS:
        r = next((x for x in runs if x["dataset"] == ds and x["label"] == lab and x["seed"] == 3 and x["K"] == 10
                  and x["IF"] == 0.01 and x["alpha"] == 0.5), None)
        ref = REF[ref_key]
        if r is None:
            rows.append(dict(check=f"{ds} {lab}", verdict="MISSING"))
            ok_all = False
            continue
        out = dict(check=f"{ds} {lab}")
        out["counts_identical"] = bool(np.array_equal(np.array(r["client_counts"]), np.array(COUNTS[ds])))
        for key in ("final_bal", "er_bal"):
            olds = np.array(list(ref[key].values()), dtype=float)
            tol = max(tol_min, 3 * olds.std(ddof=1)) if len(olds) > 1 else tol_min
            new = r.get(key)
            out[key] = dict(new=new, old_seed3=ref[key][3], tolerance=round(float(tol), 4),
                            ok=bool(new is not None and abs(new - ref[key][3]) <= tol))
        if r.get("class_sep_F") is not None:
            out["F"] = dict(new=r["class_sep_F"], old_seed3=ref["F"][3],
                            ok=bool(abs(r["class_sep_F"] - ref["F"][3]) <= 0.25 * ref["F"][3]))
            out["gate_branch"] = dict(new=r.get("gate_branch"), expected="anchor", ok=r.get("gate_branch") == "anchor")
        out["time_s"] = dict(new=dict(fe=r.get("fe_time_s"), gen=r.get("gen_time_s"), total=r.get("time_s")),
                             old=ref["time"])
        ok = bool(out["counts_identical"]) and all(v["ok"] for k, v in out.items() if isinstance(v, dict) and "ok" in v)
        out["verdict"] = "PASS" if ok else "FAIL"
        ok_all &= ok
        rows.append(out)
    verdict = dict(overall="PASS" if ok_all else "FAIL", checks=rows)
    # written to a temporary file first, so a crash never leaves a half-written acceptance.json behind
    path = os.path.join(res_dir, "acceptance.json")
    with open(path + ".tmp", "w") as f:
        json.dump(verdict, f, indent=1, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    os.replace(path + ".tmp", path)
    return verdict


if __name__ == "__main__":
    v = check(sys.argv[1])
    for row in v["checks"]:
        print(row["verdict"], row["check"], {k: row[k] for k in row if k not in ("check", "verdict", "time_s")})
    print()
    print("ACCEPTANCE:", v["overall"])


In [ ]:
%%writefile /kaggle/working/fedecs_lt/audit.py
"""Logging audit of one experiment's results (run before the report; its findings are printed on the report's cover
and listed in an appendix, and written to results/audit.json).

For every run: every per-class metric (metrics.PER_CLASS) for every class on the test set, on the validation set
(methods that evaluate it) and with the exact readout (the GeFL-F pipeline family); the class order and names; the
run time. For every per-round evaluation in the run's log: every per-class metric for every class, for the clients'
models and for the aggregated (or global) model. For every figure: its PDF, PNG and CSV.

    python audit.py RESULTS_DIR
"""
import glob
import json
import os
import sys

from metrics import PER_CLASS

ROUND_EVENTS = ("stage1_round", "stage3_round", "round_eval")


def _vec_ok(v, C):
    return isinstance(v, list) and len(v) == C and any(x is not None for x in v)


def _logs(res_dir):
    out = {}
    for f in glob.glob(os.path.join(res_dir, "logs", "*.jsonl")):
        recs = []
        for line in open(f):
            try:
                recs.append(json.loads(line))
            except json.JSONDecodeError:
                pass
        if recs and recs[0].get("event") == "run_start":
            out[recs[0]["run_id"]] = recs
    return out


def audit(res_dir, figures=True):
    problems = []
    path = os.path.join(res_dir, "runs.jsonl")
    runs = [json.loads(line) for line in open(path) if line.strip()] if os.path.exists(path) else []
    logs = _logs(res_dir)
    checked = dict(runs=len(runs), round_evaluations=0, figures=0)
    for r in runs:
        rid, pipe = r["run_id"], (r.get("method") or {}).get("pipeline")
        C = len(r.get("final_recall_c") or r.get("class_order") or [])
        if not C:
            problems.append(f"{rid}: no per-class results at all")
            continue
        need = [("final", PER_CLASS)]
        if pipe != "longtaillib":
            need.append(("val_final", PER_CLASS))
        if pipe == "gefl" and (r.get("method") or {}).get("readout", True):
            need.append(("er", PER_CLASS))
        for prefix, keys in need:
            miss = [k for k in keys if not _vec_ok(r.get(f"{prefix}_{k}"), C)]
            if miss:
                problems.append(f"{rid}: {prefix} per-class metrics missing: {', '.join(miss)}")
        if len(r.get("class_order") or []) != C:
            problems.append(f"{rid}: class order not recorded")
        if r.get("class_names") is not None and len(r["class_names"]) != C:
            problems.append(f"{rid}: class names do not match the {C} classes")
        if not r.get("time_s"):
            problems.append(f"{rid}: run time not recorded")
        if pipe == "local":
            continue            # no rounds: each client trains alone, evaluated once at the end
        ev = [e for e in logs.get(rid, []) if e.get("event") in ROUND_EVENTS and (e.get("post") or e.get("pre"))]
        if not ev:
            problems.append(f"{rid}: no per-round evaluations in the log")
            continue
        for e in ev:
            checked["round_evaluations"] += 1
            post = e.get("post") or {}
            miss = [k for k in PER_CLASS if not _vec_ok(post.get(k), C)]
            if miss:
                problems.append(f"{rid}: {e['event']} {e.get('round')}: aggregated/global model lacks {', '.join(miss)}")
            for cid, pre in (e.get("pre") or {}).items():
                miss = [k for k in PER_CLASS if not _vec_ok(pre.get(k), C)]
                if miss:
                    problems.append(f"{rid}: {e['event']} {e.get('round')}: client {cid} lacks {', '.join(miss)}")
                    break
    for pdf in (glob.glob(os.path.join(res_dir, "figures", "**", "*.pdf"), recursive=True) if figures else []):
        checked["figures"] += 1
        for ext in (".png", ".csv"):
            if not os.path.exists(pdf[:-4] + ext):
                problems.append(f"figure {os.path.relpath(pdf, res_dir)}: no {ext} file")
    # one line per kind of problem and run is enough to act on; keep at most 200
    out = dict(checked=checked, problems=problems[:200], n_problems=len(problems))
    with open(os.path.join(res_dir, "audit.json"), "w") as f:
        json.dump(out, f, indent=1)
    return out


if __name__ == "__main__":
    a = audit(sys.argv[1])
    print(json.dumps(a["checked"]), f"{a['n_problems']} problems")
    for p in a["problems"]:
        print(" -", p)


In [ ]:
%%writefile /kaggle/working/fedecs_lt/baselines.py
"""Registry of the baselines that have their own protocol (not the GeFL-F pipeline family of runner.PIPELINE).

Track 1 (different client models; frozen Stage-I extractor shared with FedECS-LT): the model-heterogeneous FL methods
ported from HtFLlib (Apache License 2.0), run in Flower by htfl_server.py / htfl_client.py. Track 2 (one shared model,
the FedLTLib benchmark) runs the authors' LongTailLib code unchanged on this suite's splits (track2.py)."""

HTFL = ["FedProto", "FedTGP", "FedGen", "FedGH", "LG-FedAvg", "FedMRL"]
SOURCE = {m: "HtFLlib (Zhang et al., KDD 2025), Apache-2.0, ported to Flower" for m in HTFL}
BASELINES = {m: dict(pipeline="htfl", name=m, source=SOURCE[m]) for m in HTFL}
TRACK2 = ["FedAvg", "FedProx", "MOON", "FedLC", "CCVR", "CReFF", "FedETF", "Fed-GraB", "FedLoGe", "FedYoYo", "RUCR",
          "CLIP2FL"]
for m in TRACK2:
    BASELINES[f"{m} (LongTailLib)"] = dict(pipeline="longtaillib", name=m,
                                          source="LongTailLib (FedLTLib benchmark), authors' code unchanged, "
                                                 "downloaded at run time")


def run_baseline(grid, context, R):
    if R["method"]["pipeline"] == "htfl":
        from htfl_server import run_htfl
        return run_htfl(grid, context, R)
    raise ValueError(f"unknown baseline pipeline {R['method']['pipeline']}")


In [ ]:
%%writefile /kaggle/working/fedecs_lt/captions.py
"""Formal figure captions (IEEE style), built from the experiment's settings. A caption says what is plotted, on
which data and setting, over which seeds, and what the error bars mean; it never names notebooks, run numbers or
internal file names (those are listed in the report's appendix)."""
import re

from settings import CITE, DATASETS

# With one seed there is no spread over seeds: these phrases are rewritten (in this order) in every caption
ONE_SEED = [
    (r"; mean ± standard deviation over seeds", ""),
    (r"[,;] mean over seeds \(band: one standard deviation\)", ""),
    (r"[,;] mean over seeds[,;] (error bars|the shaded band is) one standard deviation", ""),
    (r"mean over them and over seeds", "mean over them"),
    (r"(averaged|mean) over architectures and seeds", r"\1 over architectures"),
    (r", summed over seeds", ""),
    (r" \(all seeds\)", ""),
    (r" \(mean over seeds\)", ""),
    (r"[,;] mean over seeds", ""),
]


# Track 2 has one model shared by all clients (one architecture): the stage figures' wording for it
TRACK2_WORDS = [
    (r"aggregated heads", "aggregated head"),
    (r"the clients' local heads", "the clients' local copies of the head"),
    (r"minus local heads", "minus local copies"),
    (r"the architecture heads", "its head"),
    (r"the clients' heads", "the clients' copies of the head"),
    (r"of the heads as trained", "of the head as trained"),
    (r"[,;] (averaged|mean) over architectures", ""),
]


def track2_words(text, S):
    if DATASETS.get(S[0], {}).get("track") != 2:
        return text
    for pat, rep in TRACK2_WORDS:
        text = re.sub(pat, rep, text)
    return text


def one_seed(text, seeds):
    """The caption text for the number of seeds it covers (unchanged for two or more)."""
    if seeds is None or len(seeds) != 1:
        return text
    for pat, rep in ONE_SEED:
        text = re.sub(pat, rep, text)
    return text


# Names as printed in reports and figures. Inside the suite the baselines of the one-shared-model setting carry the
# suffix " (LongTailLib)" (it selects their code, and pairs them with earlier runs); the reports print the method's
# name only, and the methods table names the code each method ran in.
DISPLAY = {"FedECS-LT (extractor 200 rounds)": "FedECS-LT (200 extractor rounds)"}
LT_SUFFIX = " (LongTailLib)"


# Messages of the federation, as named in the communication figure and table
MESSAGES = {
    "train.stage1": "Extractor training (Stage I)", "train.gen": "Generator training (Stage II)",
    "train.head": "Head training (Stage III)", "train.htfl": "Training rounds (method's own exchange)",
    "train.local": "Local training", "query.features": "Extractor broadcast", "query.heads": "Head broadcast",
    "query.load_gen": "Generator broadcast", "secsum.stats": "Secure sums: class statistics",
    "secsum.rows": "Secure sums: generator class rows", "secsum.consensus": "Secure sums: consensus soft labels",
    "secsum.readout": "Secure sums: exact readout", "sums.stats": "Sums: class statistics",
    "sums.gram": "Sums: class second moments", "sums.rows": "Sums: generator class rows",
    "sums.consensus": "Sums: consensus soft labels", "sums.readout": "Sums: exact readout",
}


def message_name(t):
    return MESSAGES.get(t, t)


def display(label):
    if label in DISPLAY:
        return DISPLAY[label]
    return label[:-len(LT_SUFFIX)] if label.endswith(LT_SUFFIX) else label


def mname(m):
    """Method name as written in the text (with its citation where it has one)."""
    base = m.split(" (")[0].split(" + ")[0]
    if m in CITE:
        return f"{m} [{CITE[m]}]"
    if base in CITE and m.startswith(base):
        return m
    return m


def setting_text(S, seeds=None):
    ds, IF, alpha, K, pfrac, tail_perm = S
    spec = DATASETS.get(ds, dict(name=ds, imbalance="IF"))
    if spec["imbalance"] == "natural":
        data = f"{spec['name']} (natural class imbalance)"
    elif IF is None or IF >= 1.0:
        data = f"{spec['name']} (balanced)"
    else:
        data = f"{spec['name']}-LT (imbalance ratio {int(round(1 / IF))})"
    part = "IID split" if alpha is None else f"Dirichlet label split with alpha = {alpha}"
    txt = f"{data}, {K} clients, {part}"
    if spec.get("track") == 2:
        txt += ", one ResNet-8 shared by all clients (standard CIFAR-LT class profile)"
    if pfrac < 1.0:
        txt += f", {int(round(100 * pfrac))}% of the clients per round"
    if tail_perm:
        txt += ", rare classes chosen at random"
    if seeds:
        s = sorted(seeds)
        txt += ("; seeds " + ", ".join(str(x) for x in s[:-1]) + f" and {s[-1]}") if len(s) > 1 else f"; seed {s[0]}"
    return txt + "."


def _seeds(E, S):
    return sorted({r["seed"] for r in E.runs if (r["dataset"], r["IF"], r["alpha"], r["K"], r.get("pfrac", 1.0),
                                                bool(r.get("tail_perm", False))) == S})


def simple(num, E, S, text):
    seeds = _seeds(E, S)
    return track2_words(one_seed(f"{text} {setting_text(S, seeds)}", seeds), S)


def curve(num, E, S, what, split="Test-set", every=5):
    """split: "Test-set" for evaluated quantities; None for training-time quantities (losses, drift), whose text
    starts with its own capital. every: rounds between the plotted points (1: every round)."""
    head = f"{split} {what}" if split else what
    seeds = _seeds(E, S)
    when = "per round" if every == 1 else f"per evaluation round (every {every} rounds)"
    return track2_words(one_seed(f"{head}, {when}, mean over seeds; the shaded band is "
                                 f"one standard deviation. {setting_text(S, seeds)}", seeds), S)


PROTOTYPE_METHODS = ("FedProto", "FedTGP")


def proto_note(methods):
    """Methods that classify by the nearest prototype score classes by negative distances; a softmax of those is no
    probability estimate, so calibration, NLL and Brier score are not comparable with the other methods."""
    ps_ = [m for m in methods if m in PROTOTYPE_METHODS]
    if not ps_:
        return ""
    one = len(ps_) == 1
    return (" " + " and ".join(ps_) + (" classifies" if one else " classify") + " by the nearest prototype; "
            + ("its" if one else "their") + " scores are a softmax of negative distances, not probability estimates, "
            "so " + ("its" if one else "their") + " calibration, NLL and Brier values are not comparable with those of "
            "the other methods.")


def per_class(num, E, S, what, note=""):
    shared = DATASETS.get(S[0], {}).get("track") == 2
    model = "the global model, mean over seeds" if shared else "the clients' models as trained, mean over them and over seeds"
    seeds = _seeds(E, S)
    return one_seed(f"Per-class {what} on the test set after training ({model}); classes are ordered from the most to "
                    f"the least frequent in the federation's training data.{note} {setting_text(S, seeds)}", seeds)


def setting_slug(S):
    ds, IF, alpha, K, pfrac, tail_perm = S
    s = f"{ds}_IF{'nat' if IF is None else int(round(1 / IF))}_a{alpha}_K{K}"
    if pfrac < 1.0:
        s += f"_p{pfrac}"
    if tail_perm:
        s += "_randomrare"
    return s


NA = {
    7: "consensus soft labels are used only by FedECS-LT",
    11: "no saved confusion matrices for FedECS-LT or GeFL-F",
    19: "FedECS-LT is not part of this experiment", 20: "GeFL-F is not part of this experiment",
    22: "no generator in this experiment", 23: "no generator in this experiment",
    26: "made from the saved models (models.zip) after the experiment",
    30: "no class-row generator in this experiment", 33: "no class-row generator in this experiment",
    34: "no class-row generator in this experiment", 35: "no class-row generator in this experiment",
    36: "no class-row generator in this experiment", 45: "ablation figure: made from the ablation experiment",
    46: "sensitivity figure: made from the sweep experiments", 50: "all clients take part in every round",
    52: "privacy figure: made from the differential-privacy experiment",
    53: "privacy figure: made from the membership-inference experiment",
    54: "privacy figure: made from the differential-privacy experiment",
    55: "the method diagram is drawn once for the paper", 57: "no generator copies were evaluated",
    56: "made for FedECS-LT and GeFL-F when several clients share a head architecture: either neither method is "
        "part of this experiment, or every architecture has a single client (10 clients, 10 architectures), so "
        "aggregating a head leaves it unchanged",
    59: "made for FedECS-LT and GeFL-F when several clients share a head architecture: either neither method is "
        "part of this experiment, or every architecture has a single client (10 clients, 10 architectures), so "
        "aggregating a head leaves it unchanged",
    58: "no class-row generator in this experiment", 60: "local-only training is not part of this experiment",
    69: "the exact readout is computed only for FedECS-LT and the GeFL-F family of methods",
    71: "made for the one-shared-model setting only", 72: "made for the one-shared-model setting only",
    73: "made for the one-shared-model setting only",
    70: "FedECS-LT and GeFL-F are not part of this experiment (their per-round, per-class metrics are in the logs)",
    74: "the head-training probes are logged by FedECS-LT and the GeFL-F family of methods only",
    75: "no method of this experiment has a synthetic phase in its head training",
    76: "no method of this experiment has a synthetic phase in its head training",
    77: "consensus soft labels are used only by FedECS-LT",
    78: "no saved test embeddings with last layers in this experiment",
    79: "CKA compares the embeddings of several head architectures of FedECS-LT or GeFL-F",
    80: "no generator in this experiment", 81: "no generator in this experiment",
    82: "no variational generator in this experiment",
    83: "made when FedECS-LT and GeFL-F with saved test embeddings are both part of the experiment",
    84: "no saved test embeddings of FedECS-LT in this experiment",
    85: "no saved test embeddings of FedECS-LT or GeFL-F in this experiment",
}


def not_applicable(num):
    return NA.get(num, "the data this figure needs is not produced by this experiment")


# Words and patterns a caption in the paper must not contain (internal names, file names, run ids, raw keys)
LINT_BANNED = [r"\bsuite\b", r"\bTrack\b", r"\bnotebook", r"\bcache", r"\bpipeline\b", r"\bholders?\b",
               r"\bER\b", r"\(LongTailLib\)", r"\|(IF|a\d|K\d|s\d|p\d)", r"\bfinal_", r"\ber_", r"\bval_", r"_(?!\{)",
               r"\bOurs?\b", r"\bours\b", r"\bTODO\b", r"\bnan\b"]


def lint(text, figure=True):
    """Problems of a caption (figure or table): banned internal words; for figures, the setting and the seeds
    must be stated. Not-applicable reasons are checked with figure=False."""
    import re as _re
    out = [f"contains '{m.group(0)}'" for pat in LINT_BANNED for m in [_re.search(pat, text)] if m]
    if figure and not (_re.search(r"\bclients\b", text) and _re.search(r"\bseeds?\b", text)):
        out.append("does not state the setting and the seeds")
    return out


In [ ]:
%%writefile /kaggle/working/fedecs_lt/class_names.py
"""Class names of every dataset, by the original label index of its files. Sources: torchvision (MNIST,
Fashion-MNIST), the Hugging Face dataset cards (SVHN ufldl-stanford/svhn, CIFAR uoft-cs/cifar10 and cifar100,
FER-2013 Aaryan333/fer2013_train_publicTest_privateTest), MedMNIST v2 medmnist/info.py, the GTSRB signnames,
the EMNIST ByClass mapping (digits, then A-Z, then a-z)."""

CLASS_NAMES = {
    'mnist': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
    'fmnist': ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'],
    'svhn': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'],
    'cifar10': ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'],
    'cifar100': ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'cra', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm'],
    'dermamnist': ['actinic keratoses and intraepithelial carcinoma', 'basal cell carcinoma', 'benign keratosis-like lesions', 'dermatofibroma', 'melanoma', 'melanocytic nevi', 'vascular lesions'],
    'organamnist': ['bladder', 'femur-left', 'femur-right', 'heart', 'kidney-left', 'kidney-right', 'liver', 'lung-left', 'lung-right', 'pancreas', 'spleen'],
    'bloodmnist': ['basophil', 'eosinophil', 'erythroblast', 'immature granulocytes(myelocytes, metamyelocytes and promyelocytes)', 'lymphocyte', 'monocyte', 'neutrophil', 'platelet'],
    'pathmnist': ['adipose', 'background', 'debris', 'lymphocytes', 'mucus', 'smooth muscle', 'normal colon mucosa', 'cancer-associated stroma', 'colorectal adenocarcinoma epithelium'],
    'gtsrb': ['Speed limit (20km/h)', 'Speed limit (30km/h)', 'Speed limit (50km/h)', 'Speed limit (60km/h)', 'Speed limit (70km/h)', 'Speed limit (80km/h)', 'End of speed limit (80km/h)', 'Speed limit (100km/h)', 'Speed limit (120km/h)', 'No passing', 'No passing for vehicles over 3.5 metric tons', 'Right-of-way at the next intersection', 'Priority road', 'Yield', 'Stop', 'No vehicles', 'Vehicles over 3.5 metric tons prohibited', 'No entry', 'General caution', 'Dangerous curve to the left', 'Dangerous curve to the right', 'Double curve', 'Bumpy road', 'Slippery road', 'Road narrows on the right', 'Road work', 'Traffic signals', 'Pedestrians', 'Children crossing', 'Bicycles crossing', 'Beware of ice/snow', 'Wild animals crossing', 'End of all speed and passing limits', 'Turn right ahead', 'Turn left ahead', 'Ahead only', 'Go straight or right', 'Go straight or left', 'Keep right', 'Keep left', 'Roundabout mandatory', 'End of no passing', 'End of no passing by vehicles over 3.5 metric tons'],
    'emnist': ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z'],
    'fer2013': ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral'],
    'cifar10_t2': ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'],
    'cifar100_t2': ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver', 'bed', 'bee', 'beetle', 'bicycle', 'bottle', 'bowl', 'boy', 'bridge', 'bus', 'butterfly', 'camel', 'can', 'castle', 'caterpillar', 'cattle', 'chair', 'chimpanzee', 'clock', 'cloud', 'cockroach', 'couch', 'cra', 'crocodile', 'cup', 'dinosaur', 'dolphin', 'elephant', 'flatfish', 'forest', 'fox', 'girl', 'hamster', 'house', 'kangaroo', 'keyboard', 'lamp', 'lawn_mower', 'leopard', 'lion', 'lizard', 'lobster', 'man', 'maple_tree', 'motorcycle', 'mountain', 'mouse', 'mushroom', 'oak_tree', 'orange', 'orchid', 'otter', 'palm_tree', 'pear', 'pickup_truck', 'pine_tree', 'plain', 'plate', 'poppy', 'porcupine', 'possum', 'rabbit', 'raccoon', 'ray', 'road', 'rocket', 'rose', 'sea', 'seal', 'shark', 'shrew', 'skunk', 'skyscraper', 'snail', 'snake', 'spider', 'squirrel', 'streetcar', 'sunflower', 'sweet_pepper', 'table', 'tank', 'telephone', 'television', 'tiger', 'tractor', 'train', 'trout', 'tulip', 'turtle', 'wardrobe', 'whale', 'willow_tree', 'wolf', 'woman', 'worm'],
}


In [ ]:
%%writefile /kaggle/working/fedecs_lt/client.py
"""Flower ClientApp: what each client does with its own data. One message type per action:

  query.hello        partition id (the server maps nodes to clients)
  train.stage1       Stage I: local training of the feature extractor and the client's head
  train.gen          Stage II: local generator training (persistent optimiser state per client)
  train.head         Stage III: local head training on generated, then real features
  train.local        Local-only baseline: the whole model trained on the client's data only
  train              secure-sum contributions (SecAgg+ mod): class statistics, CCWA rows, consensus, readout sums
  query.sums         the same contributions in plain (protocols of baselines that send statistics unprotected)
  evaluate.*         evaluation of models the server sends, on the shared test copy and on the validation set

The simulation engine runs one persistent client process per simulation; client data, frozen features, the
generator and per-client optimiser states are kept in that process (dictionary STATE), keyed by run and client.
"""
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from flwr.clientapp import ClientApp
from flwr.client.mod import secaggplus_mod

import fedecs
import local_train as lt
from data import Federation, batches, gpu_augment
from fl_common import arrays_of, cfg_of, reply, seed_all
from metrics import arch_priors, class_metrics, evaluate_heads, log_view
from models import GraphCache, PaperHeader, make_fe, make_generator, make_head, make_headers
from secure_sum import client_secsum
from settings import DATASETS, HEADER_WIDTHS

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
STATE = {}  # process-level store: federations, features, generators, optimiser states
app = ClientApp()


# ---------------------------------------------------------------- process store
def fed_of(cfg):
    k = ("fed", cfg["ds"], cfg["IF"], cfg["alpha"], cfg["K"], cfg["seed"], cfg.get("tail_perm", False),
         cfg.get("frac_scale", 1.0))
    if k not in STATE:
        for old in [x for x in STATE if x[0] in ("fed", "feat")]:  # one dataset split per process at a time
            del STATE[old]
        torch.cuda.empty_cache() if DEV == "cuda" else None
        STATE[k] = Federation(cfg["ds"], cfg["data_root"], cfg["IF"], cfg["alpha"], cfg["K"], cfg["seed"], DEV,
                              tail_perm=cfg.get("tail_perm", False), frac_scale=cfg.get("frac_scale", 1.0))
    return STATE[k]


def pid_of(context):
    return int(context.node_config["partition-id"])


def spec_of(cfg):
    return DATASETS[cfg["ds"]]


def feats_of(cfg, fed):
    """Frozen features of every client, the test set and the validation set, for the FE the server broadcast."""
    k = ("feat", cfg["fe_key"])
    if k not in STATE:
        raise KeyError("features not computed: server must send query.features first")
    return STATE[k]


@torch.no_grad()
def compute_features(fe, x):
    fe.eval()
    return torch.cat([fe(x[s:s + 4000]) for s in range(0, len(x), 4000)])


def graphs_for(name):
    k = ("graphs", name)
    if k not in STATE:
        STATE[k] = GraphCache(DEV == "cuda")
    return STATE[k]


# ---------------------------------------------------------------- hello
@app.query("hello")
def hello(msg, context):
    if DEV == "cuda":   # a new run: this process's GPU memory peak is measured from here
        torch.cuda.reset_peak_memory_stats()
    return reply(msg, dict(pid=pid_of(context), device=DEV, pidproc=os.getpid()))


@app.query("gpu")
def gpu(msg, context):
    """Peak GPU memory of this client process since the run's hello (MB)."""
    peak = round(torch.cuda.max_memory_allocated() / 2 ** 20) if DEV == "cuda" else 0
    return reply(msg, dict(peak_mb=peak, pidproc=os.getpid()))


# ---------------------------------------------------------------- stage I
@app.train("stage1")
def stage1(msg, context):
    cfg, k = cfg_of(msg.content), pid_of(context)
    fed, spec = fed_of(cfg), spec_of(cfg)
    seed_all(cfg["seed"], "i", cfg["round"], k)
    t0 = time.time()
    g = fed.arch[k]
    fe = make_fe(spec).to(DEV)
    head = make_head(spec, g, fe.out_ch, fed.C).to(DEV)
    fe.load_state_dict(arrays_of(msg.content, "fe", DEV))
    head.load_state_dict(arrays_of(msg.content, "head", DEV))
    prior = fedecs.local_log_prior(fed.counts[k], DEV) if cfg.get("la") else None
    x_k, y_k = fed.x[k], fed.y[k]
    lr, mom = lt.protocol_lr(spec, cfg["round"])
    losses = lt.local_stage1(fe, head, x_k, y_k, spec, DEV, prior=prior, epochs=spec.get("local_epochs", 5),
                             lr=lr, momentum=mom)
    out = dict(n=len(y_k), loss=float(np.mean(losses)) if losses else None, time=time.time() - t0)
    if cfg.get("eval"):  # pre-aggregation evaluation of the local model on the test set
        with torch.no_grad():
            fte = compute_features(fe, fed.x_te)
            head.eval()
            logits = torch.cat([head(fte[s:s + 2000]) for s in range(0, len(fed.y_te), 2000)])
        m = class_metrics(logits, fed.y_te, fed.C)
        out["pre"] = log_view(m)
    return reply(msg, out, fe=fe.state_dict(), head=head.state_dict())


@app.query("features")
def features(msg, context):
    """Compute (once per process) the frozen features of every client, the test and validation sets."""
    cfg = cfg_of(msg.content)
    fed, spec = fed_of(cfg), spec_of(cfg)
    k = ("feat", cfg["fe_key"])
    if k not in STATE:
        fe = make_fe(spec).to(DEV)
        fe.load_state_dict(arrays_of(msg.content, "fe", DEV))
        STATE[k] = dict(x=[compute_features(fe, x) for x in fed.x], te=compute_features(fe, fed.x_te),
                        val=compute_features(fe, fed.x_val))
    return reply(msg, dict(ok=True, shape=list(STATE[k]["te"].shape[1:])))


# ---------------------------------------------------------------- stage II
def gen_store(cfg, fed, shape):
    """The generator module and every client's persistent optimiser, bound to its parameters (as core.GenClient)."""
    k = ("gen", cfg["gen_key"])
    if k not in STATE:
        for old in [x for x in STATE if x[0] == "gen"]:
            del STATE[old]
        spec = spec_of(cfg)
        proto = None
        if cfg["gen"]["type"] == "pcvae":
            proto = torch.zeros(fed.C, *shape, device=DEV)  # the class means arrive in the weights (buffer)
        G = make_generator(cfg["gen"]["type"], shape[0], spec["latent"], fed.C, proto).to(DEV)
        G.graphs = GraphCache(DEV == "cuda")
        STATE[k] = dict(G=G, opts={}, local={})
    return STATE[k]


def gen_opt(store, k, gspec):
    if k not in store["opts"]:
        G = store["G"]
        cls = torch.optim.AdamW if gspec.get("opt", "adam") == "adamw" else torch.optim.Adam
        store["opts"][k] = cls([{"params": list(G.parameters()), "weight_decay": gspec.get("wd", 1e-3)}],
                               lr=1e-3, fused=DEV == "cuda")
    return store["opts"][k]


@app.train("gen")
def gen(msg, context):
    cfg, k = cfg_of(msg.content), pid_of(context)
    fed = fed_of(cfg)
    F_ = feats_of(cfg, fed)
    X, Y = F_["x"][k], fed.y[k]
    store = gen_store(cfg, fed, tuple(X.shape[1:]))
    G = store["G"]
    G.load_state_dict(arrays_of(msg.content, "g", DEV))
    opt = gen_opt(store, k, cfg["gen"])
    seed_all(cfg["seed"], "ii", cfg["round"], k)
    t0 = time.time()
    losses = lt.local_gen(G, opt, X, Y, DEV)
    sd = {kk: v.detach().clone() for kk, v in G.state_dict().items()}
    store["local"][k] = sd
    out = dict(n=len(Y), loss=float(np.mean(losses)) if losses else None, time=time.time() - t0)
    if cfg.get("eval") and "ref" in msg.content.array_records:
        # pre-aggregation diagnostic: per-class fidelity of this client's local generator copy (section 17b)
        ref = PaperHeader(HEADER_WIDTHS[4], X.shape[1], True, fed.C).to(DEV)
        ref.load_state_dict(arrays_of(msg.content, "ref", DEV))
        ref.eval()
        G.eval()
        with torch.no_grad():
            out["fidelity_pre"] = [float((ref(G.sample(torch.full((200,), c, device=DEV, dtype=torch.long))).argmax(1)
                                          == c).float().mean()) for c in range(fed.C)]
        G.train()
    if G.cond:  # diagnostics: class-row norms of the local copy and Adam's second moment of the rows
        W = G.label_proj.detach()
        out["row_norm"] = W.norm(dim=1).tolist()
        st = opt.state.get(G.label_proj, {})
        if "exp_avg_sq" in st:
            out["row_v"] = st["exp_avg_sq"].mean(1).tolist()
    if cfg.get("send_rows", True):
        return reply(msg, out, g=sd)
    return reply(msg, out, g={kk: v for kk, v in sd.items() if kk not in G.cond})


@app.query("load_gen")
def load_gen(msg, context):
    """Stage III: the final generator (and moment-correction parameters) once per process."""
    cfg = cfg_of(msg.content)
    fed = fed_of(cfg)
    F_ = feats_of(cfg, fed)
    k = ("sampler", cfg["sampler_key"])
    if k not in STATE:
        # Stage II is over: its every-client Adam states and local generator copies are never used again; they held
        # about 5.5 GB of a T4 through Stage III (CIFAR-100, one shared model, 20 clients)
        for old in [x for x in STATE if x[0] in ("sampler", "gen")]:
            del STATE[old]
        if DEV == "cuda":
            torch.cuda.empty_cache()
        shape = tuple(F_["te"].shape[1:])
        spec = spec_of(cfg)
        if cfg["gen"]["type"] == "gauss":
            from models import GaussFeatureGen
            a = arrays_of(msg.content, "g", DEV)
            G = GaussFeatureGen(a["mu"], a["chol"], shape).to(DEV)
        else:
            proto = None
            if cfg["gen"]["type"] == "pcvae":
                proto = torch.zeros(fed.C, *shape, device=DEV)  # the class means arrive in the weights (buffer)
            G = make_generator(cfg["gen"]["type"], shape[0], spec["latent"], fed.C, proto).to(DEV)
            G.load_state_dict(arrays_of(msg.content, "g", DEV))
        G.eval()
        mc = arrays_of(msg.content, "mc", DEV)
        STATE[k] = dict(G=G, mc=fedecs.MCParams(mc["mu"], mc["m"], mc["scale"]) if mc else None, pool={},
                        shape=shape)
    return reply(msg, dict(ok=True))


def sampler_of(cfg):
    return STATE[("sampler", cfg["sampler_key"])]


@torch.no_grad()
@torch.no_grad()
def draw(sm, y):
    """Generated features for labels y (moment-corrected when the server sent MC parameters), written batch by batch
    into one output tensor (the correction is per sample, so this equals correcting the whole draw at once). Batches
    of 4096 as core.py; 1024 for very large features (Track 2's 32,768 dimensions), to bound the generator's memory."""
    d = int(np.prod(sm["shape"])) if "shape" in sm else 0
    bs = 4096 if d <= 8192 else 1024
    out = None
    for s in range(0, len(y), bs):
        yb = y[s:s + bs]
        xb = sm["G"].sample(yb)
        if sm["mc"] is not None:
            xb = sm["mc"].apply(xb, yb)
        if out is None:
            out = torch.empty((len(y),) + tuple(xb.shape[1:]), dtype=xb.dtype, device=xb.device)
        out[s:s + len(yb)] = xb
        del xb
    return out


POOL_CHUNK_BYTES = 2 ** 30   # the pool is generated in chunks of at most 1 GiB of features


def pool_chunk_size(cfg, sm):
    d = int(np.prod(sm["shape"]))
    return max(1, POOL_CHUNK_BYTES // (4 * d))


@torch.no_grad()
def pool_chunk(cfg, sm, j):
    """Chunk j of this round's shared synthetic pool, regenerated identically by every client from the server's seed
    (chunk j uses seed pool_seed + j). A pool that fits in one chunk (every Track-1 setting) is the same draw as one
    generated at once; with one shared architecture the pool covers all clients' slices and is far larger, so no
    client ever holds all of it."""
    L = pool_chunk_size(cfg, sm)
    n = min(L, cfg["P"] - j * L)
    gstate = torch.random.get_rng_state()
    cstate = torch.cuda.get_rng_state_all() if DEV == "cuda" else None
    torch.manual_seed(cfg["pool_seed"] + j)
    if DEV == "cuda":
        torch.cuda.manual_seed_all(cfg["pool_seed"] + j)
    allowed = torch.arange(cfg["C"], device=DEV)
    py = allowed[torch.randint(0, len(allowed), (n,), device=DEV)]
    px = draw(sm, py)
    torch.random.set_rng_state(gstate)
    if cstate is not None:
        torch.cuda.set_rng_state_all(cstate)
    return px, py


def pool_slice(cfg, sm, o, n):
    """Pool samples [o, o + n) (a client's slice): only the chunks that cover it are generated; the last slice is
    cached, so a client's head step reuses what its consensus step needed."""
    key = (cfg["round"], cfg["pool_seed"], o, n)
    if sm["pool"].get("key") != key:
        sm["pool"].clear()   # free the previous slice first: two slices at once ran a T4 out of memory (CIFAR-100,
        if DEV == "cuda":    # one shared model, ratio 10; 27 Oct 2026). The values drawn are unchanged.
            torch.cuda.empty_cache()
        L = pool_chunk_size(cfg, sm)
        xs = ys = None   # the slice is written chunk by chunk into one tensor, so at most one chunk is alive at a time
        for j in range(o // L, (o + n - 1) // L + 1):
            px, py = pool_chunk(cfg, sm, j)
            a, b = max(o, j * L) - j * L, min(o + n, (j + 1) * L) - j * L
            if xs is None:
                xs = torch.empty((n,) + tuple(px.shape[1:]), dtype=px.dtype, device=px.device)
                ys = torch.empty((n,), dtype=py.dtype, device=py.device)
            s = j * L + a - o
            xs[s:s + b - a] = px[a:b]
            ys[s:s + b - a] = py[a:b]
            del px, py
        sm["pool"].update(key=key, xy=(xs, ys))
    return sm["pool"]["xy"]


@torch.no_grad()
def pool_probs(cfg, sm, h, g):
    """softmax(h_g(pool)) over the whole pool, chunk by chunk (features are dropped after use); cached per round,
    pool and head, since every client of architecture g computes the same values."""
    key = ("pool_probs", cfg["round"], cfg["pool_seed"], cfg["heads_key"], g)
    if key not in STATE:
        for old in [x for x in STATE if x[0] == "pool_probs"]:
            del STATE[old]
        L = pool_chunk_size(cfg, sm)
        out = []
        for j in range((cfg["P"] + L - 1) // L):
            px, _ = pool_chunk(cfg, sm, j)
            out.append(torch.cat([h(px[s:s + 2048]).softmax(1) for s in range(0, len(px), 2048)]))
            del px
        STATE[key] = torch.cat(out)
    return STATE[key]


# ---------------------------------------------------------------- stage III
def head_of(cfg, fed, g, arrays, in_ch):
    """A fresh head module of architecture g with the given weights (inference only)."""
    spec = spec_of(cfg)
    h = make_head(spec, g, in_ch, fed.C).to(DEV)
    h.load_state_dict(arrays)
    return h


def train_head_module(cfg, fed, g, arrays, in_ch):
    """The persistent head module of architecture g for this run (weights loaded in place), so that CUDA graphs
    recorded on its parameters stay valid across rounds and clients (as core.py's headers list)."""
    k = ("headmod", cfg["run_id"], g)
    if k not in STATE:
        for old in [x for x in STATE if x[0] == "headmod" and x[1] != cfg["run_id"]]:
            del STATE[old]
        STATE[k] = head_of(cfg, fed, g, arrays, in_ch)
    h = STATE[k]
    h.load_state_dict(arrays)
    return h


@app.train("head")
def head(msg, context):
    cfg, k = cfg_of(msg.content), pid_of(context)
    fed, spec = fed_of(cfg), spec_of(cfg)
    F_ = feats_of(cfg, fed)
    hs = cfg["head"]
    g = fed.arch[k]
    X, Y = F_["x"][k], fed.y[k]
    h = train_head_module(cfg, fed, g, arrays_of(msg.content, "head", DEV), X.shape[1])
    h.train()
    seed_all(cfg["seed"], "iii", cfg["round"], k)
    t0 = time.time()
    B, C, n = spec["B"], fed.C, len(Y)
    nb, bs = (n // B, B) if n >= B else (1, n)
    graphs = graphs_for(("head", cfg["run_id"], g))
    full = {B}

    def fwd(x):
        return graphs((len(x),), h, h, x) if len(x) in full else h(x)
    prior = fedecs.local_log_prior(fed.counts[k], DEV) if hs["loss"] == "la" else None
    adj = fedecs.client_loss_adj(hs["loss"], fed.counts[k], hs.get("tau", 1.0), DEV)
    out = dict(n=n)
    syn = None
    if hs["ts"] > 0 and cfg.get("sampler_key"):
        sm = sampler_of(cfg)
        nsyn = lt.n_syn(n, B, hs["ts"])
        if "cons" in msg.content.array_records:   # consensus soft labels (CSL) on the shared pool
            px, py = pool_slice(cfg, sm, int(cfg["pool_off"]), nsyn)
            cons = arrays_of(msg.content, "cons", DEV)["t"].float()
            syn = ("soft", px, fedecs.csl_targets(py, cons, cfg["csl_beta"], C))
            # agreement: share of the client's pool features whose consensus class is the generated label
            out["consensus_agreement"] = float((cons.argmax(1) == py).float().mean())
        else:                                     # uniform synthetic labels, hard targets (GeFL-F)
            ylab = lt.syn_hard_labels(C, nsyn).to(DEV)
            syn = ("hard", draw(sm, ylab), ylab)
    lr, mom = lt.protocol_lr(spec, cfg.get("round_offset", 0) + cfg["round"])
    syn_losses, real_losses, probes = lt.local_head(h, fwd, X, Y, B, hs, DEV, prior=prior, adj=adj, syn=syn, lr=lr,
                                                    momentum=mom, probe_C=C)
    out.update(probe=probes, syn_loss=float(np.mean(syn_losses)) if syn_losses else None,
               real_loss=float(np.mean(real_losses)) if real_losses else None, time=time.time() - t0)
    if cfg.get("eval"):
        h.eval()
        with torch.no_grad():
            logits = torch.cat([h(F_["te"][s:s + 2000]) for s in range(0, len(fed.y_te), 2000)])
        m = class_metrics(logits, fed.y_te, C)
        out["pre"] = log_view(m)
    return reply(msg, out, head=h.state_dict())


# ---------------------------------------------------------------- local-only baseline
@app.train("local")
def local(msg, context):
    """No federation: the client trains the extractor and its head on its own data for the same number of epochs
    as the federated stages; returns metrics of its own model on the test set (per-class benefit of collaboration)."""
    cfg, k = cfg_of(msg.content), pid_of(context)
    fed, spec = fed_of(cfg), spec_of(cfg)
    seed_all(cfg["seed"], "local", 0, k)
    g = fed.arch[k]
    fe = make_fe(spec).to(DEV)
    h = make_head(spec, g, fe.out_ch, fed.C).to(DEV)
    fe.load_state_dict(arrays_of(msg.content, "fe", DEV))
    h.load_state_dict(arrays_of(msg.content, "head", DEV))
    opt = torch.optim.SGD(list(fe.parameters()) + list(h.parameters()), lr=0.1, momentum=0.0)
    x_k, y_k = fed.x[k], fed.y[k]
    for _ in range(cfg["epochs"]):
        fe.train(), h.train()
        for b in batches(len(y_k), spec["B"], DEV):
            if b.numel() < 2:
                continue
            x = gpu_augment(x_k[b]) if spec["augment"] else x_k[b]
            loss = F.cross_entropy(h(fe(x)), y_k[b])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
    out = {}
    for split, x, y in (("metrics", fed.x_te, fed.y_te), ("val_metrics", fed.x_val, fed.y_val)):
        with torch.no_grad():
            f_ = compute_features(fe, x)
            h.eval()
            logits = torch.cat([h(f_[s:s + 2000]) for s in range(0, len(y), 2000)])
        m = class_metrics(logits, y, fed.C, prior=fed.counts[k] / max(fed.counts[k].sum(), 1))   # own mix
        out[split] = {kk: v for kk, v in m.items() if kk != "probs"}
    return reply(msg, dict(n=len(y_k), **out))


# ---------------------------------------------------------------- statistics for sums (secure or plain)
def contributions(cfg, k):
    """What client k adds to a sum. Every item is a per-class sum of its own data (or of predictions on public
    generated features), so the server needs only the total."""
    fed = fed_of(cfg)
    what = cfg["what"]
    C = fed.C
    if what == "stats":                     # class statistics release: n, sum h, sum ||h||^2
        F_ = feats_of(cfg, fed)
        n, S1, S2 = fedecs.class_stats(F_["x"][k], fed.y[k], C)
        return [n.cpu().numpy(), S1.cpu().numpy(), S2.cpu().numpy()]
    if what == "gram":                      # Gaussian generator: n, class sums, sum h h^T
        F_ = feats_of(cfg, fed)
        Xf = F_["x"][k].flatten(1).double()
        n = torch.bincount(fed.y[k], minlength=C).double()
        S = torch.zeros(C, Xf.shape[1], dtype=torch.float64, device=DEV).index_add_(0, fed.y[k], Xf)
        return [n.cpu().numpy(), S.cpu().numpy(), (Xf.T @ Xf).cpu().numpy()]
    if what == "rows":                      # CCWA: E(n_kc) row_kc and E(n_kc)
        store = STATE[("gen", cfg["gen_key"])]
        rows = store["local"][k]["label_proj"].double()
        w = torch.tensor(fedecs.ccwa_weights(fed.counts[k], cfg["beta"]), dtype=torch.float64, device=DEV)
        return [(w[:, None] * rows).cpu().numpy(), w.cpu().numpy()]
    if what == "consensus":                 # w_k softmax(h_g(pool)), w_k = 1 / (G n_arch)
        F_ = feats_of(cfg, fed)
        sm = sampler_of(cfg)
        g = fed.arch[k]
        h = head_of(cfg, fed, g, {kk: torch.tensor(np.array(v)) for kk, v in STATE[("heads", cfg["heads_key"])][g].items()},
                    F_["te"].shape[1])
        h.eval()
        p = pool_probs(cfg, sm, h, g)
        return [(cfg["w"][str(k)] * p.double()).cpu().numpy()]
    if what == "readout":                   # per architecture: n, sum e, sum e e^T of the client's real features
        F_ = feats_of(cfg, fed)
        out = []
        for g in cfg["groups"]:
            h = head_of(cfg, fed, g, {kk: torch.tensor(np.array(v)) for kk, v in STATE[("heads", cfg["heads_key"])][g].items()},
                        F_["te"].shape[1])
            h.eval()
            with torch.no_grad():
                E = torch.cat([h.embed(F_["x"][k][s:s + 4096]) for s in range(0, len(fed.y[k]), 4096)])
            n, S, M = fedecs.embed_sums(E, fed.y[k], C)
            out += [n.cpu().numpy(), S.cpu().numpy(), M.cpu().numpy()]
        return out
    raise ValueError(what)


@app.train(mods=[secaggplus_mod])
def secsum(msg, context):
    k = pid_of(context)
    return client_secsum(msg, lambda cfg: contributions(cfg, k))


@app.query("sums")
def plain_sums(msg, context):
    cfg = cfg_of(msg.content)
    arrs = contributions(cfg, pid_of(context))
    return reply(msg, dict(n=len(arrs)), s={str(i): torch.from_numpy(np.ascontiguousarray(a)) for i, a in enumerate(arrs)})


@app.query("heads")
def set_heads(msg, context):
    """Cache the current global heads (needed by consensus and readout contributions)."""
    cfg = cfg_of(msg.content)
    w = arrays_of(msg.content, "heads", "cpu")
    heads = {}
    for name, v in w.items():
        g, kk = name.split("|", 1)
        heads.setdefault(int(g), {})[kk] = v.numpy()
    for old in [x for x in STATE if x[0] == "heads"]:
        del STATE[old]
    STATE[("heads", cfg["heads_key"])] = heads
    return reply(msg, dict(ok=True))


# ---------------------------------------------------------------- evaluation (clients evaluate, server collects)
@app.evaluate("heads")
def eval_heads(msg, context):
    cfg = cfg_of(msg.content)
    fed = fed_of(cfg)
    F_ = feats_of(cfg, fed)
    w = arrays_of(msg.content, "heads", DEV)
    in_ch = F_["te"].shape[1]
    heads = make_headers(spec_of(cfg), in_ch, fed.C, DEV)
    per_g = {}
    for name, v in w.items():
        g, kk = name.split("|", 1)
        per_g.setdefault(int(g), {})[kk] = v
    groups = sorted(per_g)
    for g in groups:
        heads[g].load_state_dict(per_g[g])
    lw = arrays_of(msg.content, "lastlayer", DEV)  # optional replacement last layers (readouts)
    if lw:
        for g in groups:
            heads[g].fc.weight.data.copy_(lw[f"{g}|W"])
            heads[g].fc.bias.data.copy_(lw[f"{g}|b"])
    out, probs = {}, {}
    for split in cfg.get("splits", ["test"]):
        X, y = (F_["te"], fed.y_te) if split == "test" else (F_["val"], fed.y_val)
        priors = arch_priors(fed.counts, fed.arch, groups)
        per, summ, emb = evaluate_heads(heads, groups, X, y, fed.C, priors=priors,
                                        keep_probs=cfg.get("probs", False) and split == "test",
                                        embed=cfg.get("embed", False) and split == "test")
        res = dict(summary=summ, per_arch={str(g): {kk: v for kk, v in m.items() if kk != "probs"} for g, m in per.items()})
        if cfg.get("probs") and split == "test":
            # float16 arrays (the precision save_run stores) instead of JSON lists of floats: EMNIST's 116,323 x 62
            # x 10 values took several GB of host memory per evaluation (Ray killed the process, Kaggle, 26 Oct 2026)
            probs = {str(g): torch.from_numpy(m["probs"]).half() for g, m in per.items()}
        if emb:
            sub = np.random.RandomState(0).choice(len(y), min(2000, len(y)), replace=False)
            res["embed"] = {str(g): e[sub].tolist() for g, e in emb.items()}
            res["embed_y"] = fed.y_te.cpu().numpy()[sub].tolist()
        out[split] = res
    return reply(msg, out, probs=probs) if probs else reply(msg, out)


@app.evaluate("stage1")
def eval_stage1(msg, context):
    cfg = cfg_of(msg.content)
    fed, spec = fed_of(cfg), spec_of(cfg)
    fe = make_fe(spec).to(DEV)
    fe.load_state_dict(arrays_of(msg.content, "fe", DEV))
    fte = compute_features(fe, fed.x_te)
    w = arrays_of(msg.content, "heads", DEV)
    heads = make_headers(spec, fe.out_ch, fed.C, DEV)
    per_g = {}
    for name, v in w.items():
        g, kk = name.split("|", 1)
        per_g.setdefault(int(g), {})[kk] = v
    groups = sorted(per_g)
    for g in groups:
        heads[g].load_state_dict(per_g[g])
    per, summ, _ = evaluate_heads(heads, groups, fte, fed.y_te, fed.C, priors=arch_priors(fed.counts, fed.arch, groups))
    return reply(msg, dict(summary=summ))


@app.evaluate("gen")
def eval_gen(msg, context):
    """Generator diagnostics on the client: per-class fidelity (share of generated class-c features the referee
    classifies as c, 200 per class) of the generator weights sent; pre- or post-aggregation."""
    cfg = cfg_of(msg.content)
    fed = fed_of(cfg)
    F_ = feats_of(cfg, fed)
    spec = spec_of(cfg)
    shape = tuple(F_["te"].shape[1:])
    proto = None
    if cfg["gen"]["type"] == "pcvae":
        proto = torch.zeros(fed.C, *shape, device=DEV)  # the class means arrive in the weights (buffer)
    G = make_generator(cfg["gen"]["type"], shape[0], spec["latent"], fed.C, proto).to(DEV)
    ref = PaperHeader(HEADER_WIDTHS[4], shape[0], True, fed.C).to(DEV)
    ref.load_state_dict(arrays_of(msg.content, "ref", DEV))
    ref.eval()
    out = {}
    for name in cfg["copies"]:
        G.load_state_dict(arrays_of(msg.content, "g_" + name, DEV))
        G.eval()
        seed_all(cfg["seed"], "evalgen", cfg["round"], name)
        fid = []
        with torch.no_grad():
            for c in range(fed.C):
                y = torch.full((200,), c, device=DEV, dtype=torch.long)
                fid.append(float((ref(G.sample(y)).argmax(1) == c).float().mean()))
        out[name] = fid
    return reply(msg, out)


import htfl_client  # noqa: E402,F401  (registers the Track-1 baseline handlers on this ClientApp)


In [ ]:
%%writefile /kaggle/working/fedecs_lt/data.py
"""Datasets, long-tail construction and client partition.

Tier-1 loading, preprocessing and make_partition are copied from experiments/core.py (load_dataset, make_partition,
assign_arch, permute_labels) so that the same seed gives the same clients as every earlier run; the unit test
tests/test_equivalence.py checks this against core.py. New datasets (tier 2) use the same preprocessing rules.
Only file formats that cannot run code are read (parquet, npz without pickle, idx, images).
"""
import gzip
import io
import os
import time
import urllib.request

import numpy as np
import torch
import torch.nn.functional as F

from settings import CAP_IMAGES, DATASETS


def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)


def _download(url, dst, tries=4):
    if os.path.exists(dst):
        return dst
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for i in range(tries):
        try:
            log(f"downloading {url}")
            part = f"{dst}.{os.getpid()}.part"   # one temporary file per process; the rename is atomic
            urllib.request.urlretrieve(url, part)
            os.replace(part, dst)
            return dst
        except Exception as e:  # network hiccup: retry with backoff
            log(f"download failed ({e}); retry {i + 1}/{tries}")
            time.sleep(2 ** (i + 1))
    raise RuntimeError(f"could not download {url}")


def _read_parquet_images(path, img_col, label_col, mode, size=None):
    """Images of a Hugging Face parquet file as uint8 (N, ch, H, W); size: resize every image to size x size (PIL's
    default filter, as torchvision-based loading did) for datasets whose images differ in size."""
    import pyarrow.parquet as pq
    from PIL import Image

    def img(b):
        im = Image.open(io.BytesIO(b)).convert(mode)
        return np.array(im.resize((size, size)) if size else im)
    t = pq.read_table(path).to_pydict()
    x = np.stack([img(im["bytes"]) for im in t[img_col]])
    x = torch.from_numpy(x)
    x = x.unsqueeze(1) if mode == "L" else x.permute(0, 3, 1, 2)
    return x, torch.tensor(t[label_col])


HF = "https://huggingface.co/datasets/"


class _DatasetLock:
    """Exclusive lock across processes for one dataset's files (download, conversion, tensor cache): the workers of
    a notebook start together and would otherwise download the same file into the same place at the same time."""

    def __init__(self, root, name):
        os.makedirs(root, exist_ok=True)
        self.path = os.path.join(root, f"_fedecs_lock_{name}")

    def __enter__(self):
        import fcntl
        self.f = open(self.path, "w")
        fcntl.flock(self.f, fcntl.LOCK_EX)
        return self

    def __exit__(self, *exc):
        import fcntl
        fcntl.flock(self.f, fcntl.LOCK_UN)
        self.f.close()


def _raw(name, root):
    """Raw uint8 images (N, ch, H, W) and int64 labels, train and test (one process at a time per dataset)."""
    with _DatasetLock(root, name):
        return _raw_unlocked(name, root)


def _raw_unlocked(name, root):
    import torchvision.datasets as tvd
    if name in ("mnist", "fmnist"):
        cls = tvd.MNIST if name == "mnist" else tvd.FashionMNIST
        tr, te = cls(root, train=True, download=True), cls(root, train=False, download=True)
        return tr.data.unsqueeze(1), tr.targets, te.data.unsqueeze(1), te.targets
    if name == "svhn":
        base = HF + "ufldl-stanford/svhn/resolve/main/cropped_digits/"
        out = []
        for split in ("train", "test"):
            p = _download(base + f"{split}-00000-of-00001.parquet", os.path.join(root, "svhn_hf", f"{split}.parquet"))
            out += list(_read_parquet_images(p, "image", "label", "RGB"))
        return tuple(out)
    if name in ("cifar10", "cifar100"):
        sub, lab = ("cifar10_hf", "label") if name == "cifar10" else ("cifar100_hf", "fine_label")
        base = HF + ("uoft-cs/cifar10/resolve/main/plain_text/" if name == "cifar10" else
                     "uoft-cs/cifar100/resolve/main/cifar100/")
        out = []
        for split in ("train", "test"):
            p = _download(base + f"{split}-00000-of-00001.parquet", os.path.join(root, sub, f"{split}.parquet"))
            out += list(_read_parquet_images(p, "img", lab, "RGB"))
        return tuple(out)
    if name in ("dermamnist", "organamnist", "bloodmnist", "pathmnist"):
        p = _download(f"https://zenodo.org/records/10519652/files/{name}.npz?download=1",
                      os.path.join(root, "medmnist", f"{name}.npz"))
        z = np.load(p, allow_pickle=False)

        def conv(x):
            x = torch.from_numpy(x)
            return x.unsqueeze(1) if x.dim() == 3 else x.permute(0, 3, 1, 2)
        return (conv(z["train_images"]), torch.from_numpy(z["train_labels"]).flatten().long(),
                conv(z["test_images"]), torch.from_numpy(z["test_labels"]).flatten().long())
    if name == "gtsrb":
        # the official training (39,209) and test (12,630) images, from a Hugging Face copy: the original server
        # (used by torchvision) timed out on Kaggle (27 Oct 2026); resized to 32 x 32 as before
        base = HF + "bazyl/GTSRB/resolve/main/data/"
        out = []
        for split, fn in (("train", "train-00000-of-00001-dc762c064c221993.parquet"),
                          ("test", "test-00000-of-00001-747a54d4a6461a97.parquet")):
            p = _download(base + fn, os.path.join(root, "gtsrb_hf", f"{split}.parquet"))
            out += list(_read_parquet_images(p, "Path", "ClassId", "RGB", size=32))
        return tuple(out)
    if name == "emnist":
        base = HF + "Royc30ne/emnist-byclass/resolve/main/"

        def idx(fn):
            p = _download(base + fn, os.path.join(root, "emnist_hf", fn))
            with gzip.open(p, "rb") as f:
                b = f.read()
            nd = b[3]
            dims = [int.from_bytes(b[4 + 4 * i:8 + 4 * i], "big") for i in range(nd)]
            return np.frombuffer(b, dtype=np.uint8, offset=4 + 4 * nd).reshape(dims)
        # EMNIST images are stored transposed relative to MNIST
        xtr = torch.from_numpy(idx("emnist-byclass-train-images-idx3-ubyte.gz").copy()).transpose(1, 2).unsqueeze(1)
        xte = torch.from_numpy(idx("emnist-byclass-test-images-idx3-ubyte.gz").copy()).transpose(1, 2).unsqueeze(1)
        ytr = torch.from_numpy(idx("emnist-byclass-train-labels-idx1-ubyte.gz").astype(np.int64))
        yte = torch.from_numpy(idx("emnist-byclass-test-labels-idx1-ubyte.gz").astype(np.int64))
        return xtr.contiguous(), ytr, xte.contiguous(), yte
    if name == "fer2013":
        base = HF + "Aaryan333/fer2013_train_publicTest_privateTest/resolve/main/data/"
        ptr = _download(base + "train-00000-of-00001-5eab84e1c6a2fc27.parquet", os.path.join(root, "fer_hf", "train.parquet"))
        pte = _download(base + "privateTest-00000-of-00001-4b8a0715cf1b7560.parquet",
                        os.path.join(root, "fer_hf", "test.parquet"))
        return _read_parquet_images(ptr, "image", "label", "L") + _read_parquet_images(pte, "image", "label", "L")
    raise ValueError(f"unknown dataset {name}")


def load_dataset(name, root, device="cpu", lazy=False):
    """Whole dataset as preprocessed tensors: x (N, ch, 32, 32) float, y (N,) long. Raw images are cached once as
    uint8 tensors in root (safe tensors-only file). Preprocessing as experiments/core.py: /255, 28 -> 32 bilinear
    (antialias) for small images, then the dataset's normalisation.
    lazy=True: the training images stay uint8 and data["prep"] preprocesses the ones selected. Preprocessing is per
    image, so the values are the same; it avoids a float copy of the whole training set in every process (EMNIST:
    697,932 images, 2.9 GB per copy, of which 20,000 are used; the host ran out of memory, Kaggle, 26 Oct 2026)."""
    raw = DATASETS[name].get("raw", name)
    cache = os.path.join(root, f"_fedecs_tensors_{raw}.pt")
    with _DatasetLock(root, raw):   # the cache is built by one process; the others wait and then read it
        if os.path.exists(cache):
            d = torch.load(cache, weights_only=True)
        else:
            xtr, ytr, xte, yte = _raw_unlocked(raw, root)
            d = dict(xtr=xtr.contiguous(), ytr=ytr.long(), xte=xte.contiguous(), yte=yte.long())
            tmp = f"{cache}.{os.getpid()}.tmp"
            torch.save(d, tmp)
            os.replace(tmp, cache)
    spec = DATASETS[name]

    def prep(x):
        if x.shape[-1] != 32:
            x = torch.cat([F.interpolate(x[i:i + 8192].float().div(255.0), size=32, mode="bilinear",
                                         align_corners=False, antialias=True) for i in range(0, len(x), 8192)])
        else:
            x = x.float().div(255.0)
        if spec["norm"] == "half":
            x.sub_(0.5).div_(0.5)
        elif spec["norm"] != "none":
            m = torch.tensor(spec["norm"][0]).view(1, -1, 1, 1)
            s = torch.tensor(spec["norm"][1]).view(1, -1, 1, 1)
            x.sub_(m).div_(s)
        return x
    if lazy:
        return dict(xtr=d["xtr"], ytr=d["ytr"], xte=prep(d["xte"]).to(device), yte=d["yte"].to(device), prep=prep)
    return dict(xtr=prep(d["xtr"]), ytr=d["ytr"], xte=prep(d["xte"]).to(device), yte=d["yte"].to(device))


def frac_of(name, n_train):
    spec = DATASETS[name]
    return spec["frac"] if spec["frac"] is not None else min(1.0, CAP_IMAGES / n_train)


def make_partition(ytr, C, K, frac, IF, alpha, seed, min_size=10):
    """Client index lists, as experiments/core.py make_partition (same RandomState calls in the same order).
    IF: long-tail imbalance factor in core's convention (0.01 = ratio 100; 1.0 = balanced); None = natural class
    sizes (each class keeps the fraction frac of its images). Also returns the validation set: 200 images per class
    (2000 // C when C > 10; fewer if a class is small) that no client holds."""
    rng = np.random.RandomState(seed)
    y = ytr.cpu().numpy()
    n_total = int(round(frac * len(y)))
    by_class = [rng.permutation(np.where(y == c)[0]) for c in range(C)]
    if IF is None:
        sizes = np.array([len(bc) for bc in by_class])
        want = np.floor(frac * sizes).astype(int)
    else:
        mu = IF ** (1.0 / (C - 1))
        prof = mu ** np.arange(C)
        want = np.floor(n_total * prof / prof.sum()).astype(int)
    nh = min(200, 2000 // C)
    if IF is None:  # natural imbalance: never take more than a quarter of a small class for validation
        nhs = [min(nh, len(bc) // 4) for bc in by_class]
    else:
        nhs = [nh] * C
    heldout = np.concatenate([bc[:h] for bc, h in zip(by_class, nhs)])
    pools = [bc[h:h + min(want[c], len(bc) - h)] for c, (bc, h) in enumerate(zip(by_class, nhs))]
    if alpha is None:
        allidx = rng.permutation(np.concatenate(pools))
        parts = [p.tolist() for p in np.array_split(allidx, K)]
    else:
        for _ in range(100):
            parts = [[] for _ in range(K)]
            for c in range(C):
                idx = pools[c]
                prop = rng.dirichlet([alpha] * K)
                cuts = (np.cumsum(prop) * len(idx)).astype(int)[:-1]
                for k, chunk in enumerate(np.split(idx, cuts)):
                    parts[k].extend(chunk.tolist())
            if min(len(p) for p in parts) >= min_size:
                break
        while min(len(p) for p in parts) < min_size:
            small = int(np.argmin([len(p) for p in parts]))
            big = int(np.argmax([len(p) for p in parts]))
            j = rng.randint(len(parts[big]))
            parts[small].append(parts[big].pop(j))
        parts = [rng.permutation(p).tolist() for p in parts]
    counts = np.array([[int((y[p] == c).sum()) for c in range(C)] for p in parts])
    return dict(parts=[torch.tensor(p, dtype=torch.long) for p in parts], counts=counts,
                heldout=torch.tensor(heldout, dtype=torch.long), class_counts=counts.sum(0))


def natural_order(ytr, C):
    """For natural-imbalance datasets: relabelling that orders classes from most to least frequent, so that the
    head / medium / tail groups (by label index) mean the same as under the made long tail."""
    n = torch.bincount(ytr, minlength=C).numpy()
    order = np.argsort(-n, kind="stable")
    perm = np.empty(C, dtype=np.int64)
    perm[order] = np.arange(C)
    return torch.from_numpy(perm)


def relabel(data, perm):
    return dict(data, ytr=perm.to(data["ytr"].device)[data["ytr"]], yte=perm.to(data["yte"].device)[data["yte"]])


def permute_labels(data, seed):
    """Random rare classes (sweep): relabel by np.random.RandomState(1000 + seed).permutation(C), as core.py."""
    C = int(data["ytr"].max()) + 1
    perm = torch.from_numpy(np.random.RandomState(1000 + seed).permutation(C)).long()
    return relabel(data, perm)


def assign_arch(K, M=10):
    """Contiguous blocks of clients per architecture (reference GeFL_CVAE-F.py:35)."""
    blk = max(K // M, 1)
    return [min(k // blk, M - 1) for k in range(K)]


def active_clients(K, pfrac, seed, stage, r):
    """Clients taking part in round r of a stage (all, or the same seeded subset for every method), as core.py."""
    if pfrac >= 1.0:
        return list(range(K))
    m = max(1, int(round(pfrac * K)))
    rng = np.random.RandomState((seed * 1000003 + {"i": 1, "ii": 2, "iii": 3}[stage] * 10007 + r) % 2 ** 32)
    return sorted(rng.choice(K, m, replace=False).tolist())


def gpu_augment(x):
    """RandomCrop(32, padding=4) + RandomHorizontalFlip on a batch (as core.py)."""
    B, C, H, W = x.shape
    xp = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (B,), device=x.device)
    j = torch.randint(0, 9, (B,), device=x.device)
    ar = torch.arange(H, device=x.device)
    rows = (i[:, None] + ar)[:, None, :, None].expand(B, C, H, W)
    cols = (j[:, None] + ar)[:, None, None, :].expand(B, C, H, W)
    out = xp[torch.arange(B, device=x.device)[:, None, None, None],
             torch.arange(C, device=x.device)[None, :, None, None], rows, cols]
    flip = torch.rand(B, device=x.device) < 0.5
    return torch.where(flip[:, None, None, None], out.flip(3), out)


def batches(n, B, device):
    perm = torch.randperm(n, device=device)
    for s in range(0, n, B):
        yield perm[s:s + B]


class Federation:
    """One dataset split into clients for one (setting, seed): every client's images and labels on the device,
    the validation set and the test set. Built identically by the server and by the client process."""

    def __init__(self, ds, root, IF, alpha, K, seed, device, tail_perm=False, frac_scale=1.0, clients=True):
        spec = DATASETS[ds]
        self.ds, self.C, self.K, self.seed, self.device = ds, spec["C"], K, seed, device
        data = load_dataset(ds, root, device, lazy=True)
        prep = data.pop("prep")
        to_new = np.arange(self.C)              # original label -> label used here
        if spec["imbalance"] == "natural":
            perm = natural_order(data["ytr"], self.C)
            data = relabel(data, perm)
            to_new = perm.numpy()[to_new]
            IF = None
        if tail_perm:
            data = permute_labels(data, seed)
            to_new = np.random.RandomState(1000 + seed).permutation(self.C)[to_new]   # the permutation it used
        # class_order[c] = original label of class c; class_names in that order (for every per-class table)
        self.class_order = np.argsort(to_new).tolist()
        from class_names import CLASS_NAMES
        names = CLASS_NAMES.get(ds)
        self.class_names = [names[o] for o in self.class_order] if names and len(names) == self.C else None
        if spec.get("partition") == "lt_standard":   # Track 2: the split LongTailLib's baselines get (track2.py)
            from track2 import lt_partition
            parts, heldout, counts = lt_partition(data["ytr"].numpy(), self.C, K, IF, alpha, seed)
            part = dict(parts=[torch.from_numpy(np.asarray(p)).long() for p in parts], counts=counts,
                        heldout=torch.from_numpy(heldout).long(), class_counts=counts.sum(0))
            if frac_scale < 1.0:   # smoke tests only
                part["parts"] = [p[:max(10, int(len(p) * frac_scale))] for p in part["parts"]]
                part["counts"] = np.array([[int((data["ytr"][p] == c).sum()) for c in range(self.C)]
                                           for p in part["parts"]])
                part["class_counts"] = part["counts"].sum(0)
        else:
            frac = frac_of(ds, len(data["ytr"])) * frac_scale
            part = make_partition(data["ytr"], self.C, K, frac, IF, alpha, seed)
        self.counts, self.class_counts = part["counts"], part["class_counts"]
        # the server process keeps no client data (clients=False): only sizes and, for logging, the class counts
        self.x = [prep(data["xtr"][p]).to(device) for p in part["parts"]] if clients else None
        self.y = [data["ytr"][p].to(device) for p in part["parts"]] if clients else None
        self.x_val = prep(data["xtr"][part["heldout"]]).to(device)
        self.y_val = data["ytr"][part["heldout"]].to(device)
        self.x_te, self.y_te = data["xte"].to(device), data["yte"].to(device)
        self.sizes = [len(p) for p in part["parts"]]
        self.arch = [0] * K if spec.get("single_arch") else assign_arch(K)
        self.groups = sorted(set(self.arch))


In [ ]:
%%writefile /kaggle/working/fedecs_lt/fedecs.py
"""FedECS-LT components: everything is computed from exact per-class sums over all clients (secure aggregation)
instead of from averaged models. Formulas are those of experiments/core.py (tests/test_equivalence.py checks them):

  statistics release   n_c, S1_c = sum h, S2_c = sum ||h||^2 per class           (anchor_stats)
  separation gate      F = mean_c ||mu_c - mean mu||^2 / mean_c V_c               (run_one, 'sep')
  mean anchor          generator conditioned on mu_c = S1_c / n_c                  (ProtoCVAE)
  CCWA                 class rows averaged with weights E(n_kc) over holders        (class_weights, Averager)
  moment correction    x' = ReLU(mu_c + s_c (x - m_c)), s_c = sqrt(V_c / V~_c)     (MomentCalibratedGen)
  logit adjustment     logits + tau log p_k (add-one smoothed client prior)         (local_log_prior)
  consensus labels     (1 - beta) onehot(y) + beta mean_g softmax(h_g(x))          (stage_iii, CSL)
  exact readout        class-balanced LDA from per-class sums of embeddings         (lda_readout)
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ---------------------------------------------------------------- statistics release (client side)
@torch.no_grad()
def class_stats(X, Y, C):
    """One client's contribution: count, sum of h and sum of ||h||^2 per class (float64)."""
    Xf = X.flatten(1).double()
    n = torch.bincount(Y, minlength=C).double()
    S1 = torch.zeros(C, Xf.shape[1], dtype=torch.float64, device=X.device).index_add_(0, Y, Xf)
    S2 = torch.zeros(C, dtype=torch.float64, device=X.device).index_add_(0, Y, Xf.pow(2).sum(1))
    return n, S1, S2


def moments_from_sums(n, S1, S2):
    """Exact class means and spreads V_c = E||h - mu_c||^2 from the summed statistics (counts floored at 1, as
    core.anchor_stats)."""
    n = n.clamp(min=1.0)
    mu = S1 / n[:, None]
    V = S2 / n - mu.pow(2).sum(1)
    return n, mu, V


def separation_F(n, S1, S2):
    """Class separation: between-class variance of the exact means over the mean within-class spread."""
    _, mu, V = moments_from_sums(n, S1, S2)
    Vw = V.clamp(min=1e-12)
    return float((mu - mu.mean(0)).pow(2).sum(1).mean() / Vw.mean())


# ---------------------------------------------------------------- class-count weighted averaging (CCWA)
def ccwa_weights(counts_k, beta):
    """E(n) = (1 - beta^n) / (1 - beta): zero for a client without the class, about n for few samples, saturating
    at 1 / (1 - beta) so that one large client cannot dominate a class."""
    c = np.asarray(counts_k, dtype=float)
    return (1 - beta ** c) / (1 - beta)


def ccwa_rows(row_sum, w_sum, prev_rows):
    """Average of class rows from secure sums: sum_k E(n_kc) row_kc / sum_k E(n_kc); a class no participating client
    holds keeps its previous rows."""
    shape = (-1,) + (1,) * (row_sum.dim() - 1)
    rows = row_sum / w_sum.clamp(min=1e-12).view(shape)
    return torch.where((w_sum > 0).view(shape), rows, prev_rows)


# ---------------------------------------------------------------- moment correction (server side)
class MomentCalibratedGen(nn.Module):
    """Sampling-time correction of each class's generator distribution to the exact class mean and spread.
    m_c and V~_c are measured on n samples per class drawn with a fixed generator seed (reproducible)."""

    def __init__(self, G, mu, V, n=500, clip=(0.1, 10.0), seed=0):
        super().__init__()
        self.G, self.kind = G, G.kind
        C = mu.shape[0]
        dev = mu.device
        G.eval()
        gen_state = torch.random.get_rng_state()
        cuda_state = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        torch.manual_seed(seed)
        m, Vg = [], []
        with torch.no_grad():
            for c in range(C):
                g = G.sample(torch.full((n,), c, device=dev, dtype=torch.long)).flatten(1)
                m.append(g.mean(0))
                Vg.append((g - g.mean(0)).pow(2).sum(1).mean())
        torch.random.set_rng_state(gen_state)
        if cuda_state is not None:
            torch.cuda.set_rng_state_all(cuda_state)
        m, Vg = torch.stack(m).float(), torch.stack(Vg).float()
        scale = (V.float().clamp(min=0) / Vg.clamp(min=1e-8)).sqrt().clamp(*clip)
        self.register_buffer("mu", mu.float())
        self.register_buffer("m", m)
        self.register_buffer("scale", scale)

    @torch.no_grad()
    def sample(self, y, z=None):
        x = self.G.sample(y, z)
        shp = x.shape[1:]
        out = self.mu[y] + self.scale[y, None] * (x.flatten(1) - self.m[y])
        return F.relu(out).view(-1, *shp)


class MCParams:
    """The moment-correction parameters sent to clients (they hold the generator) so that they can draw the same
    corrected samples as the server: per-class target mean, generator mean and scale."""

    def __init__(self, mu, m, scale):
        self.mu, self.m, self.scale = mu, m, scale

    @torch.no_grad()
    def apply(self, x, y):
        shp = x.shape[1:]
        return F.relu(self.mu[y] + self.scale[y, None] * (x.flatten(1) - self.m[y])).view(-1, *shp)


# ---------------------------------------------------------------- head losses
def local_log_prior(counts_k, device):
    p = np.asarray(counts_k, dtype=float) + 1.0
    p = p / p.sum()
    return torch.tensor(np.log(p), dtype=torch.float32, device=device)


def client_loss_adj(loss, counts_k, tau, device):
    """FedLC: z_c - tau (n_kc + 1)^(-1/4); FedRS: mask of classes the client lacks."""
    if loss == "lc":
        return torch.tensor(tau * (np.asarray(counts_k, dtype=float) + 1.0) ** -0.25, dtype=torch.float32, device=device)
    if loss == "rs":
        return torch.tensor(np.asarray(counts_k) == 0, device=device)
    return None


def head_loss(logits, y, loss, log_prior=None, tau=1.0, adj=None, rs_alpha=0.5):
    if loss == "lc":
        return F.cross_entropy(logits - adj, y)
    if loss == "rs":
        return F.cross_entropy(torch.where(adj[None, :], logits * rs_alpha, logits), y)
    if loss == "la" and log_prior is not None and tau:
        logits = logits + tau * log_prior
    return F.cross_entropy(logits, y)


# ---------------------------------------------------------------- consensus soft labels
def csl_pool_size(sizes, ts, B, K, n_groups):
    """Synthetic samples each client trains on per round, and the pool layout (as core.stage_iii): one shared pool
    when each architecture has one client, otherwise disjoint slices."""
    need = [ts * (sz // B if sz >= B else 1) * (B if sz >= B else sz) for sz in sizes]
    if K == n_groups:
        return max(need), [0] * K, need
    off = list(np.cumsum([0] + need[:-1]))
    return int(sum(need)), [int(o) for o in off], need


def csl_targets(py, cons, beta, C):
    return (1 - beta) * F.one_hot(py, C).float() + beta * cons


# ---------------------------------------------------------------- exact readout
@torch.no_grad()
def embed_sums(E, Y, C):
    """One client's contribution to the readout of one architecture: counts, sums of e and sum of e e^T."""
    Ed = E.double()
    n = torch.bincount(Y, minlength=C).double()
    S = torch.zeros(C, Ed.shape[1], dtype=torch.float64, device=E.device).index_add_(0, Y, Ed)
    return n, S, Ed.T @ Ed


def lda_from_sums(n, S, M, gamma=0.1):
    """Class-balanced LDA last layer (W, b) from summed statistics, as core.lda_readout; absent classes b = -1e4."""
    C, d = S.shape
    mu = S / n[:, None].clamp(min=1)
    absent = n == 0
    Sw = (M - (mu.T * n) @ mu) / max(float(n.sum()) - int((~absent).sum()), 1.0)
    Sw = (Sw + Sw.T) / 2
    Ss = (1 - gamma) * Sw + (gamma * torch.trace(Sw) / d + 1e-6) * torch.eye(d, dtype=torch.float64, device=S.device)
    # the pseudo-inverse on the CPU (milliseconds for d <= a few hundred): on a nearly full GPU cuSOLVER's syevd failed
    # (CUSOLVER_STATUS_EXECUTION_FAILED, one shared model, Kaggle, 27 Oct 2026); only rounding can differ
    W = mu @ torch.linalg.pinv(Ss.cpu(), hermitian=True).to(Ss.device)
    b = -0.5 * (W * mu).sum(1)
    b[absent] = -1e4
    return W.float(), b.float()


# ---------------------------------------------------------------- Gaussian feature generator (control)
def gaussian_from_sums(n, S, M, shape, shrink=0.1):
    """Shared-covariance class Gaussian from exact sums (as core.stage_ii_gauss)."""
    C, D = S.shape
    mu = S / n.clamp(min=1.0)[:, None]
    W = M - (mu.T * n) @ mu
    Sig = W / max(float(n.sum()) - C, 1.0)
    Sig = (Sig + Sig.T) / 2
    Sig = (1 - shrink) * Sig + shrink * (torch.trace(Sig) / D) * torch.eye(D, device=S.device, dtype=torch.float64)
    evals, evecs = torch.linalg.eigh(Sig)
    evals = evals.clamp(min=1e-6 * float(torch.trace(Sig)) / D)
    return mu.float(), (evecs * evals.sqrt()).float()


In [ ]:
%%writefile /kaggle/working/fedecs_lt/fl_common.py
"""Helpers shared by the Flower server and client: message packing, seeding, cache keys, averaging."""
import hashlib
import json
import random

import numpy as np
import torch
from flwr.app import ArrayRecord, ConfigRecord, Message, RecordDict


def jdump(d):
    return json.dumps(d, sort_keys=True, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))


def key_of(*parts):
    return hashlib.sha1(jdump(parts).encode()).hexdigest()[:16]


def seed_all(*parts):
    """Deterministic seed for one client action (run seed, stage, round, client): results do not depend on the order
    in which the simulation engine executes clients."""
    s = int(key_of(*parts), 16) % (2 ** 31 - 1)
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    return s


def pack(cfg=None, arrays=None, **named_arrays):
    """RecordDict with a JSON config and named ArrayRecords (dicts of tensors)."""
    rd = RecordDict()
    rd["cfg"] = ConfigRecord({"j": jdump(cfg or {})})
    for name, tensors in ({"w": arrays} if arrays is not None else {}).items():
        rd[name] = ArrayRecord({k: v.detach().cpu() for k, v in tensors.items()})
    for name, tensors in named_arrays.items():
        rd[name] = ArrayRecord({k: (v.detach().cpu() if torch.is_tensor(v) else torch.as_tensor(v))
                                for k, v in tensors.items()})
    return rd


def cfg_of(content):
    return json.loads(content["cfg"]["j"])


def arrays_of(content, name="w", device="cpu"):
    if name not in content.array_records:
        return None
    return {k: v.to(device) for k, v in content[name].to_torch_state_dict().items()}


def reply(msg, cfg=None, arrays=None, **named):
    return Message(pack(cfg, arrays, **named), reply_to=msg)


def nbytes(content):
    return sum(len(a.data) for rec in content.array_records.values() for a in rec.values())


class Averager:
    """Unweighted mean of state dicts (as experiments/core.py: every participating client counts once); integer
    buffers (BN num_batches_tracked) take the last client's value."""

    def __init__(self):
        self.sum, self.n = None, 0

    def add(self, sd):
        if self.sum is None:
            self.sum = {k: (v.detach().clone().float() if v.is_floating_point() else v.clone()) for k, v in sd.items()}
        else:
            for k, v in sd.items():
                if v.is_floating_point():
                    self.sum[k] += v
                else:
                    self.sum[k] = v.clone()
        self.n += 1

    def result(self):
        return {k: (v / self.n if v.is_floating_point() else v) for k, v in self.sum.items()}


In [ ]:
%%writefile /kaggle/working/fedecs_lt/flops.py
"""Floating-point operations of a forward pass (convolutions, transposed convolutions and linear layers, the layers
that dominate the cost; 2 FLOPs per multiply-accumulate), counted with forward hooks on one input."""
import torch
import torch.nn as nn


def forward_flops(module, x, *args):
    """FLOPs of module(x, *args) per input row of x."""
    total = [0]

    def hook(m, inp, out):
        if isinstance(m, nn.Conv2d):
            total[0] += 2 * out.numel() * (m.in_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
        elif isinstance(m, nn.ConvTranspose2d):
            total[0] += 2 * inp[0].numel() * (m.out_channels // m.groups) * m.kernel_size[0] * m.kernel_size[1]
        elif isinstance(m, nn.Linear):
            total[0] += 2 * out.numel() * m.in_features
    hs = [m.register_forward_hook(hook) for m in module.modules()
          if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear))]
    modes = [(m, m.training) for m in module.modules()]   # restored one by one (BatchNorm statistics untouched)
    module.eval()
    try:
        with torch.no_grad():
            module(x, *args)
    finally:
        for m, t in modes:
            m.training = t
        for h in hs:
            h.remove()
    return total[0] / x.shape[0]


In [ ]:
%%writefile /kaggle/working/fedecs_lt/htfl_client.py
"""Client side of the model-heterogeneous FL baselines (Track 1): FedProto, FedTGP, FedGen, FedGH, LG-FedAvg, FedMRL.

Ported from HtFLlib (Zhang et al., KDD 2025; https://github.com/TsingZ0/HtFLlib, commit e959261, Apache License 2.0):
system/flcore/clients/client{proto,tgp,gen,gh,lg,mrl}.py. The local updates below follow those files line by line
(losses, what is uploaded); only data loading is replaced by this suite's frozen-feature clients. As in HtFLlib, every
client model is split into base (here: the client's head body from Stage I, followed by HtFLlib's parameter-free
AdaptiveAvgPool1d to the common feature dimension, 512) and head (Linear(512, C)); models are personal and persist on
the client across rounds.
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from client import DEV, STATE, app, fed_of, feats_of, pid_of, spec_of
from fl_common import arrays_of, cfg_of, reply, seed_all
from metrics import class_metrics, log_view
from models import PaperHeader
from settings import HEADER_WIDTHS


class HtModel(nn.Module):
    """HtFLlib BaseHeadSplit for a head body: base = body + AdaptiveAvgPool1d(feature_dim), head = Linear."""

    def __init__(self, widths, in_ch, bn, C, feature_dim=512):
        super().__init__()
        body = PaperHeader(widths, in_ch, bn, C)
        self.body = body.features
        self.pool = nn.AdaptiveAvgPool1d(feature_dim)
        self.head = nn.Linear(feature_dim, C)

    def base(self, x):
        return self.pool(self.body(x).flatten(1))

    def forward(self, x):
        return self.head(self.base(x))


class Generator(nn.Module):
    """FedGen's feature generator as in HtFLlib servergen.py (based on the official FedGen code)."""

    def __init__(self, noise_dim, num_classes, hidden_dim, feature_dim, device):
        super().__init__()
        self.noise_dim, self.num_classes, self.device = noise_dim, num_classes, device
        self.fc1 = nn.Sequential(nn.Linear(noise_dim + num_classes, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU())
        self.fc = nn.Linear(hidden_dim, feature_dim)

    def forward(self, labels):
        eps = torch.rand((labels.shape[0], self.noise_dim), device=self.device)
        z = torch.cat((eps, F.one_hot(labels, self.num_classes)), dim=1)
        return self.fc(self.fc1(z))


def model_of(cfg, k, fed, in_ch, init=None):
    """The client's persistent personal model (and FedMRL's projection), created from the Stage-I head body."""
    key = ("htfl", cfg["run_id"], k)
    if key not in STATE:
        spec = spec_of(cfg)
        g = fed.arch[k]
        seed_all(cfg["seed"], "htfl_init", k)
        m = HtModel(HEADER_WIDTHS[g], in_ch, spec["header_bn"], fed.C, cfg["feature_dim"]).to(DEV)
        if init is not None:  # body weights of the architecture's Stage-I head
            m.body.load_state_dict({kk[len("features."):]: v for kk, v in init.items() if kk.startswith("features.")})
        extra = {}
        if cfg["method"] == "FedMRL":
            extra["proj"] = nn.Linear(cfg["feature_dim"] + cfg["sub_feature_dim"], cfg["feature_dim"]).to(DEV)
        STATE[key] = dict(model=m, **extra)
    return STATE[key]


def agg_protos(reps, ys, C):
    """HtFLlib agg_func: mean representation per class present on the client."""
    protos, mask = torch.zeros(C, reps.shape[1], device=reps.device), torch.zeros(C, device=reps.device)
    for c in torch.unique(ys):
        protos[c] = reps[ys == c].mean(0)
        mask[c] = 1
    return protos, mask


@app.train("htfl")
def htfl_train(msg, context):
    cfg, k = cfg_of(msg.content), pid_of(context)
    fed = fed_of(cfg)
    F_ = feats_of(cfg, fed)
    X, Y = F_["x"][k], fed.y[k]
    st = model_of(cfg, k, fed, X.shape[1], arrays_of(msg.content, "init", DEV))
    model = st["model"]
    meth, C = cfg["method"], fed.C
    srv = arrays_of(msg.content, "srv", DEV) or {}
    out = dict(n=len(Y), counts=np.bincount(Y.cpu().numpy(), minlength=C).tolist())
    # set_parameters: methods that share a head replace the local head with the server's (LG-FedAvg, FedGH, FedGen)
    if meth in ("LG-FedAvg", "FedGH", "FedGen") and "head.weight" in srv:
        model.head.weight.data.copy_(srv["head.weight"])
        model.head.bias.data.copy_(srv["head.bias"])
    if cfg.get("eval"):   # model as received (after the server's aggregation step), before local training
        out["post"] = evaluate_client(cfg, st, F_, fed, srv)
    seed_all(cfg["seed"], "htfl", cfg["round"], k)
    model.train()
    opt = torch.optim.SGD(model.parameters(), lr=cfg["lr"])
    global_protos = srv.get("protos") if meth in ("FedProto", "FedTGP") else None
    gmask = srv.get("protos_mask")
    gen = None
    if meth == "FedGen" and any(n.startswith("gen.") for n in srv):
        gen = Generator(cfg["noise_dim"], C, cfg["hidden_dim"], cfg["feature_dim"], DEV).to(DEV)
        gen.load_state_dict({n[len("gen."):]: v for n, v in srv.items() if n.startswith("gen.")})
        gen.eval()
        qualified = np.array(cfg["qualified_labels"])
    gmodel, proj = None, None
    if meth == "FedMRL":
        gmodel = HtModel(HEADER_WIDTHS[0], X.shape[1], spec_of(cfg)["header_bn"], C, cfg["sub_feature_dim"]).to(DEV)
        gmodel.load_state_dict({n[len("g."):]: v for n, v in srv.items() if n.startswith("g.")})
        gmodel.train()
        proj = st["proj"]
        opt_p = torch.optim.SGD(proj.parameters(), lr=cfg["lr"])
        opt_g = torch.optim.SGD(gmodel.parameters(), lr=cfg["lr"])
    loss_ce, loss_mse = nn.CrossEntropyLoss(), nn.MSELoss()
    proto_reps = []
    losses = []
    for _ in range(cfg["epochs"]):
        # HtFLlib clientbase.load_train_data: DataLoader(batch_size, drop_last=True, shuffle=False) over the client's
        # (already shuffled) data
        for s0 in range(0, len(Y) - cfg["B"] + 1, cfg["B"]):
            x, y = X[s0:s0 + cfg["B"]], Y[s0:s0 + cfg["B"]]
            if meth == "FedMRL":
                rep_g, rep = gmodel.base(x), model.base(x)
                rep_new = proj(torch.cat((rep_g, rep), dim=1))
                loss = loss_ce(model.head(rep_new), y) + loss_ce(gmodel.head(rep_new[:, :cfg["sub_feature_dim"]]), y)
                opt.zero_grad(); opt_p.zero_grad(); opt_g.zero_grad()  # noqa: E702
                loss.backward()
                opt.step(); opt_p.step(); opt_g.step()  # noqa: E702
                losses.append(float(loss.detach()))
                continue
            rep = model.base(x)
            loss = loss_ce(model.head(rep), y)
            if global_protos is not None:     # FedProto / FedTGP: MSE to the global prototype of each sample's class
                proto_new = rep.detach().clone()
                has = gmask[y] > 0
                proto_new[has] = global_protos[y[has]]
                loss = loss + loss_mse(proto_new, rep) * cfg["lamda"]
            if gen is not None:               # FedGen: CE of the head on generated features of qualified labels
                labels = torch.from_numpy(np.random.choice(qualified, cfg["B"])).long().to(DEV)
                z = gen(labels)
                loss = loss + loss_ce(model.head(z), labels)
            if meth == "FedProto":
                proto_reps.append((rep.detach(), y))
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(float(loss.detach()))
    out["loss"] = float(np.mean(losses)) if losses else None
    up = {}
    if meth == "FedProto":   # FedProto: prototypes from the training-time representations
        reps = torch.cat([r for r, _ in proto_reps])
        ys = torch.cat([y for _, y in proto_reps])
        p, mk = agg_protos(reps, ys, C)
        up.update(protos=p, mask=mk)
    elif meth in ("FedTGP", "FedGH"):   # collect_protos: representations of the trained model, eval mode
        model.eval()
        with torch.no_grad():
            reps = torch.cat([model.base(X[s:s + 4096]) for s in range(0, len(Y), 4096)])
        p, mk = agg_protos(reps, Y, C)
        up.update(protos=p, mask=mk)
        model.train()
    if meth in ("LG-FedAvg", "FedGen"):
        up.update({"head.weight": model.head.weight.detach(), "head.bias": model.head.bias.detach()})
    if meth == "FedMRL":
        up.update({f"g.{n}": v.detach() for n, v in gmodel.state_dict().items()})
    if cfg.get("eval"):
        out["pre"] = evaluate_client(cfg, st, F_, fed, srv, gmodel=gmodel)
    return reply(msg, out, up=up)


@torch.no_grad()
def predict(cfg, st, X, srv, gmodel=None):
    """Logits of the client's model. FedProto / FedTGP classify by the nearest global prototype (HtFLlib
    test_metrics: -MSE distance as the score); FedMRL uses the projected joint representation."""
    model = st["model"]
    model.eval()
    meth = cfg["method"]
    if meth in ("FedProto", "FedTGP") and srv.get("protos") is not None:
        P, mk = srv["protos"], srv["protos_mask"]
        out = []
        for s in range(0, len(X), 2048):
            rep = model.base(X[s:s + 2048])
            d = ((rep[:, None, :] - P[None, :, :]) ** 2).mean(-1)
            d = torch.where(mk[None, :] > 0, d, torch.full_like(d, float("inf")))
            out.append(-d)
        model.train()
        return torch.cat(out)
    if meth == "FedMRL" and gmodel is None and any(n.startswith("g.") for n in srv):
        gmodel = HtModel(HEADER_WIDTHS[0], X.shape[1], spec_of(cfg)["header_bn"], model.head.out_features,
                         cfg["sub_feature_dim"]).to(DEV)
        gmodel.load_state_dict({n[len("g."):]: v for n, v in srv.items() if n.startswith("g.")})
    if meth == "FedMRL" and gmodel is not None:
        gmodel.eval()
        out = torch.cat([model.head(st["proj"](torch.cat((gmodel.base(X[s:s + 2048]), model.base(X[s:s + 2048])), 1)))
                         for s in range(0, len(X), 2048)])
        model.train()
        return out
    out = torch.cat([model(X[s:s + 2048]) for s in range(0, len(X), 2048)])
    model.train()
    return out


def evaluate_client(cfg, st, F_, fed, srv, gmodel=None):
    logits = predict(cfg, st, F_["te"], srv, gmodel)
    logits = torch.where(torch.isfinite(logits), logits, torch.full_like(logits, -1e9))
    m = class_metrics(logits, fed.y_te, fed.C)
    return log_view(m)


@app.evaluate("htfl")
def htfl_eval(msg, context):
    """Final evaluation of the client's personal model on the test set (all metrics, probabilities, embeddings)."""
    cfg, k = cfg_of(msg.content), pid_of(context)
    fed = fed_of(cfg)
    F_ = feats_of(cfg, fed)
    st = STATE[("htfl", cfg["run_id"], k)]
    srv = arrays_of(msg.content, "srv", DEV) or {}
    logits = predict(cfg, st, F_["te"], srv)
    logits = torch.where(torch.isfinite(logits), logits, torch.full_like(logits, -1e9))
    out = {}
    prior = fed.counts[k] / max(fed.counts[k].sum(), 1)     # own-mix accuracy: the client's own class mix
    m = class_metrics(logits, fed.y_te, fed.C, prior=prior, keep_probs=cfg.get("probs", False))
    pr = m.pop("probs", None)   # sent as a float16 array (the precision saved), not as a JSON list
    out["test"] = {kk: (v.tolist() if hasattr(v, "tolist") else v) for kk, v in m.items()}
    lv = predict(cfg, st, F_["val"], srv)
    lv = torch.where(torch.isfinite(lv), lv, torch.full_like(lv, -1e9))
    mv = class_metrics(lv, fed.y_val, fed.C, prior=prior)
    out["val"] = {kk: v for kk, v in mv.items() if kk != "probs"}
    if cfg.get("embed"):
        sub = np.random.RandomState(0).choice(len(fed.y_te), min(2000, len(fed.y_te)), replace=False)
        with torch.no_grad():
            st["model"].eval()
            out["embed"] = st["model"].base(F_["te"][torch.from_numpy(sub).to(DEV)]).cpu().numpy().tolist()
            st["model"].train()
        out["embed_y"] = fed.y_te.cpu().numpy()[sub].tolist()
    weights = {f"model.{n}": v for n, v in st["model"].state_dict().items()}
    if "proj" in st:
        weights.update({f"proj.{n}": v for n, v in st["proj"].state_dict().items()})
    if pr is not None:
        return reply(msg, out, model=weights, probs=dict(p=torch.from_numpy(pr).half()))
    return reply(msg, out, model=weights)


In [ ]:
%%writefile /kaggle/working/fedecs_lt/htfl_server.py
"""Server side of the model-heterogeneous FL baselines (Track 1), ported from HtFLlib (Apache License 2.0):
system/flcore/servers/server{proto,tgp,gen,gh,lg,mrl}.py. Every method starts from the same Stage-I extractor and head
bodies as FedECS-LT (cached), trains for the same number of rounds and local epochs as our head stage, and uses the
hyperparameters of the authors' run scripts (HtFLlib README / total.sh): SGD lr 0.01, batch 10, feature dimension 512;
FedProto and FedTGP lambda 10; FedTGP server epochs 100, margin threshold 100; FedGen noise 32, hidden 512, generator
lr 0.1, server epochs 100; FedGH server lr 0.01, server epochs 1; FedMRL sub-feature dimension 128."""
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from data import active_clients
from fl_common import arrays_of, cfg_of, pack, seed_all
from flops import forward_flops
from metrics import summarize
from monitor import Monitor
from server import DEV, EVAL_EVERY, Fed, flat, gpu_peaks, save_run, stage_one, train_referee
from settings import HEADER_WIDTHS

HP = dict(lr=0.01, B=10, feature_dim=512)
METHOD_HP = {
    "FedProto": dict(lamda=10.0),
    "FedTGP": dict(lamda=10.0, server_epochs=100, margin_threthold=100.0),
    "FedGen": dict(noise_dim=32, hidden_dim=512, generator_learning_rate=0.1, server_epochs=100),
    "FedGH": dict(server_learning_rate=0.01, server_epochs=1),
    "LG-FedAvg": dict(),
    "FedMRL": dict(sub_feature_dim=128),
}


class TrainableGlobalPrototypes(nn.Module):
    """FedTGP's Trainable_Global_Prototypes (HtFLlib servertgp.py)."""

    def __init__(self, num_classes, server_hidden_dim, feature_dim):
        super().__init__()
        self.embedings = nn.Embedding(num_classes, feature_dim)
        self.middle = nn.Sequential(nn.Sequential(nn.Linear(feature_dim, server_hidden_dim), nn.ReLU()))
        self.fc = nn.Linear(server_hidden_dim, feature_dim)

    def forward(self, class_id):
        return self.fc(self.middle(self.embedings(class_id)))


def proto_cluster(protos, masks):
    """Unweighted mean over clients of each class's prototype (HtFLlib proto_aggregation / proto_cluster)."""
    P = torch.stack(protos)
    M = torch.stack(masks)
    cnt = M.sum(0)
    avg = (P * M[:, :, None]).sum(0) / cnt.clamp(min=1)[:, None]
    return avg, (cnt > 0).float()


def htfl_flops(S, hp, meth, HtModel):
    """Forward FLOPs per image of every client's model (FedMRL: with its shared small model and projection) and the
    expected training FLOPs of one round over all clients (3 x forward; the local epochs over full batches)."""
    from models import make_fe
    fe = make_fe(S.spec).to(DEV)
    f_fe = forward_flops(fe, torch.zeros(2, S.spec["in_ch"], 32, 32, device=DEV))
    with torch.no_grad():
        shape = tuple(fe(torch.zeros(1, S.spec["in_ch"], 32, 32, device=DEV)).shape[1:])
    f_head = {}
    for g in S.groups:
        f = forward_flops(HtModel(HEADER_WIDTHS[g], shape[0], S.spec["header_bn"], S.C, hp["feature_dim"]).to(DEV),
                          torch.zeros(2, *shape, device=DEV))
        if meth == "FedMRL":
            f += forward_flops(HtModel(HEADER_WIDTHS[0], shape[0], S.spec["header_bn"], S.C,
                                       hp["sub_feature_dim"]).to(DEV), torch.zeros(2, *shape, device=DEV))
            f += 2 * (hp["feature_dim"] + hp["sub_feature_dim"]) * hp["feature_dim"]
        f_head[str(g)] = f
    per_round = sum(3 * f_head[str(S.arch[k])] * hp["epochs"] * (int(S.fed.sizes[k]) // hp["B"]) * hp["B"]
                    for k in range(S.K))
    return dict(flops_fe=f_fe, flops_head=f_head, head_stage_gflops_per_round=S.R.get("pfrac", 1.0) * per_round / 1e9)


def run_htfl(grid, context, R):
    from htfl_client import Generator, HtModel
    mon = Monitor(os.path.join(R["out_dir"], "logs"), R["run_id"], tensorboard=R.get("tensorboard", True))
    t0 = time.time()
    S = Fed(grid, context, R, mon)
    meth = R["method"]["name"]
    spec = S.spec
    T = dict(T_FE=spec["T_FE"], T_KA=spec["T_KA"], T_TN=spec["T_TN"])
    T.update(R.get("rounds") or {})
    hp = dict(HP, **METHOD_HP[meth], epochs=5)
    res = dict(class_order=S.fed.class_order, class_names=S.fed.class_names,
               test_class_counts=np.bincount(S.fed.y_te.cpu().numpy(), minlength=S.C).tolist(),
               run_id=R["run_id"], label=R["label"], dataset=R["ds"], IF=R["IF"], alpha=R["alpha"], K=R["K"],
               seed=R["seed"], pfrac=R.get("pfrac", 1.0), tail_perm=R.get("tail_perm", False), method=R["method"],
               n_train=S.N, min_client=int(min(S.fed.sizes)), client_counts=S.fed.counts.astype(int).tolist(),
               rounds=T, framework="flower", hyperparameters=hp)
    mon.log("run_start", **{k: v for k, v in res.items() if k != "client_counts"})
    feat_key, s1 = stage_one(S, T["T_FE"])
    res["fe_time_s"] = s1["time_s"]
    S.ask("query.features", [(S.nid[k], pack(S.base(fe_key=feat_key), fe=s1["fe_sd"])) for k in range(S.K)])
    ref, Xval = train_referee(S, s1["fe_sd"])
    in_ch = Xval.shape[1]
    C = S.C
    seed_all(R["seed"], "htfl_server")
    srv = {}
    TGP = gen = gmodel = None
    qualified = None
    if meth == "FedTGP":
        TGP = TrainableGlobalPrototypes(C, hp["feature_dim"], hp["feature_dim"]).to(DEV)
    if meth == "FedGen":
        gen = Generator(hp["noise_dim"], C, hp["hidden_dim"], hp["feature_dim"], DEV).to(DEV)
        # qualified labels: every client's samples per class (HtFLlib FedGen.__init__ reads them from the clients)
        counts = S.fed.counts.sum(0)
        qualified = [int(c) for c in range(C) for _ in range(int(counts[c]))]
        srv.update({f"gen.{n}": v for n, v in gen.state_dict().items()})
    if meth in ("LG-FedAvg", "FedGH", "FedGen"):
        seed_all(R["seed"], "htfl_init", 0)   # HtFLlib: the server's head starts as client 0's initial head
        h0 = HtModel(HEADER_WIDTHS[S.arch[0]], in_ch, spec["header_bn"], C, hp["feature_dim"]).to(DEV)
        srv.update({"head.weight": h0.head.weight.detach(), "head.bias": h0.head.bias.detach()})
    if meth == "FedMRL":
        gmodel = HtModel(HEADER_WIDTHS[0], in_ch, spec["header_bn"], C, hp["sub_feature_dim"]).to(DEV)
        srv.update({f"g.{n}": v for n, v in gmodel.state_dict().items()})
    base_cfg = S.base(fe_key=feat_key, method=meth, **hp)
    if qualified is not None:
        base_cfg["qualified_labels"] = qualified
    history = []
    mon.start("stage3")
    for r in range(T["T_TN"]):
        act = active_clients(S.K, R.get("pfrac", 1.0), R["seed"], "iii", r)
        ev = r == 0 or (r + 1) % EVAL_EVERY == 0 or r == T["T_TN"] - 1
        items = []
        for k in act:
            named = dict(srv=srv) if srv else {}
            if r == 0:
                named["init"] = s1["hd_sd"][S.arch[k]]
            items.append((S.nid[k], pack(dict(base_cfg, round=r, eval=ev), **named)))
        rep = S.ask("train.htfl", items)
        ups, infos = [], []
        for m in rep:
            k = S.pid[m.metadata.src_node_id]
            ups.append((k, arrays_of(m.content, "up", DEV)))
            infos.append((k, cfg_of(m.content)))
        w = np.array([i["n"] for _, i in infos], dtype=float)
        w = w / w.sum()
        rec = dict(round=r + 1, clients=act, loss={str(k): i["loss"] for k, i in infos})
        if meth in ("FedProto", "FedTGP", "FedGH"):
            protos = [u["protos"] for _, u in ups]
            masks = [u["mask"] for _, u in ups]
        if meth == "FedProto":
            P, M = proto_cluster(protos, masks)
            srv = dict(protos=P, protos_mask=M)
        elif meth == "FedTGP":
            avg, Mk = proto_cluster(protos, masks)
            # class-wise minimum distance between averaged prototypes (receive_protos)
            gap = torch.full((C,), 1e9, device=DEV)
            ks = [c for c in range(C) if Mk[c] > 0]
            for a in ks:
                for b in ks:
                    if a > b:
                        dis = torch.norm(avg[a] - avg[b], p=2)
                        gap[a] = torch.min(gap[a], dis)
                        gap[b] = torch.min(gap[b], dis)
            min_gap = torch.min(gap)
            gap = torch.where(gap > 1e8, min_gap, gap)
            max_gap = torch.max(gap)
            data = [(p[c], c) for p, mk in zip(protos, masks) for c in range(C) if mk[c] > 0]
            opt = torch.optim.SGD(TGP.parameters(), lr=hp["lr"])
            TGP.train()
            for _ in range(hp["server_epochs"]):
                for proto, y in DataLoader(data, hp["B"], drop_last=False, shuffle=True):
                    y = torch.as_tensor(y, dtype=torch.int64, device=DEV)
                    proto_gen = TGP(torch.arange(C, device=DEV))
                    dist = (proto.pow(2).sum(1, keepdim=True) - 2 * proto @ proto_gen.T
                            + proto_gen.pow(2).sum(1, keepdim=True).T).sqrt()
                    margin = min(max_gap.item(), hp["margin_threthold"])
                    dist = dist + F.one_hot(y, C) * margin
                    loss = F.cross_entropy(-dist, y)
                    opt.zero_grad()
                    loss.backward()
                    opt.step()
            TGP.eval()
            with torch.no_grad():
                srv = dict(protos=TGP(torch.arange(C, device=DEV)).detach(), protos_mask=torch.ones(C, device=DEV))
            rec["server_loss"] = float(loss)
        elif meth == "FedGH":
            data = [(p[c], c) for p, mk in zip(protos, masks) for c in range(C) if mk[c] > 0]
            head = nn.Linear(hp["feature_dim"], C).to(DEV)
            head.weight.data.copy_(srv["head.weight"])
            head.bias.data.copy_(srv["head.bias"])
            opt = torch.optim.SGD(head.parameters(), lr=hp["server_learning_rate"])
            for _ in range(hp["server_epochs"]):
                for p, y in DataLoader(data, hp["B"], drop_last=False, shuffle=True):
                    loss = F.cross_entropy(head(p), torch.as_tensor(y, dtype=torch.int64, device=DEV))
                    opt.zero_grad()
                    loss.backward()
                    opt.step()
            srv = {"head.weight": head.weight.detach(), "head.bias": head.bias.detach()}
        elif meth in ("LG-FedAvg", "FedGen"):
            heads = [{n: u[n] for n in ("head.weight", "head.bias")} for _, u in ups]
            if meth == "FedGen":   # train_generator on the uploaded heads (weights by training samples), then average
                gen.train()
                gopt = torch.optim.Adam(gen.parameters(), lr=hp["generator_learning_rate"], betas=(0.9, 0.999),
                                        eps=1e-08, weight_decay=0, amsgrad=False)
                lin = []
                for h in heads:
                    lm = nn.Linear(hp["feature_dim"], C).to(DEV)
                    lm.weight.data.copy_(h["head.weight"])
                    lm.bias.data.copy_(h["head.bias"])
                    lm.eval()
                    lin.append(lm)
                for _ in range(hp["server_epochs"]):
                    labels = torch.from_numpy(np.random.choice(qualified, hp["B"])).long().to(DEV)
                    z = gen(labels)
                    logits = sum(lm(z) * wi for lm, wi in zip(lin, w))
                    loss = F.cross_entropy(logits, labels)
                    gopt.zero_grad()
                    loss.backward()
                    gopt.step()
                rec["server_loss"] = float(loss)
            agg = {n: sum(h[n] * wi for h, wi in zip(heads, w)) for n in ("head.weight", "head.bias")}
            srv = dict(agg)
            if meth == "FedGen":
                srv.update({f"gen.{n}": v for n, v in gen.state_dict().items()})
        elif meth == "FedMRL":
            gs = [{n: u[n] for n in u if n.startswith("g.")} for _, u in ups]
            srv = {n: sum(g[n].float() for g in gs) / len(gs) if gs[0][n].is_floating_point() else gs[-1][n]
                   for n in gs[0]}
        if ev:
            rec["pre"] = {str(k): i["pre"] for k, i in infos if "pre" in i}
            rec["post_clients"] = {str(k): i["post"] for k, i in infos if "post" in i}
            pre = [i["pre"] for _, i in infos if "pre" in i]
            if pre:   # mean over the clients' personal models of every scalar and every per-class metric
                rec["post"] = {}
                for k in pre[0]:
                    vals = [p[k] for p in pre if p.get(k) is not None]
                    if not vals:
                        continue
                    if isinstance(vals[0], list):
                        rec["post"][k] = [round(float(x), 4) for x in np.nanmean(np.array(vals, dtype=float), 0)]
                    else:
                        rec["post"][k] = round(float(np.mean(vals)), 4)
        mon.log("stage3_round", **rec)
        history.append(rec)
    mon.stop("stage3")
    # final evaluation: each client's personal model on the test and validation sets
    rep = S.ask("evaluate.htfl", [(S.nid[k], pack(dict(base_cfg, probs=True, embed=(k == 0)), srv=srv))
                                  for k in range(S.K)], account=False)
    per = {}
    probs, emb = {}, None
    personal = {}
    for m in rep:
        k = S.pid[m.metadata.src_node_id]
        o = cfg_of(m.content)
        per[k] = o
        personal[k] = arrays_of(m.content, "model", "cpu")
        pr = arrays_of(m.content, "probs", "cpu")
        if pr is not None:
            probs[str(k)] = pr["p"].numpy()
        if "embed" in o:
            emb = (o["embed"], o["embed_y"])
    for split, prefix in (("test", "final"), ("val", "val_final")):
        summ = summarize({k: v[split] for k, v in per.items()})
        res.update(flat(prefix, summ))
    res["final_per_client"] = {str(k): {kk: v["test"][kk] for kk in ("acc", "bal", "f1", "tail", "worst", "local")
                                        if kk in v["test"]}
                               for k, v in per.items()}
    res["time_s"] = time.time() - t0
    res["stage_time_s"] = dict(mon.stage_t)
    res["bytes"] = {k: dict(down=v[0], up=v[1], messages=v[2]) for k, v in mon.bytes.items()}
    res["comm_mb_per_client"] = sum(v[0] + v[1] for v in mon.bytes.values()) / 2 ** 20 / S.K
    res.update(gpu_peaks(S))
    res.update(htfl_flops(S, hp, meth, HtModel))
    fin = dict(test=dict(probs={k: v for k, v in probs.items()}, embed={"0": emb[0]} if emb else {},
                         embed_y=emb[1] if emb else []))
    extra = dict(y_full=S.fed.y_te.cpu().numpy())
    if "head.weight" in personal.get(0, {}):   # client 0's classifier (its embeddings are the saved ones)
        extra["fcw_0"] = np.asarray(personal[0]["head.weight"], dtype=np.float32)
        extra["fcb_0"] = np.asarray(personal[0]["head.bias"], dtype=np.float32)
    save_run(R, res, s1, personal, None, ref, fin, None, None, None, extra)
    mon.log("run_end", time_s=res["time_s"])
    mon.close()
    return res


In [ ]:
%%writefile /kaggle/working/fedecs_lt/local_train.py
"""One client's local training steps, exactly as experiments/core.py's inner loops (tests/test_local_steps.py checks
weight-for-weight equality with core.stage_i, core.stage_ii and core.stage_iii for one client and one round)."""
import numpy as np
import torch
import torch.nn.functional as F

import fedecs
from data import batches, gpu_augment


def local_stage1(fe, head, x_k, y_k, spec, device, prior=None, epochs=5, lr=0.1, momentum=0.0):
    """Stage I: SGD(0.1) on the extractor and the client's head jointly, 5 local epochs (T_w). Track 2 passes the
    FedLTLib protocol instead (10 epochs, momentum 0.9, decayed learning rate)."""
    fe.train(), head.train()
    opt = torch.optim.SGD(list(fe.parameters()) + list(head.parameters()), lr=lr, momentum=momentum)
    losses = []
    for _ in range(epochs):
        for b in batches(len(y_k), spec["B"], device):
            if b.numel() < 2:
                continue
            x, y = x_k[b], y_k[b]
            if spec["augment"]:
                x = gpu_augment(x)
            logits = head(fe(x))
            loss = F.cross_entropy(logits + prior if prior is not None else logits, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.detach()))
    return losses


def local_gen(G, opt, X, Y, device, B=64, epochs=5):
    """Stage II: 5 local epochs of the generator loss with the client's persistent optimiser."""
    G.train()
    losses = []
    for _ in range(epochs):
        for b in batches(len(Y), B, device):
            if b.numel() < 2:
                continue
            x, y = X[b], Y[b]
            graphs = getattr(G, "graphs", None)
            loss = graphs(("loss", len(y)), G, G.loss, x, y) if (graphs is not None and len(y) == B) else G.loss(x, y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            losses.append(float(loss.detach()) / len(y))
    return losses


def syn_hard_labels(C, n):
    """GeFL-F: uniform synthetic labels drawn with NumPy's global generator (as core.stage_iii)."""
    return torch.from_numpy(np.random.choice(C, size=n, p=np.full(C, 1.0 / C))).long()


def probe_head(h, X, Y, C):
    """The head on the client's own real features, without changing it (eval mode, no gradients, so BatchNorm
    statistics and the random streams are untouched): logit spread (mean over images of the standard deviation of
    the logits over classes) and balanced accuracy over the classes the client holds."""
    was = h.training
    h.eval()
    with torch.no_grad():
        lg = torch.cat([h(X[s:s + 2000]) for s in range(0, len(X), 2000)]).float()
    h.train(was)
    pred = lg.argmax(1)
    rec = [float((pred[Y == c] == c).float().mean()) for c in range(C) if (Y == c).any()]
    return dict(logit_spread=float(lg.std(1).mean()), bal=float(np.mean(rec)) if rec else None)


def local_head(h, fwd, X, Y, B, hs, device, prior=None, adj=None, syn=None, lr=0.1, momentum=0.0, probe_C=None):
    """Stage III: synthetic phase (one pass, batch bs) then real phase (tr epochs, batch B), fresh SGD(0.1) each.
    syn: None, ("hard", xs, ylab) or ("soft", xs, targets). probe_C: number of classes; when given, the head is
    probed on the client's real features at the start, after the synthetic phase and at the end (probe_head);
    returned as a third value."""
    n = len(Y)
    nb, bs = (n // B, B) if n >= B else (1, n)
    syn_losses, real_losses = [], []
    probes = {}
    if probe_C:
        probes["start"] = probe_head(h, X, Y, probe_C)
    if syn is not None:
        kind, xs, t = syn
        opt = torch.optim.SGD(h.parameters(), lr=lr, momentum=momentum)
        for s in range(0, len(xs), bs):
            if kind == "soft":
                loss = -(t[s:s + bs] * fwd(xs[s:s + bs]).log_softmax(1)).sum(1).mean()
            else:
                loss = F.cross_entropy(fwd(xs[s:s + bs]), t[s:s + bs])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            syn_losses.append(float(loss.detach()))
        if probe_C:
            probes["after_syn"] = probe_head(h, X, Y, probe_C)
    opt = torch.optim.SGD(h.parameters(), lr=lr, momentum=momentum)
    for _ in range(hs["tr"]):
        for b in batches(n, B, device):
            if b.numel() < 2:
                continue
            loss = fedecs.head_loss(fwd(X[b]), Y[b], hs["loss"], prior, hs.get("tau", 1.0), adj)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            real_losses.append(float(loss.detach()))
    if probe_C:
        probes["end"] = probe_head(h, X, Y, probe_C)
        return syn_losses, real_losses, probes
    return syn_losses, real_losses


def protocol_lr(spec, global_round):
    """(learning rate, momentum) of a local update: GeFL-F's SGD 0.1 without momentum (Track 1), or the FedLTLib
    protocol (Track 2: 0.1 x 0.99^round, momentum 0.9)."""
    o = spec.get("opt")
    if not o:
        return 0.1, 0.0
    return o["lr"] * o.get("decay", 1.0) ** global_round, o.get("momentum", 0.0)


def n_syn(n, B, ts):
    nb, bs = (n // B, B) if n >= B else (1, n)
    return ts * nb * bs


In [ ]:
%%writefile /kaggle/working/fedecs_lt/metrics.py
"""Every classification metric of action plan section 14, computed from one model's test logits.

Definitions follow experiments/core.py where it defined them (recall, precision, macro-F1, G-mean with recalls
floored at 0.01, ECE with 15 equal-width bins, NLL with probabilities floored at 1e-6, Brier); the others use the
standard definitions (scikit-learn for AUC-ROC and average precision; tests/test_metrics.py checks the rest against
scikit-learn). Multi-architecture summaries average over architectures (each architecture counts once), as core.py.
"""
import numpy as np
import torch

from settings import class_groups

SCALARS = ("acc", "bal", "precision", "f1", "mcc", "kappa", "gmean", "worst", "head", "medium", "tail", "rec_std",
           "auc_roc", "auc_pr", "top5", "ece", "cw_ece", "nll", "brier", "local")
# Per-class versions of every metric that has one (the class-averaged metrics above are their means, except NLL and
# Brier, which average over images): recall, precision, F1, average precision (AUC-PR), AUC-ROC and MCC of class c
# against the rest, calibration error of p(c|x), NLL and Brier score on the images of class c, top-5 recall, and the
# share of all predictions that go to class c (pred_share_c against the true share shows the bias towards frequent
# classes).
PER_CLASS = ("recall_c", "precision_c", "f1_c", "ap_c", "auc_c", "mcc_c", "ece_c", "nll_c", "brier_c", "top5_c",
             "pred_share_c")


def _ece(conf, correct, bins=15):
    edges = np.linspace(0, 1, bins + 1)
    e = 0.0
    for i in range(bins):
        sel = (conf > edges[i]) & (conf <= edges[i + 1])
        if sel.any():
            e += sel.mean() * abs(conf[sel].mean() - correct[sel].mean())
    return float(e)


def _classwise_ece(prob, y, C, bins=15):
    """Class-wise ECE (Kull et al., 2019): per class, calibration of p(c|x) against 1[y = c], averaged over classes."""
    return float(np.mean(_ece_per_class(prob, y, C, bins)))


def _ece_per_class(prob, y, C, bins=15):
    return np.array([_ece(prob[:, c], (y == c).astype(float), bins) for c in range(C)])


def class_metrics(logits, y, C, prior=None, keep_probs=False):
    """All metrics of one model. logits (N, C) tensor, y (N,) tensor. prior: the model's own clients' label
    distribution (accuracy re-weighted to it = what its own clients would see)."""
    from sklearn.metrics import average_precision_score, roc_auc_score
    prob_t = logits.float().softmax(1)
    pred_t = prob_t.argmax(1)
    prob, pred, yy = prob_t.cpu().numpy().astype(np.float64), pred_t.cpu().numpy(), y.cpu().numpy()
    N = len(yy)
    cm = np.zeros((C, C), dtype=np.int64)
    np.add.at(cm, (yy, pred), 1)
    tp = np.diag(cm).astype(float)
    ny = cm.sum(1).astype(float)
    npred = cm.sum(0).astype(float)
    rec = tp / np.maximum(ny, 1)
    prec = tp / np.maximum(npred, 1)
    f1_c = 2 * tp / np.maximum(ny + npred, 1)
    acc = float(tp.sum() / N)
    # MCC and Cohen's kappa (multiclass, from the confusion matrix)
    t, p, c_, s = ny, npred, tp.sum(), float(N)
    mcc_den = np.sqrt((s ** 2 - (p ** 2).sum()) * (s ** 2 - (t ** 2).sum()))
    mcc = float((c_ * s - (t * p).sum()) / mcc_den) if mcc_den > 0 else 0.0
    pe = float((t * p).sum() / s ** 2)
    kappa = float((acc - pe) / (1 - pe)) if pe < 1 else 0.0
    present = ny > 0
    onehot = np.eye(C)[yy]
    try:
        auc_roc = float(roc_auc_score(onehot[:, present], prob[:, present], average="macro"))
    except ValueError:
        auc_roc = float("nan")
    ap_c = np.array([average_precision_score(onehot[:, c], prob[:, c]) if present[c] else np.nan for c in range(C)])
    auc_c = np.array([roc_auc_score(onehot[:, c], prob[:, c]) if present[c] and ny[c] < N else np.nan
                      for c in range(C)])
    k5 = min(5, C)
    in5 = (np.argsort(-prob, 1)[:, :k5] == yy[:, None]).any(1)
    top5 = float(in5.mean())
    # per class: MCC of "class c" against the rest; NLL, Brier and top-5 recall on the images of class c
    fp, fn = npred - tp, ny - tp
    tn = N - tp - fp - fn
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    mcc_c = np.where(den > 0, (tp * tn - fp * fn) / np.where(den > 0, den, 1), 0.0)
    nll_i = -np.log(np.clip(prob[np.arange(N), yy], 1e-6, None))
    brier_i = ((prob - onehot) ** 2).sum(1)
    by_class = [yy == c for c in range(C)]
    nll_c = np.array([nll_i[s].mean() if s.any() else np.nan for s in by_class])
    brier_c = np.array([brier_i[s].mean() if s.any() else np.nan for s in by_class])
    top5_c = np.array([in5[s].mean() if s.any() else np.nan for s in by_class])
    conf = prob.max(1)
    hd, md, tl = class_groups(C)
    out = dict(
        acc=acc, bal=float(rec[present].mean()), precision=float(prec.mean()), f1=float(f1_c.mean()), mcc=mcc,
        kappa=kappa, gmean=float(np.exp(np.log(np.clip(rec[present], 0.01, None)).mean())), worst=float(rec[present].min()),
        head=float(rec[hd].mean()), medium=float(rec[md].mean()), tail=float(rec[tl].mean()), rec_std=float(rec.std()),
        auc_roc=auc_roc, auc_pr=float(np.nanmean(ap_c)), top5=top5,
        ece=_ece(conf, (pred == yy).astype(float)), cw_ece=_classwise_ece(prob, yy, C),
        nll=float(nll_i.mean()), brier=float(brier_i.mean()),
        recall_c=rec.tolist(), precision_c=prec.tolist(), f1_c=f1_c.tolist(), ap_c=ap_c.tolist(), auc_c=auc_c.tolist(),
        mcc_c=mcc_c.tolist(), ece_c=_ece_per_class(prob, yy, C).tolist(), nll_c=nll_c.tolist(),
        brier_c=brier_c.tolist(), top5_c=top5_c.tolist(), pred_share_c=(npred / N).tolist(), confusion=cm.tolist())
    if prior is not None:
        out["local"] = float((np.asarray(prior, dtype=float) * rec).sum())
    if keep_probs:
        out["probs"] = prob.astype(np.float32)
    return out


def log_view(m, digits=4):
    """What a per-round evaluation logs: every scalar and every per-class metric (rounded; no confusion matrix or
    probabilities, which only the final evaluation keeps)."""
    out = {k: (round(float(m[k]), digits) if m[k] is not None and not np.isnan(m[k]) else None)
           for k in SCALARS + ("worst_arch", "ens_acc", "ens_bal") if k in m}
    for k in PER_CLASS:
        if k in m:
            out[k] = [None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), digits)
                      for v in m[k]]
    return out


def summarize(per_arch, ens=None):
    """Mean over architectures of every scalar; class-wise vectors averaged; balanced accuracy of the mean recall
    (as core.summarize_eval); worst_arch = balanced accuracy of the weakest architecture."""
    vals = list(per_arch.values())
    out = {}
    for k in SCALARS:
        xs = [v[k] for v in vals if k in v and v[k] is not None and not (isinstance(v[k], float) and np.isnan(v[k]))]
        if xs:
            out[k] = float(np.mean(xs))
    rec = np.mean([v["recall_c"] for v in vals], 0)
    C = len(rec)
    hd, md, tl = class_groups(C)
    out.update(bal=float(rec.mean()), head=float(rec[hd].mean()), medium=float(rec[md].mean()),
               tail=float(rec[tl].mean()), worst=float(rec.min()), rec_std=float(rec.std()),
               worst_arch=float(min(np.mean(v["recall_c"]) for v in vals)), recall_c=rec.tolist())
    for k in PER_CLASS[1:]:
        if all(k in v for v in vals):
            out[k] = np.nanmean([v[k] for v in vals], 0).tolist()
    out["confusion"] = np.sum([v["confusion"] for v in vals], 0).tolist()
    if ens is not None:
        out["ens_acc"], out["ens_bal"] = ens["acc"], ens["bal"]
    return out


@torch.no_grad()
def evaluate_heads(heads, groups, X, y, C, priors=None, keep_probs=False, embed=False):
    """Per-architecture metrics of the heads on features X (chunks of 2000), plus the averaged-softmax ensemble."""
    per, prob_sum = {}, None
    emb = {}
    for g in groups:
        h = heads[g]
        was = h.training
        h.eval()
        logits = torch.cat([h(X[s:s + 2000]) for s in range(0, len(y), 2000)])
        if embed:
            emb[g] = torch.cat([h.embed(X[s:s + 2000]) for s in range(0, len(y), 2000)]).cpu().numpy()
        h.train(was)
        prob_sum = logits.softmax(1) if prob_sum is None else prob_sum + logits.softmax(1)
        per[g] = class_metrics(logits, y, C, prior=None if priors is None else priors.get(g), keep_probs=keep_probs)
    ens = None
    if len(groups) > 1:
        e = class_metrics(prob_sum.log(), y, C)
        ens = dict(acc=e["acc"], bal=e["bal"])
    return per, summarize(per, ens), emb


def arch_priors(counts, arch, groups):
    out = {}
    for g in groups:
        own = counts[[k for k in range(len(arch)) if arch[k] == g]].sum(0).astype(np.float64)
        out[g] = own / max(own.sum(), 1.0)
    return out


In [ ]:
%%writefile /kaggle/working/fedecs_lt/models.py
"""Networks of the GeFL-F pipeline (copied from experiments/core.py so results pair with the earlier runs):
the frozen feature extractor (FE), the ten client head architectures CNN-1..CNN-10, the CVAE-F generator, the
mean-anchored PC-VAE generator of FedECS-LT and the Gaussian feature generator; plus CUDA-graph replay for speed."""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from settings import HEADER_WIDTHS, N_ARCH


class PaperFE(nn.Module):
    """GeFL-F Tables XXI-XXIII: 3x16x16 features (MNIST-type), 10x16x16 (CIFAR-type)."""

    def __init__(self, in_ch, variant):
        super().__init__()
        if variant == "mnist":
            layers = [nn.Conv2d(in_ch, 3, 3, 1, 1), nn.BatchNorm2d(3), nn.ReLU(inplace=True)]
            self.out_ch = 3
        else:
            layers = [nn.Conv2d(in_ch, 3, 3, 1, 1), nn.BatchNorm2d(3), nn.ReLU(inplace=True),
                      nn.Conv2d(3, 10, 3, 1, 1), nn.BatchNorm2d(10), nn.ReLU(inplace=True)]
            self.out_ch = 10
        self.net = nn.Sequential(*layers, nn.MaxPool2d(2, 2))

    def forward(self, x):
        return self.net(x)


class PaperHeader(nn.Module):
    """Client head CNN-g: conv blocks of HEADER_WIDTHS[g] (body) and a linear last layer fc."""

    def __init__(self, widths, in_ch, bn, C, S=16):
        super().__init__()
        layers, ci, s = [], in_ch, S
        for w in widths:
            layers.append(nn.Conv2d(ci, w, 3, 1, 1))
            if bn:
                layers.append(nn.BatchNorm2d(w))
            layers += [nn.ReLU(inplace=True), nn.MaxPool2d(2, 2)]
            ci, s = w, s // 2
        self.features = nn.Sequential(*layers)
        self.fc = nn.Linear(ci * s * s, C)
        self.emb_dim = ci * s * s

    def embed(self, h):
        return self.features(h).flatten(1)

    def forward(self, h):
        return self.fc(self.embed(h))


def build_headers(in_ch, bn, C, device):
    return [PaperHeader(HEADER_WIDTHS[g], in_ch, bn, C).to(device) for g in range(N_ARCH)]


class PaperCVAE(nn.Module):
    """CVAE-F (GeFL-F Table XX). Class identity enters only through label_proj (C x 512), one row per class."""
    kind = "cvae"
    cond = {"label_proj": 0}

    def __init__(self, in_ch, latent, C, S=16):
        super().__init__()
        n = int(round(math.log2(S)))
        widths = [min(64 * 2 ** i, 512) for i in range(n)]
        enc, ci = [], in_ch + 1
        for w in widths:
            enc += [nn.Conv2d(ci, w, 4, 2, 1), nn.BatchNorm2d(w), nn.ReLU(inplace=True)]
            ci = w
        self.enc = nn.Sequential(*enc)
        self.flat = widths[-1]
        self.latent = latent
        self.mu, self.logvar = nn.Linear(self.flat, latent), nn.Linear(self.flat, latent)
        self.dec_z = nn.Linear(latent, self.flat)
        self.label_proj = nn.Parameter(torch.empty(C, self.flat))
        b = 1.0 / math.sqrt(latent + C)
        for p in (self.dec_z.weight, self.dec_z.bias, self.label_proj):
            nn.init.uniform_(p, -b, b)
        dec, rev = [], list(reversed(widths))
        for i, w in enumerate(rev):
            co = rev[i + 1] if i + 1 < len(rev) else in_ch
            dec.append(nn.ConvTranspose2d(w, co, 4, 2, 1))
            if i + 1 < len(rev):
                dec += [nn.BatchNorm2d(co), nn.ReLU(inplace=True)]
        self.dec = nn.Sequential(*dec)

    def decode(self, z, y):
        h = F.relu(self.dec_z(z) + self.label_proj[y]).view(-1, self.flat, 1, 1)
        return F.relu(self.dec(h))

    def encode(self, x, y):
        ymap = y.float().view(-1, 1, 1, 1).expand(-1, 1, x.size(2), x.size(3))
        h = self.enc(torch.cat([x, ymap], 1)).flatten(1)
        return self.mu(h), self.logvar(h)

    def loss(self, x, y):
        ymap = y.float().view(-1, 1, 1, 1).expand(-1, 1, x.size(2), x.size(3))
        h = self.enc(torch.cat([x, ymap], 1)).flatten(1)
        mu, logvar = self.mu(h), self.logvar(h)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        rec = self.decode(z, y)
        return F.mse_loss(rec, x, reduction="sum") - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    @torch.no_grad()
    def sample(self, y, z=None):
        z = torch.randn(y.size(0), self.latent, device=y.device) if z is None else z
        return self.decode(z, y)


class ProtoCVAE(nn.Module):
    """PC-VAE, the mean-anchored generator of FedECS-LT: CVAE-F without class rows; the class enters through its
    exact federated mean mu_c (from secure-aggregated sums) and a class-shared projection,
    x = ReLU(mu_y + dec(z, mu_y)). No class-specific parameter exists, so averaging cannot shrink a class."""
    kind = "cvae"
    cond = {}

    def __init__(self, in_ch, latent, proto, S=16):
        super().__init__()
        n = int(round(math.log2(S)))
        widths = [min(64 * 2 ** i, 512) for i in range(n)]
        enc, ci = [], 2 * in_ch
        for w in widths:
            enc += [nn.Conv2d(ci, w, 4, 2, 1), nn.BatchNorm2d(w), nn.ReLU(inplace=True)]
            ci = w
        self.enc = nn.Sequential(*enc)
        self.flat, self.latent = widths[-1], latent
        self.mu, self.logvar = nn.Linear(self.flat, latent), nn.Linear(self.flat, latent)
        self.dec_z = nn.Linear(latent, self.flat)
        self.proto_proj = nn.Linear(proto[0].numel(), self.flat)
        dec, rev = [], list(reversed(widths))
        for i, w in enumerate(rev):
            co = rev[i + 1] if i + 1 < len(rev) else in_ch
            dec.append(nn.ConvTranspose2d(w, co, 4, 2, 1))
            if i + 1 < len(rev):
                dec += [nn.BatchNorm2d(co), nn.ReLU(inplace=True)]
        self.dec = nn.Sequential(*dec)
        self.register_buffer("proto", proto.clone())
        self.register_buffer("zm", torch.zeros(latent))
        self.register_buffer("zL", torch.eye(latent))

    def decode(self, z, y):
        p = self.proto[y]
        h = F.relu(self.dec_z(z) + self.proto_proj(p.flatten(1))).view(-1, self.flat, 1, 1)
        return F.relu(p + self.dec(h))

    def encode(self, x, y):
        h = self.enc(torch.cat([x, self.proto[y]], 1)).flatten(1)
        return self.mu(h), self.logvar(h)

    def loss(self, x, y):
        mu, logvar = self.encode(x, y)
        z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        rec = self.decode(z, y)
        return F.mse_loss(rec, x, reduction="sum") - 0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())

    @torch.no_grad()
    def sample(self, y, z=None):
        z = torch.randn(y.size(0), self.latent, device=y.device) if z is None else z
        return self.decode(self.zm + z @ self.zL.T, y)


class GaussFeatureGen(nn.Module):
    """Class-conditional Gaussian with the pooled within-class covariance, from exact sums (control method)."""
    kind = "gauss"
    cond = {}

    def __init__(self, mu, chol, shape):
        super().__init__()
        self.register_buffer("mu", mu)
        self.register_buffer("chol", chol)
        self.shape = tuple(shape)

    @torch.no_grad()
    def sample(self, y, z=None):
        eps = torch.randn(len(y), self.mu.shape[1], device=self.mu.device) if z is None else z
        return (self.mu[y] + eps @ self.chol.T).clamp(min=0).view(len(y), *self.shape)


def make_generator(kind, in_ch, latent, C, proto=None):
    if kind == "pcvae":
        return ProtoCVAE(in_ch, latent, proto)
    if kind == "cvae":
        return PaperCVAE(in_ch, latent, C)
    raise ValueError(kind)


# ---------------------------------------------------------------- CUDA graphs (as core.py: same maths, ~2.6x faster)
class _Call(nn.Module):
    def __init__(self, fn, owner):
        super().__init__()
        self.owner = owner
        self.fn = fn

    def forward(self, *a):
        return self.fn(*a)


class GraphCache:
    """Records forward+backward of full-size batches once as a CUDA graph and replays it; other shapes run eagerly."""

    def __init__(self, enabled):
        self.store, self.enabled = {}, enabled

    def __call__(self, key, module, fn, *args):
        if not self.enabled or not module.training:
            return fn(*args)
        g = self.store.get(key)
        if g is None:
            saved = {k: v.detach().clone() for k, v in module.state_dict().items()}
            g = torch.cuda.make_graphed_callables(_Call(fn, module), tuple(a.detach().clone() for a in args))
            module.load_state_dict(saved)  # undo BN-statistics updates from the warm-up passes
            self.store[key] = g
        return g(*args)


def sd_clone(m):
    return {k: v.detach().clone() for k, v in m.state_dict().items()}


def sd_to_numpy(sd):
    """State dict -> (names, numpy arrays) for Flower ArrayRecords; float16/32 kept, integer buffers kept."""
    return {k: v.detach().cpu().numpy() for k, v in sd.items()}


def sd_from_numpy(arrs, device):
    return {k: torch.from_numpy(v.copy()).to(device) for k, v in arrs.items()}


def n_params(m):
    return sum(p.numel() for p in m.parameters())


# ---------------------------------------------------------------- Track 2: ResNet-8 (FedLTLib / CReFF protocol)
class BasicBlock(nn.Module):
    """Standard CIFAR ResNet basic block (He et al., 2016), as in the ResNet-8 of CReFF / LongTailLib."""

    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.down = None
        if stride != 1 or cin != cout:
            self.down = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + (x if self.down is None else self.down(x)))


class ResNet8FE(nn.Module):
    """ResNet-8 (one block per stage, widths 64 / 128 / 256, i.e. scaling 4) up to the second stage: the shared,
    frozen feature extractor of FedECS-LT in Track 2; output 128 x 16 x 16."""

    def __init__(self, in_ch=3, scaling=4):
        super().__init__()
        w = 16 * scaling
        self.net = nn.Sequential(nn.Conv2d(in_ch, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU(inplace=True),
                                 BasicBlock(w, w), BasicBlock(w, 2 * w, 2))
        self.out_ch = 2 * w

    def forward(self, x):
        return self.net(x)


class ResNet8Head(nn.Module):
    """The rest of ResNet-8 (third stage, global average pooling, linear layer): the head of every client."""

    def __init__(self, in_ch, C, scaling=4):
        super().__init__()
        w = 64 * scaling
        self.features = nn.Sequential(BasicBlock(in_ch, w, 2), nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Linear(w, C)
        self.emb_dim = w

    def embed(self, h):
        return self.features(h).flatten(1)

    def forward(self, h):
        return self.fc(self.embed(h))


def make_fe(spec):
    return ResNet8FE(spec["in_ch"]) if spec["fe"] == "resnet8" else PaperFE(spec["in_ch"], spec["fe"])


def make_head(spec, g, in_ch, C):
    if spec.get("heads") == "resnet8":
        return ResNet8Head(in_ch, C)
    return PaperHeader(HEADER_WIDTHS[g], in_ch, spec["header_bn"], C)


def make_headers(spec, in_ch, C, device):
    """All architectures' heads (Track 1: CNN-1..CNN-10; Track 2: one ResNet-8 head)."""
    n = 1 if spec.get("heads") == "resnet8" else N_ARCH
    return [make_head(spec, g, in_ch, C).to(device) for g in range(n)]


In [ ]:
%%writefile /kaggle/working/fedecs_lt/monitor.py
"""Training monitor: every round's signals as JSON lines (one file per run), the bytes of every message type, time
per stage and GPU memory; TensorBoard scalars when the tensorboard package is available."""
import json
import os
import time

import numpy as np
import torch


def _clean(v):
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    if isinstance(v, np.ndarray):
        return v.tolist()
    if torch.is_tensor(v):
        return v.detach().cpu().tolist()
    if isinstance(v, dict):
        return {str(k): _clean(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):
        return [_clean(x) for x in v]
    return v


class Monitor:
    def __init__(self, log_dir, run_id, tensorboard=True):
        os.makedirs(log_dir, exist_ok=True)
        safe = "".join(ch if ch.isalnum() or ch in "-_.=" else "_" for ch in run_id)
        self.path = os.path.join(log_dir, safe + ".jsonl")
        self.f = open(self.path, "a")
        self.bytes = {}          # message type -> [bytes server->client, bytes client->server, messages]
        self.stage_t = {}
        self._t = {}
        self.tb = None
        if tensorboard:
            try:
                from torch.utils.tensorboard import SummaryWriter
                self.tb = SummaryWriter(os.path.join(log_dir, "tb", safe))
            except Exception:
                self.tb = None

    def log(self, event, **fields):
        rec = dict(t=round(time.time(), 3), event=event, **_clean(fields))
        self.f.write(json.dumps(rec) + "\n")
        self.f.flush()
        if self.tb is not None and "round" in fields:
            step = int(fields["round"])
            for k, v in fields.items():
                if isinstance(v, (int, float)) and not isinstance(v, bool) and k != "round":
                    self.tb.add_scalar(f"{event}/{k}", v, step)

    def count(self, mtype, down, up, n=1):
        b = self.bytes.setdefault(mtype, [0, 0, 0])
        b[0] += down
        b[1] += up
        b[2] += n

    def start(self, stage):
        self._t[stage] = time.time()

    def stop(self, stage):
        self.stage_t[stage] = self.stage_t.get(stage, 0.0) + time.time() - self._t.pop(stage)

    def close(self):
        self.f.close()
        if self.tb is not None:
            self.tb.close()


In [ ]:
%%writefile /kaggle/working/fedecs_lt/pack.py
"""Packs one experiment's outputs into the three files a helper sends back:
  <name>__results.zip  every figure (PDF, PNG, CSV), every metric (runs.jsonl, runs.csv), every log, failures
  <name>__report.pdf   the readable report (report.py)
  <name>__models.zip   trained weights (feature extractor, heads, generator, referee, readout) for later re-plotting
  <name>__summary.zip  small (a few MB), to send when the others are too big to upload: runs.jsonl / runs.csv, every
                       per-round log, the audit, failures and the data (CSV) of every figure; no predictions, images,
                       TensorBoard files or weights
Called after every run, so a timeout never loses finished work."""
import csv
import json
import os
import shutil
import sys
import zipfile


def runs_csv(res_dir):
    p = os.path.join(res_dir, "runs.jsonl")
    if not os.path.exists(p):
        return
    rows = [json.loads(line) for line in open(p) if line.strip()]
    keys = sorted({k for r in rows for k, v in r.items() if isinstance(v, (int, float, str)) and not isinstance(v, bool)})
    with open(os.path.join(res_dir, "runs.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(keys)
        for r in rows:
            w.writerow([r.get(k, "") for k in keys])


def zip_dir(src, dst, skip=()):
    tmp = dst + ".part"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for root, _, files in os.walk(src):
            if any(s in root for s in skip):
                continue
            for fn in files:
                p = os.path.join(root, fn)
                z.write(p, os.path.relpath(p, os.path.dirname(src)))
    os.replace(tmp, dst)


def summary_zip(res, dst):
    """The small results file: every metric and log as text, no predictions, images or TensorBoard files."""
    tmp = dst + ".part"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        for root, _, files in os.walk(res):
            rel = os.path.relpath(root, res)
            if rel.split(os.sep)[0] in ("predictions", "tb") or os.sep + "tb" in root:
                continue
            for fn in files:
                if fn.endswith((".jsonl", ".json", ".csv", ".txt")):
                    p = os.path.join(root, fn)
                    z.write(p, os.path.relpath(p, os.path.dirname(res)))
    os.replace(tmp, dst)


def pack(work, name, title, report=True, out_dir=None, models=True):
    """work holds results/ and models/; the three files go to out_dir (default: work)."""
    out_dir = out_dir or work
    os.makedirs(out_dir, exist_ok=True)
    res = os.path.join(work, "results")
    mdir = os.path.join(work, "models")
    os.makedirs(res, exist_ok=True)
    runs_csv(res)
    info = None
    if report:
        sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
        import report as rep
        import audit
        figs = os.path.join(res, "figures")
        shutil.rmtree(figs, ignore_errors=True)
        audit.audit(res, figures=False)          # runs and logs; the report shows its findings
        try:
            info = rep.build(res, title, os.path.join(out_dir, f"{name}__report.pdf"), figs)
        except Exception:                        # the results are still zipped: the report can be rebuilt from them
            import traceback
            err = traceback.format_exc()
            print("REPORT FAILED (results are still packed):\n" + err, flush=True)
            with open(os.path.join(res, "report_error.txt"), "w") as f:
                f.write(err)
            info = dict(report_error=err[-500:])
        a = audit.audit(res)                     # again with the figures the report has drawn
        info = dict(info or {}, audit_problems=a["n_problems"])
        with open(os.path.join(res, "report_info.json"), "w") as f:   # checked by the CPU tests
            json.dump(info, f)
    zip_dir(res, os.path.join(out_dir, f"{name}__results.zip"))
    summary_zip(res, os.path.join(out_dir, f"{name}__summary.zip"))
    if models and os.path.isdir(mdir):
        zip_dir(mdir, os.path.join(out_dir, f"{name}__models.zip"))
    return info


def quick(work, name, out_dir):
    """After every run: results.zip only (fast), under a lock so parallel workers do not collide."""
    import fcntl
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(work, ".pack.lock"), "w") as lk:
        fcntl.flock(lk, fcntl.LOCK_EX)
        runs_csv(os.path.join(work, "results"))
        zip_dir(os.path.join(work, "results"), os.path.join(out_dir, f"{name}__results.zip"))
        summary_zip(os.path.join(work, "results"), os.path.join(out_dir, f"{name}__summary.zip"))


if __name__ == "__main__":
    # python pack.py WORK NAME "TITLE" OUT_DIR
    print(pack(sys.argv[1], sys.argv[2], sys.argv[3], out_dir=sys.argv[4] if len(sys.argv) > 4 else None))


In [ ]:
%%writefile /kaggle/working/fedecs_lt/plot_style.py
"""IEEE figure rules (action plan section 21), applied to every plot; nothing else sets fonts or sizes.

Sizes are the printed sizes: one column 3.5 in (88.9 mm), two columns 7.16 in (181.9 mm). Text is Times (or a
Times-compatible font: Liberation Serif, Nimbus Roman, TeX Gyre Termes, or matplotlib's own STIXGeneral) at 8 pt; lines at
least 0.5 pt; fonts embedded as TrueType (no Type 3). Every method has one fixed colour, marker and line style, chosen
to be distinguishable by colour-blind readers and in greyscale print. Figures are saved as vector PDF (dense bitmaps
at 600 dpi) together with a PNG preview and a CSV of the plotted data.
"""
import csv
import os
import zlib

import numpy as np

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib import font_manager  # noqa: E402

COLUMN_IN, PAGE_IN = 3.5, 7.16
FONT_PT = 8
MIN_LINE_PT = 0.5

# Times, or a metric-compatible clone; STIXGeneral (a Times design shipped with matplotlib) is always available.
_SERIFS = ["Times New Roman", "Times", "Liberation Serif", "Nimbus Roman", "Nimbus Roman No9 L", "TeX Gyre Termes",
           "STIXGeneral"]


def _serif():
    have = {f.name for f in font_manager.fontManager.ttflist}
    for name in _SERIFS:
        if name in have:
            return name
    return "STIXGeneral"


def apply():
    plt.rcParams.update({
        "font.family": "serif", "font.serif": [_serif()] + _SERIFS, "mathtext.fontset": "stix",
        "font.size": FONT_PT, "axes.labelsize": FONT_PT, "axes.titlesize": FONT_PT, "xtick.labelsize": FONT_PT,
        "ytick.labelsize": FONT_PT, "legend.fontsize": FONT_PT, "figure.titlesize": FONT_PT,
        "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6, "xtick.minor.width": 0.5,
        "ytick.minor.width": 0.5, "lines.linewidth": 1.0, "lines.markersize": 3.5, "patch.linewidth": 0.5,
        "grid.linewidth": 0.5, "grid.alpha": 0.35, "axes.grid": False, "legend.frameon": False,
        "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.dpi": 600, "figure.dpi": 150,
        "axes.spines.top": False, "axes.spines.right": False, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
        "image.cmap": "viridis",
    })


# Colour-blind-safe palette (Okabe and Ito, 2008) plus greys; markers and dashes make lines distinct in greyscale.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#882255", "#000000", "#F0E442", "#7F7F7F",
           "#56B4E9", "#44AA99", "#117733", "#332288", "#AA4499", "#999933", "#DDCC77", "#661100", "#6699CC"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "<", ">", "h", "*", "p", "d", "8", "H", "1", "2", "3"]
DASHES = ["-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1)), (0, (1, 1)), (0, (5, 2, 1, 2))]
OURS = "FedECS-LT"
# Order of the methods in tables and legends: the main comparison, the ablations, then the one-shared-model baselines
ORDER = ["FedECS-LT", "FedECS-LT (200 extractor rounds)", "GeFL-F", "Balanced Softmax", "FedLC", "FedRS", "FedDF", "Local",
         "FedProto", "FedTGP", "FedGen", "FedGH", "LG-FedAvg", "FedMRL",
         "GeFL-F w/o generator", "GeFL-F + LA", "GeFL-F w/ Gaussian generator + LA", "FedDF + LA",
         "FedAvg", "FedProx", "MOON", "CCVR", "CReFF", "FedETF", "Fed-GraB", "FedLoGe", "FedYoYo", "RUCR", "CLIP2FL",
         "FedNH", "FedVLS"]
# Colour and marker index of every method (18 of each): unique among the methods one report can show together
# (main comparison 0-12, ablations 13-16 with 0-1, the variant 17; the one-shared-model baselines avoid 0, 1, 3, 8
# and 17, the indices of FedECS-LT, GeFL-F, FedLC, FedTGP and the variant, which share reports with them)
STYLE_INDEX = {m: i for i, m in enumerate(["FedECS-LT", "GeFL-F", "Balanced Softmax", "FedLC", "FedRS", "FedDF",
                                           "Local", "FedProto", "FedTGP", "FedGen", "FedGH", "LG-FedAvg", "FedMRL",
                                           "GeFL-F w/o generator", "GeFL-F + LA",
                                           "GeFL-F w/ Gaussian generator + LA", "FedDF + LA",
                                           "FedECS-LT (200 extractor rounds)"])}
STYLE_INDEX.update({"FedAvg": 2, "FedProx": 4, "MOON": 5, "CCVR": 6, "CReFF": 7, "FedETF": 9, "Fed-GraB": 10,
                    "FedLoGe": 11, "FedYoYo": 12, "RUCR": 13, "CLIP2FL": 14, "FedNH": 15, "FedVLS": 16})


def style(method):
    """Fixed (colour, marker, line style) for a method; ours is always the first colour, solid, circle."""
    variant = 0
    for v, suffix in enumerate((" (LongTailLib)",), 1):   # same colour as the method, other dashes
        if method.endswith(suffix) and method[:-len(suffix)] in STYLE_INDEX:
            method, variant = method[:-len(suffix)], v
    # unknown names: a stable index (str hash() changes between processes)
    i = STYLE_INDEX[method] if method in STYLE_INDEX else zlib.crc32(method.encode()) % len(PALETTE)
    return dict(color=PALETTE[i % len(PALETTE)], marker=MARKERS[i % len(MARKERS)],
                linestyle=DASHES[(i + 3 * variant) % len(DASHES)] if method != OURS or variant else "-",
                linewidth=1.4 if method == OURS else 1.0, zorder=5 if method == OURS else 2)


def figure(width="column", height=2.4, ncols=1, nrows=1, **kw):
    apply()
    w = COLUMN_IN if width == "column" else PAGE_IN if width == "page" else float(width) / 25.4
    return plt.subplots(nrows, ncols, figsize=(w, height), **kw)


def panel_labels(axes, y=-0.32):
    """(a), (b), ... below each panel (IEEE subfigure convention). y is a first guess; enforce() moves each label
    just below everything its panel draws (tick and axis labels), whatever their length."""
    for i, ax in enumerate(list(axes)):
        t = ax.text(0.5, y, f"({chr(97 + i)})", transform=ax.transAxes, ha="center", va="top", fontsize=FONT_PT)
        t.set_gid("panel_label")


def room_between_rows(fig, axes, ncols, gap_px=4, tries=12):
    """Grows the space between rows of panels (and the figure with it) until everything a panel draws below itself,
    rotated tick labels and its panel label included, ends above the next row (long method names otherwise run into
    the panel below)."""
    rows = [axes[i:i + ncols] for i in range(0, len(axes), ncols)]
    if len(rows) < 2:
        return
    for _ in range(tries):
        fig.canvas.draw()
        _place_panel_labels(fig)
        r = fig.canvas.get_renderer()
        clash = False
        for upper, lower in zip(rows, rows[1:]):
            low = min(min(ax.get_tightbbox(r).y0 for ax in upper),
                      min((t.get_window_extent(r).y0 for ax in upper for t in ax.texts
                           if t.get_gid() == "panel_label" and t.get_visible()), default=float("inf")))
            if low < max(ax.get_window_extent(r).y1 for ax in lower) + gap_px:
                clash = True
        if not clash:
            return
        h = fig.subplotpars.hspace
        fig.set_figheight(fig.get_figheight() * 1.12)
        fig.subplots_adjust(hspace=h * 1.25)


def _place_panel_labels(fig):
    from matplotlib.text import Text
    r = fig.canvas.get_renderer()
    for ax in fig.axes:
        labels = [t for t in ax.texts if isinstance(t, Text) and t.get_gid() == "panel_label"]
        if not labels:
            continue
        for t in labels:
            t.set_visible(False)
        lowest = ax.get_tightbbox(r).y0                    # display units, without the panel label itself
        y = ax.transAxes.inverted().transform((0, lowest - 4))[1]
        for t in labels:
            t.set_visible(True)
            t.set_y(y)


def enforce(fig):
    """IEEE minimums on every figure, whatever the plotting code set: text at least FONT_PT (8 pt), drawn lines at
    least MIN_LINE_PT (0.5 pt). Returns the number of artists changed."""
    from matplotlib.lines import Line2D
    from matplotlib.text import Text
    n = 0
    for t in fig.findobj(Text):
        if t.get_visible() and t.get_text().strip() and t.get_fontsize() < FONT_PT - 1e-6:
            t.set_fontsize(FONT_PT)
            n += 1
    for ln in fig.findobj(Line2D):
        if ln.get_visible() and ln.get_linestyle() not in ("None", "none", "") and 0 < ln.get_linewidth() < MIN_LINE_PT:
            ln.set_linewidth(MIN_LINE_PT)
            n += 1
    from matplotlib.ticker import AutoLocator, MaxNLocator
    for ax in fig.axes:   # rounds are whole numbers
        if ax.get_xlabel() == "Round" and isinstance(ax.xaxis.get_major_locator(), AutoLocator):
            ax.xaxis.set_major_locator(MaxNLocator(integer=True, min_n_ticks=1))   # also with one round
    place_legends(fig)
    fit_width(fig)
    _place_panel_labels(fig)
    return n


def fit_width(fig, tol=0.01):
    """Keep the saved figure at its nominal width (3.5 or 7.16 in): when tick labels, legends or colour bars reach
    past the figure's edges, the axes are narrowed (text keeps its 8 pt size) until everything fits."""
    W = fig.get_figwidth()
    r = fig.canvas.get_renderer()
    for _ in range(8):
        tb = fig.get_tightbbox(r)
        if tb.width <= W + tol:
            return
        a, b = -tb.x0 / W + tol / W, 1 - (tb.x1 - W) / W - tol / W   # where [0, 1] of the axes' span must map to
        for ax in fig.axes:
            pos = ax.get_position()
            ax.set_position([a + pos.x0 * (b - a), pos.y0, pos.width * (b - a), pos.height])


WIDTH_PROBLEMS = []   # (figure name, saved width in inches, nominal width) of every figure wider than it should be


def log_axis(ax, values, axis="x"):
    """A logarithmic axis when the positive values span at least a factor of 10, else a linear one: on a narrower log
    axis matplotlib labels the minor ticks (1.3e2, 1.35e2, ...), which then overlap. Minor ticks are never labelled.
    Returns True when the axis is logarithmic (the caption says so only then)."""
    from matplotlib.ticker import NullFormatter
    v = np.asarray([x for x in values if x is not None and np.isfinite(x) and x > 0], dtype=float)
    if len(v) == 0 or v.max() < 10 * v.min():
        return False
    (ax.set_xscale if axis == "x" else ax.set_yscale)("log")
    getattr(ax, axis + "axis").set_minor_formatter(NullFormatter())
    return True


def points_under(ax, leg, r):
    """Plotted points (line vertices, scatter points) of an axes that its legend covers."""
    from matplotlib.collections import PathCollection
    lb = leg.get_window_extent(r)
    n = 0
    xys = []
    for ln in ax.get_lines():
        if not ln.get_visible() or not len(ln.get_xydata()):
            continue
        xy = ln.get_transform().transform(ln.get_xydata())
        if ln.get_linestyle() not in ("None", "none", "") and len(xy) > 1:   # the segments too, not only the vertices
            t = np.linspace(0, 1, 25)[:, None]
            xy = np.concatenate([a + t * (b - a) for a, b in zip(xy[:-1], xy[1:])])
        xys.append(xy)
    xys += [c.get_offset_transform().transform(c.get_offsets()) for c in ax.collections
            if isinstance(c, PathCollection) and c.get_visible() and len(c.get_offsets())]
    for xy in xys:
        n += int(((xy[:, 0] > lb.x0) & (xy[:, 0] < lb.x1) & (xy[:, 1] > lb.y0) & (xy[:, 1] < lb.y1)).sum())
    return n


def place_legends(fig):
    """Legends never cover data: one with more than 6 entries goes below the axes (under the tick and axis labels)
    with as many columns as fit the figure's width; a bar chart's short legend goes in one row above the axes."""
    from matplotlib.container import BarContainer
    r = fig.canvas.get_renderer()
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None or leg.get_gid() == "keep":   # placed on purpose by the figure (e.g. inside a scatter plot)
            continue
        handles, labels = leg.legend_handles, [t.get_text() for t in leg.texts]
        bars = any(isinstance(c, BarContainer) for c in ax.containers)
        too_wide = leg.get_window_extent(r).width > 0.98 * fig.get_figwidth() * fig.dpi
        if len(labels) <= 6 and not bars and not too_wide and not points_under(ax, leg, r):
            continue
        if len(labels) <= 6 and not bars and not too_wide:   # first the free place inside the axes, if there is one
            leg.remove()
            leg = ax.legend(handles, labels, loc="best", frameon=False, markerscale=leg.markerscale)
            fig.canvas.draw()
            if not points_under(ax, leg, r):
                continue
        kw = dict(frameon=False, borderaxespad=0.0, handlelength=2.2, columnspacing=1.2,
                  markerscale=leg.markerscale)
        leg.remove()
        box = ax.get_window_extent(r)
        if len(labels) <= 6 and bars and not too_wide:
            ax.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=len(labels), **kw)
            continue
        tight = ax.get_tightbbox(r)
        fig_w, fig_h = fig.get_figwidth() * fig.dpi, fig.get_figheight() * fig.dpi
        y = tight.y0 / fig_h - 0.02            # just under the axes' tick and axis labels, in figure coordinates
        x = 0.5 if len(fig.axes) == 1 else (box.x0 + box.x1) / 2 / fig_w
        # widest column count whose legend still fits inside the figure's width (centred on the figure)
        for ncol in range(min(6, len(labels)), 0, -1):
            new = ax.legend(handles, labels, loc="upper center", bbox_to_anchor=(x, y), bbox_transform=fig.transFigure,
                            ncol=ncol, **kw)
            if new.get_window_extent(r).width <= 0.98 * fig_w or ncol == 1:
                break
            new.remove()


LAYOUT_PROBLEMS = []   # (figure name, problem) found by layout_problems on every saved figure


def _inter(a, b):
    w = min(a.x1, b.x1) - max(a.x0, b.x0)
    h = min(a.y1, b.y1) - max(a.y0, b.y0)
    return w * h if w > 0 and h > 0 else 0.0


def _text_polygon(t, bb, r):
    """Corners (pixels) of the rotated rectangle a text occupies: its unrotated size, rotated by its angle about the
    centre of its (axis-aligned) window box."""
    import numpy as np
    ang = t.get_rotation()
    if ang % 180 == 0:
        return np.array([[bb.x0, bb.y0], [bb.x1, bb.y0], [bb.x1, bb.y1], [bb.x0, bb.y1]])
    t.set_rotation(0)
    flat = t.get_window_extent(r)
    t.set_rotation(ang)
    w, h = flat.width, flat.height
    c = np.array([(bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2])
    a = np.radians(ang)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    corners = np.array([[-w / 2, -h / 2], [w / 2, -h / 2], [w / 2, h / 2], [-w / 2, h / 2]])
    return corners @ R.T + c


def _polys_overlap(P, Q, shrink=0.9):
    """Separating-axis test of two convex quadrilaterals, each shrunk about its centre (so touching is no overlap)."""
    import numpy as np
    P = (P - P.mean(0)) * shrink + P.mean(0)
    Q = (Q - Q.mean(0)) * shrink + Q.mean(0)
    for poly in (P, Q):
        for i in range(len(poly)):
            e = poly[(i + 1) % len(poly)] - poly[i]
            n = np.array([-e[1], e[0]])
            a, b = P @ n, Q @ n
            if a.max() < b.min() or b.max() < a.min():
                return False
    return True


def layout_problems(fig):
    """What a reader would see as broken, measured on the drawn figure: (1) two pieces of text that overlap (more than
    10% of the smaller one's box); (2) text that runs into another panel's plotting area or a colour bar; (3) a legend
    placed inside the axes that covers plotted points. Returns a list of problem strings (empty = clean)."""
    from matplotlib.text import Text
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    # tick labels exist for ticks outside the view limits too; only the drawn ones count
    all_ticklabels, drawn = set(), set()
    for ax in fig.axes:
        for axis in (ax.xaxis, ax.yaxis):
            for tk in axis.get_major_ticks() + axis.get_minor_ticks():
                all_ticklabels.update((id(tk.label1), id(tk.label2)))
            for tk in axis._update_ticks():
                drawn.update(id(lb) for lb in (tk.label1, tk.label2) if lb.get_visible())
    texts = []
    for t in fig.findobj(Text):
        if id(t) in all_ticklabels and id(t) not in drawn:
            continue
        if t.get_visible() and t.get_text().strip() and t.get_alpha() != 0:
            try:
                bb = t.get_window_extent(r)
            except Exception:
                continue
            if bb.width > 0 and bb.height > 0:
                texts.append((t, bb))
    polys = [_text_polygon(t, bb, r) for t, bb in texts]
    out = []
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            (ta, a), (tb, b) = texts[i], texts[j]
            if _inter(a, b) > 0 and _polys_overlap(polys[i], polys[j]):
                out.append(f"text '{ta.get_text()[:30]}' overlaps '{tb.get_text()[:30]}'")
    axes_boxes = [(ax, ax.patch.get_window_extent(r)) for ax in fig.axes if ax.get_visible()]
    for t, bb in texts:        # text of one panel inside another panel's plotting area (or a colour bar)
        own = t.axes
        if own is None:
            continue
        if any(ax is not own and _inter(bb, box) > 0.10 * bb.width * bb.height for ax, box in axes_boxes):
            out.append(f"text '{t.get_text()[:30]}' runs into another panel")
    for ax in fig.axes:
        leg = ax.get_legend()
        if leg is None:
            continue
        lb = leg.get_window_extent(r)
        if _inter(lb, ax.patch.get_window_extent(r)) < 0.5 * lb.width * lb.height:
            continue          # legend outside the axes
        inside = points_under(ax, leg, r)
        if inside:
            out.append(f"legend covers {inside} plotted points")
    return sorted(set(out))


def save(fig, out_dir, name, data_rows=None, header=None, raster=False):
    """Save PDF (vector; 600 dpi when raster), PNG preview and the plotted data as CSV. Returns the PDF path."""
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.join(out_dir, name)
    for ax in fig.axes:
        ax.set_title("")  # no titles inside images: the caption says what the figure shows
    enforce(fig)
    nominal = fig.get_figwidth()
    width = fig.get_tightbbox(fig.canvas.get_renderer()).width   # inches, as saved with bbox_inches="tight"
    if width > nominal + 0.02:
        WIDTH_PROBLEMS.append((name, round(width, 2), nominal))
    LAYOUT_PROBLEMS.extend((name, p) for p in layout_problems(fig))
    fig.savefig(base + ".pdf", dpi=600 if raster else None, metadata={"Creator": None, "Producer": None,
                                                                      "CreationDate": None})
    fig.savefig(base + ".png", dpi=200)
    plt.close(fig)
    if data_rows is not None:
        with open(base + ".csv", "w", newline="") as f:
            w = csv.writer(f)
            if header:
                w.writerow(header)
            w.writerows(data_rows)
    return base + ".pdf"


def check_pdf(path):
    """IEEE check of a saved figure: width within the column / page width, embedded TrueType fonts only (no Type 3).
    Returns a list of problems (empty = passes)."""
    problems = []
    raw = open(path, "rb").read()
    if b"/Type3" in raw:
        problems.append("Type 3 font")
    import re
    m = re.search(rb"/MediaBox\s*\[\s*([\d.\-]+)\s+([\d.\-]+)\s+([\d.\-]+)\s+([\d.\-]+)\s*\]", raw)
    if m:
        w_in = (float(m.group(3)) - float(m.group(1))) / 72.0
        if w_in > PAGE_IN + 0.05:
            problems.append(f"width {w_in:.2f} in exceeds the page width")
    return problems


In [ ]:
%%writefile /kaggle/working/fedecs_lt/plots.py
"""All figures of one experiment (action plan sections 15 and 17b), in IEEE format (plot_style), each with a formal
caption (captions.py) and a CSV of its data. A figure whose data the experiment does not contain (e.g. privacy plots in
an experiment without DP runs) is listed as not applicable, never silently dropped.

Catalogue (numbering fixed here and in docs/review/action_plan.md section 15):
 A  training curves        1 bal. acc. vs round (Stage III)   2 Stage I curves      3 generator loss      4 head losses
                           5 macro-F1 and tail recall vs round 6 ECE vs round         7 consensus entropy   8 client drift
 B  class imbalance        9 per-class recall   10 recall vs class size   11 confusion matrices   12 per-class precision
                           13 per-class F1      14 head/medium/tail recall 15 precision-recall curves (rare classes)
                           16 reliability diagram 17 class-wise ECE      18 worst-class recall and G-mean
 C  representation         19 t-SNE embeddings (ours) 20 t-SNE embeddings (GeFL-F) 21 PCA embeddings
                           22 t-SNE real vs generated (ours) 23 t-SNE real vs generated (GeFL-F)
                           24 real vs generated class means 25 classifier norm per class 26 classifier cosine similarity
                           27 neural-collapse statistics
 D  generator quality      28 per-class fidelity 29 spread ratio 30 class-row norms 31 fidelity over rounds
                           32 moment-correction scales
 E  mechanism              33 class-row norm over rounds 34 holder gain vs non-holder shrink 35 row norm vs holders
                           36 Adam second moment of the rows 37 readout gap
 F  comparison             38 balanced accuracy bars 39 forest plot 40 critical-difference diagram 41 radar chart
                           42 accuracy vs communication 43 accuracy vs time 44 metric heatmap 45 ablation waterfall
                           46 sensitivity lines
 G  federated system       47 communication by message type 48 time per stage 49 GPU memory 50 participation
                           51 accuracy vs cumulative communication
 H  privacy                52 accuracy vs DP epsilon 53 membership-inference ROC 54 release noise vs class size
 I  diagram                55 pipeline diagram
 17b aggregation effect    56 aggregation effect per class over rounds 57 fidelity before vs after aggregation
                           58 row norms of holders' copies, non-holders' copies and the average 59 effect vs holders
                           60 federated vs local-only per class
 B' per-class metrics      61-68 per-class AP, AUC, MCC, ECE, NLL, Brier, top-5, prediction share 69 recall after the
                           exact readout 70 per-class recall over the rounds
 F' one shared model       71 balanced accuracy over rounds 72 tail recall over rounds 73 per-class recall over rounds
 A' head stage             74 logit spread 75 its change in the synthetic phase 76 retention 77 consensus agreement
 C' representation         78 NC2 / NC3 / minority-collapse angle 79 CKA between architectures
 D' generator quality      80 per-class MMD 81 per-class mean mismatch 82 active latent units
"""
import glob
import json
import os

import numpy as np

import captions as cap
import plot_style as ps
import representation as rp
from settings import DATASETS, class_groups

OURS = ps.OURS
REF = "GeFL-F"


# ---------------------------------------------------------------- loading
class Exp:
    """Everything one experiment produced: runs (one record per run), per-round logs and saved predictions."""

    def __init__(self, res_dir):
        self.dir = res_dir
        self.runs = []
        p = os.path.join(res_dir, "runs.jsonl")
        if os.path.exists(p):
            for line in open(p):
                try:
                    self.runs.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
        self.logs = {}
        for f in glob.glob(os.path.join(res_dir, "logs", "*.jsonl")):
            recs = []
            for line in open(f):
                try:
                    recs.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
            if recs and recs[0].get("event") == "run_start":
                self.logs[recs[0]["run_id"]] = recs
        for r in self.runs:   # names as printed (captions.display); the suite's own label is kept
            r.setdefault("label_internal", r["label"])
            r["label"] = cap.display(r["label_internal"])
            derived(r)
        self.methods = [m for m in ps.ORDER if any(r["label"] == m for r in self.runs)]
        self.methods += sorted({r["label"] for r in self.runs} - set(self.methods))
        self.settings = sorted({setting_key(r) for r in self.runs})

    def of(self, label, setting=None):
        return [r for r in self.runs if r["label"] == label and (setting is None or setting_key(r) == setting)]

    def log(self, run, event):
        return [e for e in self.logs.get(run["run_id"], []) if e["event"] == event]

    def pred(self, run):
        safe = "".join(ch if ch.isalnum() or ch in "-_.=" else "_" for ch in run["run_id"])
        p = os.path.join(self.dir, "predictions", safe + ".npz")
        return np.load(p) if os.path.exists(p) else None


def derived(r):
    """Quantities derived from a run's record for the tables: forward MFLOPs per image (extractor plus the mean head,
    or the whole model), training GFLOPs per round, and the spread and minimum over the clients' models (or head
    architectures) of the accuracy on their own class mix."""
    if r.get("flops_model"):
        r["fwd_mflops"] = r["flops_model"] / 1e6
    elif r.get("flops_head"):
        r["fwd_mflops"] = (r.get("flops_fe", 0.0) + float(np.mean(list(r["flops_head"].values())))) / 1e6
    g = r.get("head_stage_gflops_per_round", r.get("train_gflops_per_round"))
    if g is not None:
        r["train_gflops_round"] = g
    units = r.get("final_per_arch") or r.get("final_per_client") or r.get("local_per_client") or {}
    own = [u["local"] for u in units.values() if isinstance(u, dict) and u.get("local") is not None]
    if len(own) > 1:
        r["final_own_mix_sd"] = float(np.std(own))
        r["final_own_mix_min"] = float(np.min(own))


def setting_key(r):
    return (r["dataset"], r["IF"], r["alpha"], r["K"], r.get("pfrac", 1.0), bool(r.get("tail_perm", False)))


def ms(vals):
    v = np.array([x for x in vals if x is not None and not (isinstance(x, float) and np.isnan(x))], dtype=float)
    if len(v) == 0:
        return float("nan"), float("nan"), 0
    return float(v.mean()), float(v.std(ddof=1)) if len(v) > 1 else 0.0, len(v)


def metric(E, label, key, setting):
    return ms([r.get(key) for r in E.of(label, setting)])


# ---------------------------------------------------------------- figure registry
FIGS = []


def fig(num, group):
    def deco(fn):
        FIGS.append((num, group, fn))
        return fn
    return deco


def _curve(E, label, setting, event, getter):
    """Mean over seeds of a per-round quantity: rounds, mean, sd."""
    per = []
    for r in E.of(label, setting):
        pts = [(e["round"], getter(e)) for e in E.log(r, event)]
        pts = [(a, b) for a, b in pts if b is not None]
        if pts:
            per.append(dict(pts))
    if not per:
        return None
    rounds = sorted(set.intersection(*[set(p) for p in per]))
    if not rounds:
        return None
    vals = np.array([[p[x] for x in rounds] for p in per], dtype=float)
    return np.array(rounds), vals.mean(0), vals.std(0, ddof=1) if len(per) > 1 else np.zeros(len(rounds))


def _one_client_per_arch(r):
    """True when every head architecture has a single client (K <= 10 with ten architectures): aggregating a head
    then averages one copy and leaves it unchanged, so the aggregation-effect figures say nothing."""
    from settings import DATASETS, N_ARCH
    return not DATASETS.get(r["dataset"], {}).get("single_arch") and r["K"] <= N_ARCH


def _round_ticks(ax, rounds, n=6):
    """Heatmap columns are evaluation rounds (not evenly spaced: round 1, then every 5th): ticks at the column
    centres, labelled with the round numbers."""
    idx = sorted(set(np.linspace(0, len(rounds) - 1, min(n, len(rounds))).round().astype(int)))
    ax.set_xticks(idx)
    ax.set_xticklabels([str(rounds[i]) for i in idx])


def _lines(E, S, out, num, event, getter, ylabel, name, what, scale=100.0, methods=None, width="column",
           split="Test-set"):
    f, ax = ps.figure(width, 2.4)
    rows = []
    for m in methods or E.methods:
        c = _curve(E, m, S, event, getter)
        if c is None:
            continue
        x, mu, sd = c
        st = ps.style(m)
        ax.plot(x, mu * scale, label=m, markevery=max(1, len(x) // 6), **st)
        ax.fill_between(x, (mu - sd) * scale, (mu + sd) * scale, color=st["color"], alpha=0.12, linewidth=0)
        rows += [[m, int(a), float(b * scale), float(s * scale)] for a, b, s in zip(x, mu, sd)]
    if not rows:
        return []
    ax.set_xlabel("Round")
    ax.set_ylabel(ylabel)
    ax.legend(ncol=1 if len(set(r[0] for r in rows)) < 6 else 2)
    p = ps.save(f, out, f"fig_{num:02d}_{name}", rows, ["method", "round", "mean", "sd"])
    gaps = {b[1] - a[1] for a, b in zip(rows, rows[1:]) if a[0] == b[0]}   # rounds between the plotted points
    every = 1 if gaps <= {1} else 5
    return [(p, cap.curve(num, E, S, what, split, every))]


@fig(1, "A")
def f01(E, S, out):
    return _lines(E, S, out, 1, "stage3_round", lambda e: (e.get("post") or {}).get("bal"),
                  "Balanced accuracy (%)", "balanced_accuracy_rounds", "balanced accuracy of the aggregated heads "
                  "during head training (Stage III)", width="page")


@fig(2, "A")
def f02(E, S, out):
    return _lines(E, S, out, 2, "stage1_round", lambda e: (e.get("post") or {}).get("bal"), "Balanced accuracy (%)",
                  "stage1_rounds", "balanced accuracy of the shared feature extractor with the architecture heads "
                  "during Stage I", methods=[m for m in E.methods if m in (OURS,)] or E.methods[:1])


@fig(3, "A")
def f03(E, S, out):
    return _lines(E, S, out, 3, "stage2_round", lambda e: np.mean(list(e["loss"].values())) if e.get("loss") else None,
                  "Generator loss per sample", "generator_loss", "Mean training loss of the clients' generator copies "
                  "(reconstruction + KL) during Stage II", scale=1.0, split=None)


@fig(4, "A")
def f04(E, S, out):
    a = _lines(E, S, out, 4, "stage3_round",
               lambda e: np.mean([v for v in e["real_loss"].values() if v is not None]) if e.get("real_loss") else None,
               "Real-phase loss", "head_real_loss", "Mean real-feature training loss of the clients' heads during "
               "Stage III", scale=1.0, split=None)
    return a


@fig(5, "A")
def f05(E, S, out):
    return _lines(E, S, out, 5, "stage3_round", lambda e: (e.get("post") or {}).get("tail"), "Tail-class recall (%)",
                  "tail_recall_rounds", "mean recall of the tail classes (the rarest 40%) of the aggregated heads "
                  "during Stage III")


@fig(6, "A")
def f06(E, S, out):
    return _lines(E, S, out, 6, "stage3_round", lambda e: (e.get("post") or {}).get("ece"),
                  "Expected calibration error (%)", "ece_rounds", "expected calibration error (15 bins) of the "
                  "aggregated heads during Stage III")


@fig(7, "A")
def f07(E, S, out):
    return _lines(E, S, out, 7, "stage3_round", lambda e: e.get("consensus_entropy"), "Consensus entropy (nats)",
                  "consensus_entropy", "Mean entropy of the consensus soft labels on the round's generated features",
                  scale=1.0, methods=[OURS], split=None)


@fig(8, "A")
def f08(E, S, out):
    return _lines(E, S, out, 8, "stage1_round", lambda e: float(np.mean(e["drift"])) if e.get("drift") else None,
                  "Client drift (L2)", "client_drift", "Mean L2 distance of the clients' feature extractors from "
                  "their average after each Stage I round (client drift)", scale=1.0,
                  methods=E.methods[:1], split=None)


# ---------------------------------------------------------------- B: class imbalance
def _per_class(E, S, key, methods):
    mat = {}
    for m in methods:
        rs = [r.get(key) for r in E.of(m, S) if r.get(key)]
        if rs:
            mat[m] = np.array(rs, dtype=float)
    return mat


def _class_lines(E, S, out, num, key, ylabel, name, what, scale=100.0, ref=None, prob_metric=False):
    """One line per method over the classes (most to least frequent). ref = (label, values) adds a grey reference
    line (e.g. the true class share)."""
    mat = _per_class(E, S, key, E.methods)
    if not mat:
        return []
    C = len(next(iter(mat.values()))[0])
    f, ax = ps.figure("page", 2.6)
    rows = []
    x = np.arange(C)
    for m, v in mat.items():
        mu = np.nanmean(v, 0) * scale
        sd = np.nanstd(v, 0, ddof=1) * scale if len(v) > 1 else np.zeros(C)
        st = ps.style(m)
        ax.plot(x, mu, label=m, markevery=max(1, C // 12), **st)
        rows += [[m, int(c), float(a), float(b)] for c, a, b in zip(x, mu, sd)]
    if ref is not None:
        ax.plot(x, np.asarray(ref[1]) * scale, color="0.5", linestyle=(0, (1, 1)), linewidth=1.2, label=ref[0])
        rows += [[ref[0], int(c), float(a), 0.0] for c, a in zip(x, np.asarray(ref[1]) * scale)]
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel(ylabel)
    ax.legend(ncol=3, loc="best")
    p = ps.save(f, out, f"fig_{num:02d}_{name}", rows, ["method", "class", "mean", "sd"])
    return [(p, cap.per_class(num, E, S, what, note=cap.proto_note(mat) if prob_metric else ""))]


@fig(9, "B")
def f09(E, S, out):
    return _class_lines(E, S, out, 9, "final_recall_c", "Recall (%)", "per_class_recall", "recall")


@fig(12, "B")
def f12(E, S, out):
    return _class_lines(E, S, out, 12, "final_precision_c", "Precision (%)", "per_class_precision", "precision")


@fig(13, "B")
def f13(E, S, out):
    return _class_lines(E, S, out, 13, "final_f1_c", "F1 score (%)", "per_class_f1", "F1 score")


# every other metric with a per-class form (metrics.PER_CLASS), one figure each
@fig(61, "B")
def f61(E, S, out):
    return _class_lines(E, S, out, 61, "final_ap_c", "Average precision (%)", "per_class_auc_pr",
                        "average precision (area under the precision-recall curve)")


@fig(62, "B")
def f62(E, S, out):
    return _class_lines(E, S, out, 62, "final_auc_c", "AUC-ROC (%)", "per_class_auc_roc",
                        "area under the ROC curve of the class against all others")


@fig(63, "B")
def f63(E, S, out):
    return _class_lines(E, S, out, 63, "final_mcc_c", "MCC (%)", "per_class_mcc",
                        "Matthews correlation coefficient of the class against all others")


@fig(64, "B")
def f64(E, S, out):
    return _class_lines(E, S, out, 64, "final_ece_c", "Calibration error (%)", "per_class_calibration",
                        "calibration error of the predicted probability of the class (15 bins; lower is better)",
                        prob_metric=True)


@fig(65, "B")
def f65(E, S, out):
    return _class_lines(E, S, out, 65, "final_nll_c", "NLL (nats)", "per_class_nll",
                        "negative log-likelihood on the test images of the class (lower is better)", scale=1.0,
                        prob_metric=True)


@fig(66, "B")
def f66(E, S, out):
    return _class_lines(E, S, out, 66, "final_brier_c", "Brier score", "per_class_brier",
                        "Brier score on the test images of the class (lower is better)", scale=1.0,
                        prob_metric=True)


@fig(67, "B")
def f67(E, S, out):
    return _class_lines(E, S, out, 67, "final_top5_c", "Top-5 recall (%)", "per_class_top5",
                        "top-5 recall (true class among the five most probable)")


@fig(68, "B")
def f68(E, S, out):
    true = None
    for r in E.runs:
        if setting_key(r) == S and r.get("final_confusion"):
            cm = np.array(r["final_confusion"], dtype=float)
            true = cm.sum(1) / cm.sum()
            break
    return _class_lines(E, S, out, 68, "final_pred_share_c", "Share of predictions (%)", "per_class_prediction_share",
                        "share of all test predictions that go to the class; the dotted grey line is the class's "
                        "true share of the test set (a line above it: the class is over-predicted)",
                        ref=("Test set (true share)", true) if true is not None else None)


@fig(69, "B")
def f69(E, S, out):
    return _class_lines(E, S, out, 69, "er_recall_c", "Recall, exact readout (%)", "per_class_recall_exact_readout",
                        "recall after the exact class-balanced readout")


@fig(70, "B")
def f70(E, S, out):
    """Per-class recall of the aggregated heads at every evaluation round of Stage III (FedECS-LT and GeFL-F)."""
    res = []
    for m in [x for x in (OURS, REF) if x in E.methods]:
        rs = E.of(m, S)
        if not rs:
            continue
        ev = [e for e in E.log(rs[0], "stage3_round") if (e.get("post") or {}).get("recall_c")]
        if len(ev) < 2:
            continue
        A = np.array([e["post"]["recall_c"] for e in ev], dtype=float).T * 100
        rr = [e["round"] for e in ev]
        f, ax = ps.figure("column", 2.6)
        im = ax.imshow(A, aspect="auto", cmap="viridis", origin="lower", vmin=0, vmax=100)
        _round_ticks(ax, rr)
        ax.set_xlabel("Round")
        ax.set_ylabel("Class (0 = most frequent)")
        f.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Recall (%)")
        name = "".join(ch for ch in m if ch.isalnum()).lower()
        p = ps.save(f, out, f"fig_70_per_class_recall_rounds_{name}",
                    [[rr[j], c, float(A[c, j])] for c in range(A.shape[0]) for j in range(A.shape[1])],
                    ["round", "class", "recall"])
        res.append((p, cap.simple(70, E, S, f"Recall of every class at each evaluation round of head training "
                                            f"(Stage III) for {cap.mname(m)}: aggregated heads, test set, first seed. "
                                            "Every per-class metric of every evaluation round is in the run logs.")))
    return res


# ---------------------------------------------------------------- Track 2: over all communication rounds
def _global_series(E, r):
    """(round, evaluation) pairs of a run on one round axis: LongTailLib's own evaluations (every 5th round); for the
    suite's pipeline, Stage I's evaluations (extractor and heads) and Stage III's, the latter counted after the T_FE
    rounds of Stage I (the generator stage in between is not a training round of the classifier)."""
    out = []
    if r["method"].get("pipeline") == "longtaillib":
        out = [(e["round"], e["post"]) for e in E.log(r, "round_eval") if e.get("post")]
    else:
        T_FE = (r.get("rounds") or {}).get("T_FE", 0)
        out = [(e["round"], e["post"]) for e in E.log(r, "stage1_round") if e.get("post")]
        out += [(T_FE + e["round"], e["post"]) for e in E.log(r, "stage3_round") if e.get("post")]
    return sorted(out, key=lambda t: t[0])


def _track2(S):
    return DATASETS.get(S[0], {}).get("track") == 2


def _global_lines(E, S, out, num, key, ylabel, name, what):
    if not _track2(S):
        return []
    f, ax = ps.figure("page", 2.8)
    rows = []
    for m in E.methods:
        per = []
        for r in E.of(m, S):
            pts = {g: p.get(key) for g, p in _global_series(E, r) if p.get(key) is not None}
            if pts:
                per.append(pts)
        if not per:
            continue
        rr = sorted(set.intersection(*[set(p) for p in per]))
        if not rr:
            continue
        V = np.array([[p[g] for g in rr] for p in per], dtype=float) * 100
        mu, sd = V.mean(0), (V.std(0, ddof=1) if len(V) > 1 else np.zeros(len(rr)))
        st = ps.style(m)
        ax.plot(rr, mu, label=m, markevery=max(1, len(rr) // 8), **st)
        ax.fill_between(rr, mu - sd, mu + sd, color=st["color"], alpha=0.12, linewidth=0)
        rows += [[m, int(g), float(a), float(b)] for g, a, b in zip(rr, mu, sd)]
    if not rows:
        return []
    ax.set_xlabel("Round")
    ax.set_ylabel(ylabel)
    ax.legend(ncol=3)
    p = ps.save(f, out, f"fig_{num:02d}_{name}", rows, ["method", "round", "mean", "sd"])
    return [(p, cap.simple(num, E, S, f"Test-set {what} of the global model over the communication rounds, mean over "
                                      "seeds (band: one standard deviation). LongTailLib methods: every 5th round; "
                                      "FedECS-LT: its extractor stage, then its head stage counted after the "
                                      "extractor's rounds (its generator stage in between is not shown)."))]


@fig(71, "F")
def f71(E, S, out):
    return _global_lines(E, S, out, 71, "bal", "Balanced accuracy (%)", "track2_balanced_accuracy_rounds",
                         "balanced accuracy")


@fig(72, "F")
def f72(E, S, out):
    return _global_lines(E, S, out, 72, "tail", "Tail-class recall (%)", "track2_tail_recall_rounds",
                         "recall of the tail classes (the rarest 40%)")


@fig(73, "F")
def f73(E, S, out):
    """Per-class recall over the rounds, one heatmap per method (Track 2; first seed)."""
    if not _track2(S):
        return []
    res = []
    for m in E.methods:
        rs = E.of(m, S)
        if not rs:
            continue
        ser = [(g, p["recall_c"]) for g, p in _global_series(E, rs[0]) if p.get("recall_c")]
        if len(ser) < 2:
            continue
        A = np.array([v for _, v in ser], dtype=float).T * 100
        rr = [g for g, _ in ser]
        f, ax = ps.figure("column", 2.6)
        im = ax.imshow(A, aspect="auto", cmap="viridis", origin="lower", vmin=0, vmax=100)
        _round_ticks(ax, rr)
        ax.set_xlabel("Round")
        ax.set_ylabel("Class (0 = most frequent)")
        f.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Recall (%)")
        name = "".join(ch for ch in m if ch.isalnum()).lower()
        p = ps.save(f, out, f"fig_73_track2_per_class_recall_rounds_{name}",
                    [[rr[j], c, float(A[c, j])] for c in range(A.shape[0]) for j in range(A.shape[1])],
                    ["round", "class", "recall"])
        res.append((p, cap.simple(73, E, S, f"Recall of every class of {cap.mname(m)}'s global model at each "
                                            "evaluation over the communication rounds (test set, first seed). Every "
                                            "per-class metric of every evaluation is in the run logs.")))
    return res


@fig(10, "B")
def f10(E, S, out):
    f, ax = ps.figure("column", 2.4)
    rows = []
    for m in E.methods:
        rs = E.of(m, S)
        if not rs or not rs[0].get("final_recall_c"):
            continue
        n = np.array(rs[0]["client_counts"]).sum(0)
        rec = np.mean([r["final_recall_c"] for r in rs], 0) * 100
        st = ps.style(m)
        ax.scatter(n, rec, s=8, marker=st["marker"], color=st["color"], label=m, linewidths=0.4)
        rows += [[m, int(a), float(b)] for a, b in zip(n, rec)]
    if not rows:
        return []
    logx = ps.log_axis(ax, [r[1] for r in rows])
    ax.set_xlabel("Training images of the class (federation)")
    ax.set_ylabel("Recall (%)")
    ax.legend(ncol=2)
    p = ps.save(f, out, "fig_10_recall_vs_class_size", rows, ["method", "class_size", "recall"])
    return [(p, cap.simple(10, E, S, "Per-class recall against the number of training images of the class in the "
                                     "federation" + (" (logarithmic axis)" if logx else "") + "; each point is one class, mean over "
                                     "seeds."))]


@fig(11, "B")
def f11(E, S, out):
    res = []
    for m in [x for x in (OURS, REF) if x in E.methods]:
        cms = [np.array(r["final_confusion"], dtype=float) for r in E.of(m, S) if r.get("final_confusion")]
        if not cms:
            continue
        cm = np.sum(cms, 0)
        cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
        C = len(cm)
        f, ax = ps.figure("column", 3.0)
        im = ax.imshow(cmn * 100, vmin=0, vmax=100, cmap="viridis")
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        if C <= 20:
            ax.set_xticks(range(C))
            ax.set_yticks(range(C))
        cb = f.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label("Share of the true class (%)")
        name = "ours" if m == OURS else "geflf"
        rows = [[i, j, float(cmn[i, j])] for i in range(C) for j in range(C)]
        p = ps.save(f, out, f"fig_11_confusion_{name}", rows, ["true", "pred", "share"], raster=C > 50)
        res.append((p, cap.simple(11, E, S, f"Row-normalised confusion matrix of {cap.mname(m)} on the test set "
                                            "(all client models, summed over seeds); classes ordered from most "
                                            "to least frequent.")))
    return res


@fig(14, "B")
def f14(E, S, out):
    rows = []
    for m in E.methods:
        vals = [metric(E, m, f"final_{g}", S) for g in ("head", "medium", "tail")]
        if np.isnan(vals[0][0]):
            continue
        rows.append([m] + [v[0] * 100 for v in vals] + [v[1] * 100 for v in vals])
    if not rows:
        return []
    f, ax = ps.figure("page", 2.6)
    w = 0.8 / 3
    x = np.arange(len(rows))
    for i, (g, hatch) in enumerate(zip(("Head", "Medium", "Tail"), ("", "///", "..."))):
        ax.bar(x + (i - 1) * w, [r[1 + i] for r in rows], w, yerr=[r[4 + i] for r in rows], label=g, hatch=hatch,
               color=["#BBBBBB", "#777777", "#333333"][i], edgecolor="black", linewidth=0.4, error_kw=dict(lw=0.6))
    ax.set_xticks(x)
    ax.set_xticklabels([r[0] for r in rows], rotation=35, ha="right")
    ax.set_ylabel("Recall (%)")
    ax.legend(ncol=3)
    p = ps.save(f, out, "fig_14_head_medium_tail", rows, ["method", "head", "medium", "tail", "sd_head", "sd_medium",
                                                         "sd_tail"])
    return [(p, cap.simple(14, E, S, "Mean recall of the head (most frequent 30%), medium (next 30%) and tail "
                                     "(rarest 40%) classes; mean over seeds, error bars one standard deviation."))]


def _probs(E, m, S):
    """Test-set probabilities of every architecture's head (all seeds), with labels."""
    P, Y = [], []
    for r in E.of(m, S):
        d = E.pred(r)
        if d is None:
            continue
        for k in d.files:
            if k.startswith("probs_"):
                P.append(d[k].astype(np.float64))
                Y.append(d["y_full"] if "y_full" in d.files else None)
    return P, Y


@fig(15, "B")
def f15(E, S, out):
    from sklearn.metrics import precision_recall_curve
    res = []
    for m in [x for x in (OURS, REF) if x in E.methods]:
        rs = E.of(m, S)
        d = E.pred(rs[0]) if rs else None
        if d is None or "y_full" not in d.files:
            continue
        y = d["y_full"]
        C = int(y.max()) + 1
        probs = np.mean([d[k].astype(np.float64) for k in d.files if k.startswith("probs_")], 0)
        tail = list(range(C))[class_groups(C)[2]][-3:]
        f, ax = ps.figure("column", 2.4)
        rows = []
        for i, c in enumerate(tail):
            pr, rc, _ = precision_recall_curve(y == c, probs[:, c])
            ax.plot(rc * 100, pr * 100, linestyle=ps.DASHES[i], color=ps.PALETTE[i], label=f"Class {c}")
            rows += [[c, float(a), float(b)] for a, b in zip(rc[::max(1, len(rc) // 200)], pr[::max(1, len(pr) // 200)])]
        ax.set_xlabel("Recall (%)")
        ax.set_ylabel("Precision (%)")
        ax.legend()
        name = "ours" if m == OURS else "geflf"
        p = ps.save(f, out, f"fig_15_pr_curves_{name}", rows, ["class", "recall", "precision"])
        res.append((p, cap.simple(15, E, S, f"Precision-recall curves of the three rarest classes for {cap.mname(m)} "
                                            "(averaged softmax of all client models, first seed).")))
    return res


@fig(16, "B")
def f16(E, S, out):
    f, ax = ps.figure("column", 2.6)
    rows = []
    for m in [x for x in (OURS, REF, "GeFL-F + LA") if x in E.methods]:
        rs = E.of(m, S)
        d = E.pred(rs[0]) if rs else None
        if d is None or "y_full" not in d.files:
            continue
        y = d["y_full"]
        conf_all, acc_all = [], []
        for k in d.files:
            if k.startswith("probs_"):
                p = d[k].astype(np.float64)
                conf_all.append(p.max(1))
                acc_all.append(p.argmax(1) == y)
        conf, acc = np.concatenate(conf_all), np.concatenate(acc_all)
        edges = np.linspace(0, 1, 16)
        mids, accs = [], []
        for i in range(15):
            sel = (conf > edges[i]) & (conf <= edges[i + 1])
            if sel.sum() > 20:
                mids.append(conf[sel].mean())
                accs.append(acc[sel].mean())
        ax.plot(np.array(mids) * 100, np.array(accs) * 100, label=m, **ps.style(m))
        rows += [[m, float(a), float(b)] for a, b in zip(mids, accs)]
    if not rows:
        return []
    ax.plot([0, 100], [0, 100], color="#999999", linewidth=0.6, linestyle=":")
    ax.set_xlabel("Confidence (%)")
    ax.set_ylabel("Accuracy (%)")
    ax.legend()
    p = ps.save(f, out, "fig_16_reliability", rows, ["method", "confidence", "accuracy"])
    return [(p, cap.simple(16, E, S, "Reliability diagram: accuracy against confidence in 15 equal-width bins "
                                     "(bins with more than 20 test images); the dotted line is perfect calibration."))]


@fig(17, "B")
def f17(E, S, out):
    rows = []
    for m in E.methods:
        v = metric(E, m, "final_cw_ece", S)
        if not np.isnan(v[0]):
            rows.append([m, v[0] * 100, v[1] * 100])
    return _bars(E, S, out, 17, rows, "Class-wise ECE (%)", "classwise_ece",
                 "Class-wise expected calibration error (mean over classes of the calibration error of p(c|x)); mean "
                 "over seeds, error bars one standard deviation.")


@fig(18, "B")
def f18(E, S, out):
    res = []
    for key, lab, name, txt in (("final_worst", "Worst-class recall (%)", "worst_class", "Recall of the worst class"),
                                ("final_gmean", "G-mean of recalls (%)", "gmean",
                                 "Geometric mean of the class recalls (each floored at 1%)")):
        rows = []
        for m in E.methods:
            v = metric(E, m, key, S)
            if not np.isnan(v[0]):
                rows.append([m, v[0] * 100, v[1] * 100])
        res += _bars(E, S, out, 18, rows, lab, name, txt + "; mean over seeds, error bars one standard deviation.")
    return res


def _bars(E, S, out, num, rows, ylabel, name, text, width="page"):
    if not rows:
        return []
    f, ax = ps.figure(width, 2.4)
    x = np.arange(len(rows))
    for i, r in enumerate(rows):
        st = ps.style(r[0])
        ax.bar(i, r[1], 0.7, yerr=r[2], color=st["color"], edgecolor="black", linewidth=0.4,
               hatch="///" if r[0] == OURS else None, error_kw=dict(lw=0.6))
    ax.set_xticks(x)
    ax.set_xticklabels([r[0] for r in rows], rotation=35, ha="right")
    ax.set_ylabel(ylabel)
    p = ps.save(f, out, f"fig_{num:02d}_{name}", rows, ["method", "mean", "sd"])
    return [(p, cap.simple(num, E, S, text))]


# ---------------------------------------------------------------- C: representation
def _tsne(X, seed=0, perplexity=30):
    """t-SNE as recommended by Kobak and Berens (Nat. Commun. 2019): PCA to 50 dimensions, PCA initialisation (also
    makes the map reproducible), perplexity 30 for a few thousand points, learning rate n/12 ("auto")."""
    from sklearn.manifold import TSNE
    X = X.astype(np.float32)
    if X.shape[1] > 50:
        from sklearn.decomposition import PCA
        X = PCA(50, random_state=seed).fit_transform(X)
    return TSNE(2, init="pca", random_state=seed, perplexity=min(perplexity, (len(X) - 1) / 3),
                learning_rate="auto").fit_transform(X)


TSNE_NOTE = (" As in every t-SNE map, cluster sizes and distances between clusters carry no meaning (Wattenberg et "
             "al., Distill 2016); PCA to 50 dimensions, PCA initialisation, perplexity 30 (Kobak and Berens, Nat. "
             "Commun. 2019).")


def _knn_text(H, y, C):
    """Caption sentence with the 10-NN accuracy of the embeddings (all classes and the tail classes)."""
    ok = rp.knn_correct(H, y)
    tail = np.isin(y, np.arange(C)[class_groups(C)[2]])
    return (f" Leave-one-out 10-nearest-neighbour accuracy of these embeddings: {100 * ok.mean():.1f}% (tail classes "
            f"{100 * ok[tail].mean():.1f}%)." if tail.any() else f" 10-NN accuracy {100 * ok.mean():.1f}%."), ok


def _scatter_classes(ax, Z, y, C, marker="o", alpha=0.8, label_prefix=None, emphasise_tail=False):
    """Points coloured by class. emphasise_tail: the tail classes (rarest 40%) drawn larger, opaque and on top, the
    others smaller and lighter, so that whether the rare classes form their own clusters can be seen (as in the
    feature-space figures of federated long-tail papers)."""
    import matplotlib.pyplot as plt
    cmap = plt.get_cmap("tab10" if C <= 10 else "tab20")
    tail = set(np.arange(C)[class_groups(C)[2]].tolist()) if emphasise_tail else set()
    for c in range(C):
        sel = y == c
        if sel.any():
            # filled markers need no edge; line markers ("x") are drawn only by their edge, so they need a width
            lw = 0.7 if marker in ("x", "+", "1", "2", "3", "4") else 0
            size = (4 if lw == 0 else 9) * (1.8 if c in tail else (0.8 if tail else 1.0))
            ax.scatter(Z[sel, 0], Z[sel, 1], s=size, color=cmap(c % cmap.N), marker=marker,
                       alpha=0.95 if c in tail else (0.35 if tail else alpha), linewidths=lw, zorder=3 if c in tail else 2,
                       label=f"{label_prefix or ''}{c}" if C <= 10 else None, rasterized=True)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():   # embedding axes have no scale: a full thin frame, no ticks
        sp.set_visible(True)
        sp.set_linewidth(0.5)


def _class_labels(C, names):
    """Legend labels: the class index, with its name when the dataset has short names (CIFAR-10: "0 airplane");
    datasets whose names are digits or long phrases keep the index (the class table gives every name)."""
    if not names or len(names) < C:
        return [str(c) for c in range(C)]
    nm = [str(n) for n in names[:C]]
    if all(n == str(c) for c, n in enumerate(nm)) or max(len(n) for n in nm) > 12:
        return [str(c) for c in range(C)]
    return [f"{c} {n}" for c, n in enumerate(nm)]


def _class_legend(ax, C, sources=False, names=None):
    """Class legend inside the axes, top right (embedding scatter plots): one entry per class (10 classes or fewer;
    with more, the caption says the points are coloured by class), plus the marker of real and generated features.
    The view is then extended upwards (and if needed to the right) only as far as it takes for the legend to cover
    no point, so the points keep as much of the panel as possible."""
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    if C > 10 and not sources:
        return
    cmap = plt.get_cmap("tab10")
    labels = _class_labels(C, names) if C <= 10 else []
    handles = [Line2D([], [], marker="o", linestyle="", color=cmap(c), markersize=4, label=lab)
               for c, lab in enumerate(labels)]
    if sources:
        handles += [Line2D([], [], marker="o", linestyle="", color="0.45", markersize=4, label="Real"),
                    Line2D([], [], marker="x", linestyle="", color="0.45", markersize=4, label="Generated")]
    long_ = any(len(h.get_label()) > 3 for h in handles)
    leg = ax.legend(handles=handles, loc="upper right", ncol=2 if long_ else min(5, len(handles)), frameon=True,
                    framealpha=0.9, edgecolor="0.7", fancybox=False, handletextpad=0.1, columnspacing=0.6,
                    borderpad=0.3, labelspacing=0.2, borderaxespad=0.3)
    leg.get_frame().set_linewidth(0.5)
    leg.set_gid("keep")   # plot_style.place_legends leaves it where it is
    fig = ax.figure
    r = fig.canvas.get_renderer()
    y0, y1 = ax.get_ylim()
    x0, x1 = ax.get_xlim()
    for step in range(1, 26):   # headroom in steps of 4% until the legend covers no point (at most 100%)
        fig.canvas.draw()
        if not ps.points_under(ax, leg, r):
            return
        ax.set_ylim(y0, y1 + 0.04 * step * (y1 - y0))
    for step in range(1, 16):   # a legend taller than the cloud: room on the right as well
        ax.set_xlim(x0, x1 + 0.05 * step * (x1 - x0))
        fig.canvas.draw()
        if not ps.points_under(ax, leg, r):
            return


def _colour_note(C):
    """Caption note of the embedding scatter plots: more than 10 classes have no legend and share 20 colours."""
    if C <= 10:
        return ""
    return (f" With {C} classes the 20 colours repeat (class c and class c + 20 share one), so the colours show how "
            "classes cluster, not which class is which; the class of every point is in the figure's data file.")


def _emb_fig(E, S, out, num, m, name):
    rs = E.of(m, S)
    d = E.pred(rs[0]) if rs else None
    if d is None:
        return []
    keys = sorted(k for k in d.files if k.startswith("embed_"))
    if not keys:
        return []
    y = d["y"]
    C = int(y.max()) + 1
    g = keys[0]
    Z = _tsne(d[g].astype(np.float32))
    knn, _ = _knn_text(d[g].astype(np.float64), y, C)
    f, ax = ps.figure("column", 3.3)
    _scatter_classes(ax, Z, y, C, emphasise_tail=True)
    _class_legend(ax, C, names=rs[0].get("class_names"))
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    p = ps.save(f, out, f"fig_{num:02d}_{name}", [[float(a), float(b), int(c)] for (a, b), c in zip(Z, y)],
                ["tsne1", "tsne2", "class"], raster=True)
    arch = int(g.split("_")[1]) + 1
    head = "ResNet-8 head" if DATASETS.get(S[0], {}).get("track") == 2 else f"head CNN-{arch}"
    return [(p, cap.simple(num, E, S, f"t-SNE of the test-set embeddings of {cap.mname(m)}'s {head} "
                                      f"(2,000 test images, first seed), coloured by true class (classes numbered from the most to the "
                                      f"least frequent; the tail classes, the rarest 40%, drawn larger and opaque)."
                                      f"{knn}{_colour_note(C)}{TSNE_NOTE}"))]


@fig(19, "C")
def f19(E, S, out):
    return _emb_fig(E, S, out, 19, OURS, "tsne_ours")


@fig(20, "C")
def f20(E, S, out):
    return _emb_fig(E, S, out, 20, REF, "tsne_geflf")


@fig(21, "C")
def f21(E, S, out):
    from sklearn.decomposition import PCA
    res = []
    rows = []
    ms_ = [m for m in (OURS, REF) if m in E.methods]
    if not ms_:
        return []
    f, axes = ps.figure("page", 2.6, ncols=len(ms_))
    axes = np.atleast_1d(axes)
    ok = False
    for ax, m in zip(axes, ms_):
        rs = E.of(m, S)
        d = E.pred(rs[0]) if rs else None
        keys = sorted(k for k in d.files if k.startswith("embed_")) if d is not None else []
        if not keys:
            continue
        y = d["y"]
        Z = PCA(2, random_state=0).fit_transform(d[keys[0]].astype(np.float32))
        _scatter_classes(ax, Z, y, int(y.max()) + 1)
        _class_legend(ax, int(y.max()) + 1)
        ax.set_xlabel("Principal component 1")
        ax.set_ylabel("Principal component 2")
        rows += [[m, float(a), float(b), int(c)] for (a, b), c in zip(Z, y)]
        ok, C21 = True, int(y.max()) + 1
    if not ok:
        return []
    ps.panel_labels(axes, y=-0.1)   # no tick labels on embedding plots: the labels sit just under the axis title
    p = ps.save(f, out, "fig_21_pca_embeddings", rows, ["method", "pc1", "pc2", "class"], raster=True)
    res.append((p, cap.simple(21, E, S, "First two principal components of the test-set embeddings: (a) "
                              + " (b) ".join(cap.mname(m) for m in ms_) + "; coloured by true class."
                              + _colour_note(C21))))
    return res


def _real_vs_gen(E, S, out, num, m, name):
    rs = E.of(m, S)
    d = E.pred(rs[0]) if rs else None
    if d is None or "gen_x" not in d.files:
        return []
    Xr, yr, Xg, yg = d["val_x"].astype(np.float32), d["val_y"], d["gen_x"].astype(np.float32), d["gen_y"]
    Z = _tsne(np.concatenate([Xr, Xg]))
    C = int(max(yr.max(), yg.max())) + 1
    f, ax = ps.figure("column", 3.3)
    _scatter_classes(ax, Z[:len(Xr)], yr, C, marker="o", alpha=0.5)
    _scatter_classes(ax, Z[len(Xr):], yg, C, marker="x", alpha=0.9)
    _class_legend(ax, C, sources=True, names=rs[0].get("class_names"))
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    rows = [[float(a), float(b), int(c), "real"] for (a, b), c in zip(Z[:len(Xr)], yr)] + \
           [[float(a), float(b), int(c), "generated"] for (a, b), c in zip(Z[len(Xr):], yg)]
    p = ps.save(f, out, f"fig_{num:02d}_{name}", rows, ["tsne1", "tsne2", "class", "source"], raster=True)
    return [(p, cap.simple(num, E, S, f"t-SNE of real validation features (circles) and features generated by "
                                      f"{cap.mname(m)}'s generator (crosses) in the frozen feature space, coloured by "
                                      f"class (numbered from the most to the least frequent; first seed).{_colour_note(C)}{TSNE_NOTE}"))]


@fig(22, "C")
def f22(E, S, out):
    return _real_vs_gen(E, S, out, 22, OURS, "tsne_real_vs_generated_ours")


@fig(23, "C")
def f23(E, S, out):
    return _real_vs_gen(E, S, out, 23, REF, "tsne_real_vs_generated_geflf")


@fig(24, "C")
def f24(E, S, out):
    res = []
    for m in [x for x in (OURS, REF) if x in E.methods]:
        rs = E.of(m, S)
        d = E.pred(rs[0]) if rs else None
        if d is None or "gen_x" not in d.files:
            continue
        Xr, yr, Xg, yg = d["val_x"].astype(np.float64), d["val_y"], d["gen_x"].astype(np.float64), d["gen_y"]
        C = int(max(yr.max(), yg.max())) + 1
        mr = np.stack([Xr[yr == c].mean(0) if (yr == c).any() else np.zeros(Xr.shape[1]) for c in range(C)])
        mg = np.stack([Xg[yg == c].mean(0) if (yg == c).any() else np.zeros(Xg.shape[1]) for c in range(C)])
        # row by row (same values): the C x C x d array was 24 GB for CIFAR-100 with one shared model (d = 32,768)
        D = np.stack([np.sqrt(((r - mg) ** 2).sum(-1)) for r in mr])
        f, ax = ps.figure("column", 3.0)
        im = ax.imshow(D, cmap="cividis")
        ax.set_xlabel("Generated class")
        ax.set_ylabel("Real class")
        f.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Distance of class means")
        name = "ours" if m == OURS else "geflf"
        p = ps.save(f, out, f"fig_24_class_mean_distances_{name}",
                    [[i, j, float(D[i, j])] for i in range(C) for j in range(C)], ["real", "generated", "distance"])
        res.append((p, cap.simple(24, E, S, f"Euclidean distance between the mean real feature of each class "
                                            f"(validation set) and the mean generated feature of each class for "
                                            f"{cap.mname(m)}; a dark diagonal means generated classes sit at their real "
                                            "class.")))
    return res


@fig(25, "C")
def f25(E, S, out):
    f, ax = ps.figure("page", 2.4)
    rows = []
    for m in E.methods:
        vals = []
        for r in E.of(m, S):
            ev = E.log(r, "stage3_round")
            if ev and ev[-1].get("fc_norm"):
                vals.append(np.mean([np.array(v) for v in ev[-1]["fc_norm"].values()], 0))
        if not vals:
            continue
        v = np.mean(vals, 0)
        ax.plot(np.arange(len(v)), v, label=m, markevery=max(1, len(v) // 12), **ps.style(m))
        rows += [[m, i, float(x)] for i, x in enumerate(v)]
    if not rows:
        return []
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel("Last-layer weight norm")
    ax.legend(ncol=3)
    p = ps.save(f, out, "fig_25_classifier_norms", rows, ["method", "class", "norm"])
    return [(p, cap.simple(25, E, S, "L2 norm of each class's last-layer weight vector after Stage III, averaged over "
                                     "architectures and seeds; a falling line means the classifier favours frequent "
                                     "classes."))]


@fig(26, "C")
def f26(E, S, out):
    return []  # needs the trained last layers: produced from models.zip by tools/figures_from_models.py


@fig(27, "C")
def f27(E, S, out):
    rows = []
    for m in E.methods:
        vals = []
        for r in E.of(m, S):
            d = E.pred(r)
            if d is None:
                continue
            for k in d.files:
                if k.startswith("embed_") and "y" in d.files:
                    H, y = d[k].astype(np.float64), d["y"]
                    v = rp.nc1(H, y, int(y.max()) + 1)
                    if v is not None:
                        vals.append(v)
        if vals:
            rows.append([m, float(np.mean(vals)), float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0])
    return _bars(E, S, out, 27, rows, "NC1 (within / between)", "neural_collapse_nc1",
                 "Neural-collapse statistic NC1 = tr(S_{W} S_{B}^{+})/C (Papyan et al., PNAS 2020) of the test-set "
                 "embeddings, with S_{W} the within-class and S_{B} the between-class covariance (pseudo-inverse over "
                 "its C - 1 non-zero directions); lower = tighter, better separated classes; mean over architectures "
                 "and seeds.")


# ---------------------------------------------------------------- D: generator
@fig(28, "D")
def f28(E, S, out):
    f, ax = ps.figure("page", 2.4)
    rows = []
    for m in E.methods:
        for key, suffix in (("fidelity", ""), ("fidelity_mc", " (moment-corrected)")):
            v = [r[key] for r in E.of(m, S) if r.get(key)]
            if not v:
                continue
            mu = np.mean(v, 0) * 100
            st = ps.style(m)
            if suffix:
                st = dict(st, linestyle=":")
            ax.plot(np.arange(len(mu)), mu, label=m + suffix, markevery=max(1, len(mu) // 12), **st)
            rows += [[m + suffix, i, float(x)] for i, x in enumerate(mu)]
    if not rows:
        return []
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel("Fidelity (%)")
    ax.legend(ncol=2)
    p = ps.save(f, out, "fig_28_generator_fidelity", rows, ["generator", "class", "fidelity"])
    return [(p, cap.simple(28, E, S, "Per-class fidelity of the trained generator: share of 300 generated class-c "
                                     "features that a referee classifier trained on the server's validation features "
                                     "assigns to class c; mean over seeds."))]


@fig(29, "D")
def f29(E, S, out):
    f, ax = ps.figure("column", 2.4)
    rows = []
    for m in E.methods:
        v = [r["spread_ratio"] for r in E.of(m, S) if r.get("spread_ratio")]
        if not v:
            continue
        mu = np.mean(v, 0)
        ax.plot(np.arange(len(mu)), mu, label=m, **ps.style(m))
        rows += [[m, i, float(x)] for i, x in enumerate(mu)]
    if not rows:
        return []
    ax.axhline(1.0, color="#999999", linewidth=0.6, linestyle=":")
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel("Generated / real spread")
    ax.legend()
    p = ps.save(f, out, "fig_29_spread_ratio", rows, ["method", "class", "ratio"])
    return [(p, cap.simple(29, E, S, "Within-class spread of the generated features over the exact real spread "
                                     "(1 = realistic diversity, 0 = copies of the class mean), per class."))]


@fig(30, "D")
def f30(E, S, out):
    f, ax = ps.figure("column", 2.4)
    rows = []
    for m in E.methods:
        v = [r["cond_norms_end"] for r in E.of(m, S) if r.get("cond_norms_end")]
        if not v:
            continue
        mu = np.mean(v, 0)
        ax.plot(np.arange(len(mu)), mu, label=m, **ps.style(m))
        rows += [[m, i, float(x)] for i, x in enumerate(mu)]
    if not rows:
        return []
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel("Class-row norm")
    ax.legend()
    p = ps.save(f, out, "fig_30_class_row_norms", rows, ["method", "class", "norm"])
    return [(p, cap.simple(30, E, S, "L2 norm of each class's conditioning row of the generator after Stage II "
                                     "(class-row generators only); mean over seeds."))]


@fig(31, "D")
def f31(E, S, out):
    return _lines(E, S, out, 31, "stage2_round",
                  lambda e: (np.mean(e["fidelity_post"][class_groups(len(e["fidelity_post"]))[2]])
                             if e.get("fidelity_post") else None),
                  "Tail-class fidelity (%)", "fidelity_rounds", "Fidelity of the aggregated generator on the tail "
                  "classes during Stage II (share of generated features the server's referee assigns to their class)",
                  split=None)


@fig(32, "D")
def f32(E, S, out):
    f, ax = ps.figure("column", 2.4)
    rows = []
    for m in E.methods:
        v = [r["mc_scale"] for r in E.of(m, S) if r.get("mc_scale")]
        if not v:
            continue
        mu = np.mean(v, 0)
        ax.plot(np.arange(len(mu)), mu, label=m, **ps.style(m))
        rows += [[m, i, float(x)] for i, x in enumerate(mu)]
    if not rows:
        return []
    ax.axhline(1.0, color="#999999", linewidth=0.6, linestyle=":")
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel("Spread scale $s_c$")
    p = ps.save(f, out, "fig_32_mc_scales", rows, ["method", "class", "scale"])
    return [(p, cap.simple(32, E, S, "Moment-correction scale s_{c} = (V_{c} / G_{c})^{1/2} per class, with V_{c} the exact "
                                     "within-class variance (from secure sums) and G_{c} that of the generated features: "
                                     "the factor by which the generator's within-class spread is stretched to the exact "
                                     "spread."))]


# ---------------------------------------------------------------- E: mechanism
@fig(33, "E")
def f33(E, S, out):
    res = []
    for m in E.methods:
        rs = E.of(m, S)
        if not rs:
            continue
        ev = E.log(rs[0], "stage2_round")
        ev = [e for e in ev if e.get("row_norm_global")]
        if not ev:
            continue
        A = np.array([e["row_norm_global"] for e in ev]).T
        rr = [e["round"] for e in ev]
        f, ax = ps.figure("column", 2.4)
        im = ax.imshow(A, aspect="auto", cmap="viridis", origin="lower")
        _round_ticks(ax, rr)
        ax.set_xlabel("Round")
        ax.set_ylabel("Class")
        f.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Class-row norm")
        name = "".join(ch for ch in m if ch.isalnum()).lower()
        p = ps.save(f, out, f"fig_33_row_norm_rounds_{name}",
                    [[rr[j], c, float(A[c, j])] for c in range(A.shape[0]) for j in range(A.shape[1])],
                    ["round", "class", "norm"])
        res.append((p, cap.simple(33, E, S, f"Norm of each class's conditioning row of the aggregated generator over "
                                            f"the rounds of Stage II for {cap.mname(m)} (first seed).")))
    return res


@fig(34, "E")
def f34(E, S, out):
    rows = []
    for m in E.methods:
        for r in E.of(m, S)[:1]:
            counts = np.array(r["client_counts"])
            ev = E.log(r, "stage2_round")
            prev = None
            for e in ev:
                if not e.get("row_norm_local") or not e.get("row_norm_global"):
                    continue
                if prev is not None:
                    gain = np.zeros(counts.shape[1])
                    shrink = np.zeros(counts.shape[1])
                    nh = np.zeros(counts.shape[1])
                    nn_ = np.zeros(counts.shape[1])
                    for k, v in e["row_norm_local"].items():
                        hold = counts[int(k)] > 0
                        d = np.array(v) - np.array(prev)
                        gain[hold] += d[hold]
                        nh[hold] += 1
                        shrink[~hold] -= d[~hold]
                        nn_[~hold] += 1
                    rows.append([m, e["round"], float(np.mean(gain / np.maximum(nh, 1))),
                                 float(np.mean(shrink / np.maximum(nn_, 1)))])
                prev = e["row_norm_global"]
    if not rows:
        return []
    f, ax = ps.figure("column", 2.4)
    for m in sorted({r[0] for r in rows}):
        rr = [r for r in rows if r[0] == m]
        st = ps.style(m)
        ax.plot([r[1] for r in rr], [r[2] for r in rr], label=f"{m}: clients holding the class (growth)", **st)
        ax.plot([r[1] for r in rr], [r[3] for r in rr], label=f"{m}: clients lacking the class (shrinkage)", **dict(st, linestyle=":"))
    ax.set_xlabel("Round")
    ax.set_ylabel("Change of the row norm per round")
    ax.legend()
    p = ps.save(f, out, "fig_34_holder_gain_vs_shrink", rows, ["method", "round", "holder_gain", "nonholder_shrink"])
    return [(p, cap.simple(34, E, S, "Mean change of a class row's norm in one round of local training: growth on "
                                     "clients that hold the class against shrinkage on clients that do not (class-row "
                                     "generators, first seed)."))]


@fig(35, "E")
def f35(E, S, out):
    rows = []
    for m in E.methods:
        for r in E.of(m, S):
            if r.get("cond_norms_end"):
                holders = (np.array(r["client_counts"]) > 0).sum(0)
                rows += [[m, int(h), float(v)] for h, v in zip(holders, r["cond_norms_end"])]
    if not rows:
        return []
    f, ax = ps.figure("column", 2.4)
    for m in sorted({r[0] for r in rows}):
        rr = [r for r in rows if r[0] == m]
        st = ps.style(m)
        ax.scatter([r[1] for r in rr], [r[2] for r in rr], s=8, marker=st["marker"], color=st["color"], label=m)
    ax.set_xlabel("Clients holding the class")
    ax.set_ylabel("Final class-row norm")
    ax.legend()
    p = ps.save(f, out, "fig_35_row_norm_vs_holders", rows, ["method", "holders", "norm"])
    return [(p, cap.simple(35, E, S, "Final norm of each class's conditioning row against the number of clients that "
                                     "hold the class; one point per class and seed."))]


@fig(36, "E")
def f36(E, S, out):
    return _lines(E, S, out, 36, "stage2_round",
                  lambda e: (float(np.mean([np.mean(v) for v in e["row_v"].values()])) if e.get("row_v") else None),
                  "Adam second moment of rows", "adam_row_v", "Mean Adam second-moment estimate of the class rows over "
                  "clients during Stage II", scale=1.0, split=None)


@fig(37, "E")
def f37(E, S, out):
    rows = []
    for m in E.methods:
        a, b = metric(E, m, "final_bal", S), metric(E, m, "er_bal", S)
        if not np.isnan(a[0]) and not np.isnan(b[0]):
            rows.append([m, a[0] * 100, b[0] * 100, a[1] * 100, b[1] * 100])
    if not rows:
        return []
    f, ax = ps.figure("page", 2.5)
    x = np.arange(len(rows))
    ax.bar(x - 0.18, [r[1] for r in rows], 0.36, yerr=[r[3] for r in rows], label="As trained",
           color="#BBBBBB", edgecolor="black", linewidth=0.4, error_kw=dict(lw=0.6))
    ax.bar(x + 0.18, [r[2] for r in rows], 0.36, yerr=[r[4] for r in rows], label="Exact readout", color="#333333",
           edgecolor="black", linewidth=0.4, hatch="///", error_kw=dict(lw=0.6))
    ax.set_xticks(x)
    ax.set_xticklabels([r[0] for r in rows], rotation=35, ha="right")
    ax.set_ylabel("Balanced accuracy (%)")
    ax.legend(ncol=2)
    p = ps.save(f, out, "fig_37_readout_gap", rows, ["method", "as_trained", "exact_readout", "sd_trained", "sd_er"])
    return [(p, cap.simple(37, E, S, "Balanced accuracy of the heads as trained and after the exact class-balanced "
                                     "readout (last layer recomputed from secure sums of the clients' embeddings); "
                                     "mean over seeds, error bars one standard deviation."))]


# ---------------------------------------------------------------- F: comparison
@fig(38, "F")
def f38(E, S, out):
    rows = []
    for m in E.methods:
        v = metric(E, m, "final_bal", S)
        if not np.isnan(v[0]):
            rows.append([m, v[0] * 100, v[1] * 100])
    return _bars(E, S, out, 38, rows, "Balanced accuracy (%)", "balanced_accuracy",
                 "Balanced accuracy on the test set (mean of the class recalls) of every method; mean over seeds, "
                 "error bars one standard deviation.")


@fig(39, "F")
def f39(E, S, out):
    from scipy import stats
    ours = {r["seed"]: r for r in E.of(OURS, S)}
    if not ours:
        return []
    rows = []
    for m in E.methods:
        if m == OURS:
            continue
        other = {r["seed"]: r for r in E.of(m, S)}
        common = sorted(set(ours) & set(other))
        d = [100 * (ours[s]["final_bal"] - other[s]["final_bal"]) for s in common
             if ours[s].get("final_bal") is not None and other[s].get("final_bal") is not None]
        if len(d) < 1:
            continue
        mu = float(np.mean(d))
        half = float(stats.t.ppf(0.975, len(d) - 1) * np.std(d, ddof=1) / np.sqrt(len(d))) if len(d) > 1 else 0.0
        rows.append([m, mu, mu - half, mu + half, len(d)])
    if not rows:
        return []
    f, ax = ps.figure("column", 0.25 * len(rows) + 0.8)
    y = np.arange(len(rows))[::-1]
    for yi, r in zip(y, rows):
        ax.errorbar(r[1], yi, xerr=[[r[1] - r[2]], [r[3] - r[1]]], fmt="o", color="black", markersize=3,
                    elinewidth=0.8, capsize=2)
    ax.axvline(0, color="#999999", linewidth=0.6, linestyle=":")
    ax.set_yticks(y)
    ax.set_yticklabels([r[0] for r in rows])
    ax.set_xlabel("Difference in balanced accuracy (points)")
    p = ps.save(f, out, "fig_39_forest", rows, ["baseline", "mean_diff", "ci_low", "ci_high", "n_seeds"])
    if max(r[4] for r in rows) == 1:   # one seed: a difference on the same split, no interval
        return [(p, cap.simple(39, E, S, "Difference in balanced accuracy, FedECS-LT minus each baseline, on the same "
                                         "client split (one seed, so no confidence interval); right of the dotted line "
                                         "FedECS-LT is better."))]
    return [(p, cap.simple(39, E, S, "Paired difference in balanced accuracy, FedECS-LT minus each baseline, over the "
                                     "same seeds and splits; points are means, bars 95% t confidence intervals (none "
                                     "for a baseline with one common seed); right of the dotted line FedECS-LT is "
                                     "better."))]


@fig(40, "F")
def f40(E, S, out):
    """Average rank over seeds x main metrics (critical-difference style, Nemenyi distance shown)."""
    keys = ["final_bal", "final_f1", "final_precision", "final_mcc", "final_gmean", "final_tail", "final_worst",
            "final_auc_pr"]
    seeds = sorted({r["seed"] for r in E.runs if setting_key(r) == S})
    blocks = []
    for s in seeds:
        for k in keys:
            vals = {}
            for m in E.methods:
                rr = [r for r in E.of(m, S) if r["seed"] == s and r.get(k) is not None]
                if rr:
                    vals[m] = rr[0][k]
            if len(vals) == len(E.methods):
                blocks.append(vals)
    if len(blocks) < 2 or len(E.methods) < 3:
        return []
    from scipy.stats import rankdata
    R = np.array([rankdata([-b[m] for m in E.methods]) for b in blocks])
    avg = R.mean(0)
    k, N = len(E.methods), len(blocks)
    q = {3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164}.get(k, 3.164 + 0.02 * (k - 10))
    cd = q * np.sqrt(k * (k + 1) / (6 * N))
    order = np.argsort(avg)
    f, ax = ps.figure("page", 0.18 * k + 1.0)
    for i, j in enumerate(order):
        ax.plot([avg[j]], [i], "o", color=ps.style(E.methods[j])["color"], markersize=4)
        ax.text(avg[j] + 0.08, i, E.methods[j], va="center")
    ax.plot([1, 1 + cd], [-1, -1], color="black", linewidth=1.0)
    ax.text(1 + cd / 2, -1.6, f"CD = {cd:.2f}", ha="center")
    ax.set_yticks([])
    ax.set_xlabel("Average rank (1 = best)")
    ax.set_ylim(-2.2, k)
    p = ps.save(f, out, "fig_40_critical_difference", [[E.methods[j], float(avg[j])] for j in order],
                ["method", "average_rank"])
    return [(p, cap.simple(40, E, S, f"Average rank of every method over {N} (seed, metric) pairs (balanced accuracy, "
                                     "macro-F1, macro precision, MCC, G-mean, tail recall, worst-class recall, macro "
                                     f"AUC-PR); the bar is the Nemenyi critical difference CD = {cd:.2f} at the 5% "
                                     "level."))]


@fig(41, "F")
def f41(E, S, out):
    keys = [("final_bal", "Bal. acc."), ("final_f1", "Macro-F1"), ("final_precision", "Precision"),
            ("final_mcc", "MCC"), ("final_gmean", "G-mean"), ("final_tail", "Tail recall"),
            ("final_worst", "Worst class"), ("final_auc_pr", "AUC-PR")]
    # FedECS-LT and the five other methods with the highest balanced accuracy
    others = sorted((m for m in E.methods if m != OURS and not np.isnan(metric(E, m, "final_bal", S)[0])),
                    key=lambda m: -metric(E, m, "final_bal", S)[0])
    ms_ = ([OURS] if OURS in E.methods else []) + others[:5]
    vals = {m: [metric(E, m, k, S)[0] for k, _ in keys] for m in ms_}
    vals = {m: v for m, v in vals.items() if not any(np.isnan(v))}
    if len(vals) < 2:
        return []
    import matplotlib.pyplot as plt
    ps.apply()
    f = plt.figure(figsize=(ps.COLUMN_IN, 3.0))
    ax = f.add_subplot(111, polar=True)
    ang = np.linspace(0, 2 * np.pi, len(keys), endpoint=False).tolist()
    for m, v in vals.items():
        st = ps.style(m)
        ax.plot(ang + ang[:1], [x * 100 for x in v] + [v[0] * 100], color=st["color"], linestyle=st["linestyle"],
                linewidth=st["linewidth"], label=m)
    ax.set_xticks(ang)
    ax.set_xticklabels([n for _, n in keys])
    # radial scale from 0 with at most four labelled rings, labels between two spokes (narrow ranges of values
    # otherwise give many rings whose labels overlap)
    top = max(10.0, float(np.ceil(max(max(v) for v in vals.values()) * 100 / 10.0) * 10))
    ax.set_ylim(0, top)
    step = top / 4
    ax.set_yticks([step * i for i in range(1, 5)])
    ax.set_yticklabels([f"{step * i:.0f}" for i in range(1, 5)])
    ax.set_rlabel_position(np.degrees(ang[1]) / 2)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=2)
    p = ps.save(f, out, "fig_41_radar", [[m] + v for m, v in vals.items()], ["method"] + [n for _, n in keys])
    return [(p, cap.simple(41, E, S, "Main metrics (%, test set, mean over seeds) of FedECS-LT and the five other "
                                     "methods with the highest balanced accuracy; larger is better on every axis."))]


@fig(42, "F")
def f42(E, S, out):
    return _pareto(E, S, out, 42, "comm_mb_per_client", "Communication per client (MB)", "accuracy_vs_communication",
                   "communication per client over the whole training", logx=True)


@fig(43, "F")
def f43(E, S, out):
    return _pareto(E, S, out, 43, "time_s", "Wall-clock time per run (s)", "accuracy_vs_time",
                   "wall-clock time of one run on one GPU", logx=True)


def _pareto(E, S, out, num, key, xlabel, name, what, logx=False):
    rows = []
    f, ax = ps.figure("column", 2.4)
    for m in E.methods:
        x, y = metric(E, m, key, S), metric(E, m, "final_bal", S)
        if np.isnan(x[0]) or np.isnan(y[0]):
            continue
        st = ps.style(m)
        ax.scatter([x[0]], [y[0] * 100], marker=st["marker"], color=st["color"], s=16, label=m, linewidths=0.4)
        rows.append([m, x[0], y[0] * 100])
    if not rows:
        return []
    if logx:
        logx = ps.log_axis(ax, [r[1] for r in rows])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Balanced accuracy (%)")
    ax.legend(ncol=2)
    p = ps.save(f, out, f"fig_{num:02d}_{name}", rows, ["method", "x", "balanced_accuracy"])
    return [(p, cap.simple(num, E, S, f"Balanced accuracy against {what}" + (" (logarithmic axis)" if logx else "")
                                      + "; mean over seeds. Up and left is better."))]


@fig(44, "F")
def f44(E, S, out):
    keys = [("final_bal", "Bal."), ("final_acc", "Acc."), ("final_f1", "F1"), ("final_precision", "Prec."),
            ("final_mcc", "MCC"), ("final_kappa", "Kappa"), ("final_gmean", "G-mean"), ("final_tail", "Tail"),
            ("final_worst", "Worst"), ("final_auc_pr", "AUC-PR"), ("final_auc_roc", "AUC-ROC"),
            ("er_bal", "Bal. acc. (readout)")]
    M = np.array([[metric(E, m, k, S)[0] * 100 for k, _ in keys] for m in E.methods])
    if M.size == 0 or np.all(np.isnan(M)):
        return []
    f, ax = ps.figure("page", 0.22 * len(E.methods) + 0.9)
    Z = (M - np.nanmean(M, 0)) / np.where(np.nanstd(M, 0) > 0, np.nanstd(M, 0), 1)
    im = ax.imshow(Z, aspect="auto", cmap="RdBu_r", vmin=-2.5, vmax=2.5)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            if not np.isnan(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.1f}", ha="center", va="center",
                        color="white" if abs(Z[i, j]) > 1.6 else "black")
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels([n for _, n in keys], rotation=35, ha="right")
    ax.set_yticks(range(len(E.methods)))
    ax.set_yticklabels(E.methods)
    f.colorbar(im, ax=ax, fraction=0.03, pad=0.02).set_label("Standardised score")
    p = ps.save(f, out, "fig_44_metric_heatmap", [[m] + list(r) for m, r in zip(E.methods, M)],
                ["method"] + [n for _, n in keys])
    return [(p, cap.simple(44, E, S, "Every main metric (%, test set, mean over seeds) of every method; colours are "
                                     "standardised per metric (red = better than the column mean); Bal. acc. "
                                     "(readout): balanced accuracy with the exact class-balanced readout."))]


@fig(45, "F")
def f45(E, S, out):
    return []  # ablation waterfall: made from the ablation experiment (its own notebook)


@fig(46, "F")
def f46(E, S, out):
    return []  # sensitivity lines: made from the sweep experiments (phase C)


# ---------------------------------------------------------------- G: system
@fig(47, "G")
def f47(E, S, out):
    rows = []
    for m in E.methods:
        for r in E.of(m, S)[:1]:
            for t, b in (r.get("bytes") or {}).items():
                rows.append([m, cap.message_name(t), (b["down"] + b["up"]) / 2 ** 20 / r["K"]])
    if not rows:
        return []
    types = sorted({r[1] for r in rows})
    ms_ = [m for m in E.methods if any(r[0] == m for r in rows)]
    f, ax = ps.figure("page", 2.6)
    bottom = np.zeros(len(ms_))
    for i, t in enumerate(types):
        v = np.array([sum(r[2] for r in rows if r[0] == m and r[1] == t) for m in ms_])
        ax.bar(np.arange(len(ms_)), v, 0.7, bottom=bottom, label=cap.message_name(t),
               color=ps.PALETTE[i % len(ps.PALETTE)],
               hatch=["", "///", "...", "xx", "\\\\"][i % 5], edgecolor="black", linewidth=0.3)
        bottom += v
    ax.set_xticks(np.arange(len(ms_)))
    ax.set_xticklabels(ms_, rotation=35, ha="right")
    ax.set_ylabel("MB per client")
    ax.legend(ncol=3)
    p = ps.save(f, out, "fig_47_communication", rows, ["method", "message_type", "mb_per_client"])
    return [(p, cap.simple(47, E, S, "Communication per client over the whole training, by message type (model "
                                     "weights, statistics, secure sums); first seed. Cached shared stages count once."))]


@fig(48, "G")
def f48(E, S, out):
    rows = []
    for m in E.methods:
        for st_ in ("fe_time_s", "gen_time_s", "head_time_s", "time_s"):
            v = metric(E, m, st_, S)
            if not np.isnan(v[0]):
                rows.append([m, st_, v[0]])
    if not rows:
        return []
    ms_ = [m for m in E.methods if any(r[0] == m for r in rows)]
    f, ax = ps.figure("page", 2.5)
    for i, (st_, lab) in enumerate((("fe_time_s", "Stage I"), ("gen_time_s", "Stage II"), ("time_s", "Whole run"))):
        v = [next((r[2] for r in rows if r[0] == m and r[1] == st_), 0) for m in ms_]
        ax.bar(np.arange(len(ms_)) + (i - 1) * 0.27, v, 0.27, label=lab, color=["#BBBBBB", "#777777", "#333333"][i],
               edgecolor="black", linewidth=0.3)
    ax.set_xticks(np.arange(len(ms_)))
    ax.set_xticklabels(ms_, rotation=35, ha="right")
    ax.set_ylabel("Time (s)")
    ax.legend(ncol=3)
    p = ps.save(f, out, "fig_48_time_per_stage", rows, ["method", "stage", "seconds"])
    return [(p, cap.simple(48, E, S, "Wall-clock time per stage on one GPU (simulation of all clients), mean over "
                                     "seeds; stages shared by several methods are trained once per seed and report the time of that training."))]


@fig(49, "G")
def f49(E, S, out):
    rows = []
    for m in E.methods:
        v = metric(E, m, "gpu_peak_mb", S)
        if not np.isnan(v[0]):
            rows.append([m, v[0], v[1]])
    return _bars(E, S, out, 49, rows, "Peak GPU memory (MB)", "gpu_memory",
                 "Peak GPU memory of the server process during one run (mean over seeds).")


@fig(50, "G")
def f50(E, S, out):
    rows = []
    for r in E.runs:
        if setting_key(r) != S or r.get("pfrac", 1.0) >= 1.0:
            continue
        for e in E.log(r, "stage3_round"):
            rows.append([r["label"], e["round"], len(e["clients"])])
    if not rows:
        return []
    f, ax = ps.figure("column", 2.2)
    for m in sorted({r[0] for r in rows}):
        rr = [r for r in rows if r[0] == m]
        ax.plot([r[1] for r in rr], [r[2] for r in rr], label=m, **ps.style(m))
    ax.set_xlabel("Round")
    ax.set_ylabel("Participating clients")
    p = ps.save(f, out, "fig_50_participation", rows, ["method", "round", "clients"])
    return [(p, cap.simple(50, E, S, "Number of clients taking part in each round of Stage III (partial "
                                     "participation: the same seeded subset for every method)."))]


@fig(51, "G")
def f51(E, S, out):
    rows = []
    f, ax = ps.figure("column", 2.4)
    for m in E.methods:
        rs = E.of(m, S)
        if not rs or not rs[0].get("comm_mb_per_client"):
            continue
        c = _curve(E, m, S, "stage3_round", lambda e: (e.get("post") or {}).get("bal"))
        if c is None:
            continue
        x, mu, _ = c
        total = rs[0]["comm_mb_per_client"]
        xx = total * x / x.max()
        ax.plot(xx, mu * 100, label=m, markevery=max(1, len(x) // 6), **ps.style(m))
        rows += [[m, float(a), float(b * 100)] for a, b in zip(xx, mu)]
    if not rows:
        return []
    ax.set_xlabel("Cumulative communication per client (MB)")
    ax.set_ylabel("Balanced accuracy (%)")
    ax.legend(ncol=2)
    p = ps.save(f, out, "fig_51_accuracy_vs_cumulative_communication", rows, ["method", "mb", "bal"])
    return [(p, cap.simple(51, E, S, "Balanced accuracy of the aggregated heads against cumulative communication "
                                     "per client (Stage III rounds placed proportionally on the total)."))]


# ---------------------------------------------------------------- H, I: privacy and diagram (own experiments)
@fig(52, "H")
def f52(E, S, out):
    rows = []
    for r in E.runs:
        if setting_key(r) == S and r.get("dp_eps") is not None:
            rows.append([r["label"], r["dp_eps"], r.get("final_bal")])
    if not rows:
        return []
    f, ax = ps.figure("column", 2.3)
    eps = sorted({r[1] for r in rows})
    ax.plot(eps, [np.mean([r[2] for r in rows if r[1] == e]) * 100 for e in eps], **ps.style(OURS))
    ps.log_axis(ax, eps)
    ax.set_xlabel("Privacy budget epsilon")
    ax.set_ylabel("Balanced accuracy (%)")
    p = ps.save(f, out, "fig_52_accuracy_vs_epsilon", rows, ["method", "epsilon", "bal"])
    return [(p, cap.simple(52, E, S, "Balanced accuracy of FedECS-LT with differentially private statistics "
                                     "release against the privacy budget (delta = 1e-5)."))]


@fig(53, "H")
def f53(E, S, out):
    return []  # membership-inference ROC: privacy experiment


@fig(54, "H")
def f54(E, S, out):
    return []  # release noise vs class size: privacy experiment


@fig(55, "I")
def f55(E, S, out):
    return []  # pipeline diagram: drawn once (docs/latex), not per experiment


# ---------------------------------------------------------------- section 17b: aggregation effect
@fig(56, "17b")
def f56(E, S, out):
    res = []
    for m in [x for x in (OURS, REF) if x in E.methods]:
        rs = E.of(m, S)
        if not rs:
            continue
        r = rs[0]
        if _one_client_per_arch(r):
            continue
        rows, M, rounds = [], [], []
        for e in E.log(r, "stage3_round"):
            if not e.get("pre") or not e.get("post"):
                continue
            pre = np.mean([np.array(v["recall_c"]) for v in e["pre"].values()], 0)
            post = np.array(e["post"]["recall_c"])
            M.append((post - pre) * 100)
            rounds.append(e["round"])
            rows += [[e["round"], c, float(x)] for c, x in enumerate((post - pre) * 100)]
        if not M:
            continue
        A = np.array(M).T
        f, ax = ps.figure("column", 2.4)
        lim = max(1e-6, float(np.abs(A).max()))
        im = ax.imshow(A, aspect="auto", cmap="RdBu", vmin=-lim, vmax=lim, origin="lower")
        _round_ticks(ax, rounds)
        ax.set_xlabel("Round")
        ax.set_ylabel("Class")
        f.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("Recall change by aggregation (points)")
        name = "ours" if m == OURS else "geflf"
        p = ps.save(f, out, f"fig_56_aggregation_effect_{name}", rows, ["round", "class", "effect"])
        res.append((p, cap.simple(56, E, S, f"Aggregation effect per class for {cap.mname(m)}: recall of the aggregated "
                                            "heads minus the mean recall of the clients' local heads just before "
                                            "aggregation, at every evaluation round of Stage III (blue = aggregation "
                                            "helps); first seed.")))
    return res


@fig(57, "17b")
def f57(E, S, out):
    rows = []
    for m in E.methods:
        for r in E.of(m, S)[:1]:
            for e in E.log(r, "stage2_round"):
                if e.get("fidelity_pre") and e.get("fidelity_post"):
                    C = len(e["fidelity_post"])
                    tl = class_groups(C)[2]
                    pre = np.mean([np.array(v)[tl].mean() for v in e["fidelity_pre"].values()])
                    rows.append([m, e["round"], float(pre) * 100, float(np.mean(np.array(e["fidelity_post"])[tl])) * 100])
    if not rows:
        return []
    f, ax = ps.figure("column", 2.4)
    for m in sorted({r[0] for r in rows}):
        rr = [r for r in rows if r[0] == m]
        st = ps.style(m)
        ax.plot([r[1] for r in rr], [r[2] for r in rr], label=f"{m}: clients' copies (before aggregation)", **dict(st, linestyle=":"))
        ax.plot([r[1] for r in rr], [r[3] for r in rr], label=f"{m}: aggregated generator", **st)
    ax.set_xlabel("Round")
    ax.set_ylabel("Tail-class fidelity (%)")
    ax.legend()
    p = ps.save(f, out, "fig_57_fidelity_pre_post", rows, ["method", "round", "pre", "post"])
    return [(p, cap.simple(57, E, S, "Tail-class fidelity of the clients' generator copies just before aggregation "
                                     "(mean over clients) and of the aggregated generator, during Stage II."))]


@fig(58, "17b")
def f58(E, S, out):
    rows = []
    for m in E.methods:
        for r in E.of(m, S)[:1]:
            counts = np.array(r["client_counts"])
            C = counts.shape[1]
            tl = class_groups(C)[2]
            for e in E.log(r, "stage2_round"):
                if not e.get("row_norm_local") or not e.get("row_norm_global"):
                    continue
                h, n_ = [], []
                for k, v in e["row_norm_local"].items():
                    v = np.array(v)
                    hold = counts[int(k)] > 0
                    for c in range(C)[tl]:
                        (h if hold[c] else n_).append(v[c])
                rows.append([m, e["round"], float(np.mean(h)) if h else None, float(np.mean(n_)) if n_ else None,
                             float(np.mean(np.array(e["row_norm_global"])[tl]))])
    if not rows:
        return []
    f, ax = ps.figure("column", 2.4)
    for m in sorted({r[0] for r in rows}):
        rr = [r for r in rows if r[0] == m]
        st = ps.style(m)
        ax.plot([r[1] for r in rr], [r[2] for r in rr], label=f"{m}: clients holding the class", **dict(st, linestyle="--"))
        ax.plot([r[1] for r in rr], [r[3] for r in rr], label=f"{m}: clients lacking the class", **dict(st, linestyle=":"))
        ax.plot([r[1] for r in rr], [r[4] for r in rr], label=f"{m}: aggregated generator", **dict(st, linestyle="-", linewidth=1.6))
    ax.set_xlabel("Round")
    ax.set_ylabel("Norm of the tail-class rows")
    ax.legend()
    p = ps.save(f, out, "fig_58_row_norm_holders_nonholders", rows,
                ["method", "round", "holders", "non_holders", "aggregate"])
    return [(p, cap.simple(58, E, S, "Norm of the tail classes' conditioning rows in the copies of clients that hold "
                                     "the class, of clients that do not, and in the aggregate, during Stage II "
                                     "(class-row generators, first seed)."))]


@fig(59, "17b")
def f59(E, S, out):
    rows = []
    for m in [x for x in (OURS, REF) if x in E.methods]:
        for r in [x for x in E.of(m, S)[:1] if not _one_client_per_arch(x)]:
            holders = (np.array(r["client_counts"]) > 0).sum(0)
            effs = []
            for e in E.log(r, "stage3_round"):
                if e.get("pre") and e.get("post"):
                    pre = np.mean([np.array(v["recall_c"]) for v in e["pre"].values()], 0)
                    effs.append(np.array(e["post"]["recall_c"]) - pre)
            if effs:
                eff = np.mean(effs, 0) * 100
                rows += [[m, int(h), float(x)] for h, x in zip(holders, eff)]
    if not rows:
        return []
    f, ax = ps.figure("column", 2.4)
    for m in sorted({r[0] for r in rows}):
        rr = [r for r in rows if r[0] == m]
        st = ps.style(m)
        ax.scatter([r[1] for r in rr], [r[2] for r in rr], s=9, marker=st["marker"], color=st["color"], label=m)
    ax.axhline(0, color="#999999", linewidth=0.6, linestyle=":")
    ax.set_xlabel("Clients holding the class")
    ax.set_ylabel("Mean aggregation effect (points)")
    ax.legend()
    p = ps.save(f, out, "fig_59_effect_vs_holders", rows, ["method", "holders", "effect"])
    return [(p, cap.simple(59, E, S, "Mean aggregation effect on a class's recall (aggregate minus local heads, over "
                                     "Stage III) against the number of clients holding the class; one point per "
                                     "class."))]


@fig(60, "17b")
def f60(E, S, out):
    loc = E.of("Local", S)
    if not loc:
        return []
    l_rec = np.mean([r["final_recall_c"] for r in loc], 0) * 100
    rows = []
    f, ax = ps.figure("page", 2.4)
    x = np.arange(len(l_rec))
    for m in [x_ for x_ in (OURS, REF) if x_ in E.methods]:
        v = np.mean([r["final_recall_c"] for r in E.of(m, S)], 0) * 100
        ax.plot(x, v - l_rec, label=m, **ps.style(m))
        rows += [[m, int(c), float(d)] for c, d in zip(x, v - l_rec)]
    if not rows:
        return []
    ax.axhline(0, color="#999999", linewidth=0.6, linestyle=":")
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel("Recall gain over Local (points)")
    ax.legend()
    p = ps.save(f, out, "fig_60_federated_vs_local", rows, ["method", "class", "gain"])
    return [(p, cap.simple(60, E, S, "Per-class benefit of federation: recall of each federated method minus the "
                                     "recall of local-only training (each client trains on its own data), mean over "
                                     "seeds."))]


# ---------------------------------------------------------------- 74-82: head stage, representation, generator
def _probe_mean(e, point, key):
    vals = [c[point][key] for c in (e.get("probe") or {}).values()
            if c.get(point) and c[point].get(key) is not None]
    return float(np.mean(vals)) if vals else None


def _probe_change(e, key):
    vals = [c["after_syn"][key] - c["start"][key] for c in (e.get("probe") or {}).values()
            if c.get("after_syn") and c.get("start") and c["after_syn"].get(key) is not None
            and c["start"].get(key) is not None]
    return float(np.mean(vals)) if vals else None


@fig(74, "A")
def f74(E, S, out):
    return _lines(E, S, out, 74, "stage3_round", lambda e: _probe_mean(e, "end", "logit_spread"),
                  "Logit spread", "logit_spread_rounds",
                  "Logit spread of the clients' heads on their own real features at the end of each head-training "
                  "round (mean over images of the standard deviation of the logits over classes; mean over clients)",
                  scale=1.0, split=None)


@fig(75, "A")
def f75(E, S, out):
    return _lines(E, S, out, 75, "stage3_round", lambda e: _probe_change(e, "logit_spread"),
                  "Change of the logit spread", "logit_spread_synthetic_phase",
                  "Change of the logit spread of the clients' heads on their own real features during the synthetic "
                  "phase of each head-training round (after minus before; mean over clients; methods with a "
                  "synthetic phase)", scale=1.0, split=None)


@fig(76, "A")
def f76(E, S, out):
    return _lines(E, S, out, 76, "stage3_round", lambda e: _probe_change(e, "bal"),
                  "Change of balanced accuracy (points)", "retention_synthetic_phase",
                  "Retention: change of each client's balanced accuracy on its own real features (over the classes "
                  "it holds) during the synthetic phase of each head-training round (after minus before; mean over "
                  "clients; negative = the synthetic phase erases what the head knew)", split=None)


@fig(77, "A")
def f77(E, S, out):
    return _lines(E, S, out, 77, "stage3_round", lambda e: e.get("consensus_agreement"),
                  "Consensus agreement (%)", "consensus_agreement",
                  "Consensus agreement: share of the pooled generated features whose consensus soft label (the "
                  "averaged prediction of all head architectures) puts the most mass on the generated class, per "
                  "head-training round", split=None)


def _collapse(E, m, S):
    """Neural-collapse statistics of every head architecture (and seed) of a method from its saved test embeddings
    and last layers."""
    out = []
    for r in E.of(m, S):
        d = E.pred(r)
        if d is None or "y" not in d.files:   # runs without saved embeddings (one-shared-model baselines)
            continue
        y = d["y"]
        for k in d.files:
            if not k.startswith("embed_"):
                continue
            g = k[len("embed_"):]
            H = d[k].astype(np.float64)
            C = int(max(y.max() + 1, len(r.get("class_order") or [])))
            row = rp.nc2(H, y, C)
            ok = rp.knn_correct(H, y)
            tl = np.isin(y, np.arange(C)[class_groups(C)[2]])
            row.update(knn=float(ok.mean()), knn_tail=float(ok[tl].mean()) if tl.any() else None)
            if f"fcw_{g}" in d.files:
                W = d[f"fcw_{g}"]
                row["nc3"] = rp.nc3(H, y, C, W)
                row["minority_angle"] = rp.minority_angle(W, range(C)[class_groups(C)[2]])
            out.append(row)
    return out


@fig(78, "C")
def f78(E, S, out):
    keys = [("nc2_equinorm", "NC2 equinorm", "NC2 equinorm, the coefficient of variation of the centred class-mean "
             "norms"),
            ("nc2_equiangle", "NC2 equiangularity", "NC2 equiangularity, the standard deviation of their pairwise "
             "cosines"),
            ("nc3", "NC3 self-duality", "NC3 self-duality, the squared distance between the normalised classifier "
             "and the normalised class means"),
            ("minority_angle", "Tail classifier angle (deg)", "mean pairwise angle between the classifier vectors of "
             "the tail classes (minority collapse, Fang et al., PNAS 2021: small angles)")]
    rows = []
    for m in E.methods:
        vals = _collapse(E, m, S)
        for k, _, _ in keys:
            v = [x[k] for x in vals if x.get(k) is not None]
            if v:
                rows.append([m, k, float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0])
    keys = [kk for kk in keys if any(r[1] == kk[0] for r in rows)]   # only the statistics with data get a panel
    if not rows:
        return []
    nrows = (len(keys) + 1) // 2
    f, axes = ps.figure("page", 2.9 * nrows, ncols=2, nrows=nrows)
    f.subplots_adjust(hspace=1.0, wspace=0.3)   # room for the rotated method names and the panel labels
    axes = list(np.ravel(axes))
    for ax in axes[len(keys):]:
        ax.set_visible(False)
    axes = axes[:len(keys)]
    for ax, (k, name, _) in zip(axes, keys):
        rr = [r for r in rows if r[1] == k]
        for i, r in enumerate(rr):
            st = ps.style(r[0])
            ax.bar(i, r[2], 0.7, yerr=r[3], color=st["color"], edgecolor="black", linewidth=0.4,
                   hatch="///" if r[0] == OURS else None, error_kw=dict(lw=0.6))
        ax.set_xticks(np.arange(len(rr)))
        ax.set_xticklabels([r[0] for r in rr], rotation=45, ha="right", rotation_mode="anchor")
        ax.set_ylabel(name)
    ps.panel_labels(axes, y=-0.62)
    ps.room_between_rows(f, axes, ncols=2)
    p = ps.save(f, out, "fig_78_neural_collapse_nc2_nc3", rows, ["method", "statistic", "mean", "sd"])
    parts = "; ".join(f"({chr(97 + i)}) {text}" for i, (_, _, text) in enumerate(keys))
    return [(p, cap.simple(78, E, S, "Neural collapse (Papyan et al., PNAS 2020) of the test-set embeddings and last "
                                     f"layers (NC2 and NC3: lower = closer to collapse): {parts}; mean over "
                                     "architectures and seeds."))]


@fig(79, "C")
def f79(E, S, out):
    ms_ = [m for m in (OURS, REF) if m in E.methods]
    mats = []
    for m in ms_:
        rs = E.of(m, S)
        d = E.pred(rs[0]) if rs else None
        keys = sorted((k for k in (d.files if d is not None else []) if k.startswith("embed_")),
                      key=lambda k: int(k.split("_")[1]))
        if len(keys) < 2:
            continue
        M = np.array([[rp.linear_cka(d[a].astype(np.float64), d[b].astype(np.float64)) for b in keys] for a in keys])
        mats.append((m, M, [int(k.split("_")[1]) + 1 for k in keys]))
    if not mats:
        return []
    f, axes = ps.figure("page", 3.0, ncols=len(mats))
    f.subplots_adjust(wspace=0.55)   # the second panel's row labels clear the first panel
    axes = list(np.ravel(axes)) if len(mats) > 1 else [axes]
    rows = []
    for ax, (m, M, archs) in zip(axes, mats):
        im = ax.imshow(M, vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(range(len(archs)))
        ax.set_xticklabels([f"CNN-{a}" for a in archs], rotation=90)
        ax.set_yticks(range(len(archs)))
        ax.set_yticklabels([f"CNN-{a}" for a in archs])
        rows += [[m, archs[i], archs[j], float(M[i, j])] for i in range(len(archs)) for j in range(len(archs))]
    f.colorbar(im, ax=axes, fraction=0.046, pad=0.04).set_label("Linear CKA")
    if len(mats) > 1:
        ps.panel_labels(axes, y=-0.45)
    p = ps.save(f, out, "fig_79_cka_architectures", rows, ["method", "arch_a", "arch_b", "cka"])
    return [(p, cap.simple(79, E, S, "Linear centred kernel alignment (CKA; Kornblith et al., ICML 2019) between the test-set embeddings of every "
                                     "pair of head architectures (same 2,000 test images; 1 = the same representation "
                                     "up to rotation and scale): " + " and ".join(
                                         f"({chr(97 + i)}) {cap.mname(m)}" for i, (m, _, _) in enumerate(mats))
                                     + "; first seed."))]


def _gen_class_stat(E, S, fn):
    rows = []
    for m in E.methods:
        per = []
        for r in E.of(m, S):
            d = E.pred(r)
            if d is None or "gen_x" not in d.files or "val_x" not in d.files:
                continue
            Xg, yg, Xr, yr = (d["gen_x"].astype(np.float64), d["gen_y"], d["val_x"].astype(np.float64),
                              d["val_y"])
            C = int(max(yg.max(), yr.max()) + 1)
            per.append([fn(Xg[yg == c][:300], Xr[yr == c][:300]) if (yg == c).sum() > 1 and (yr == c).sum() > 1
                        else np.nan for c in range(C)])
        if per:
            mu = np.nanmean(np.array(per, dtype=float), 0)
            rows += [[m, c, float(v)] for c, v in enumerate(mu)]
    return rows


def _gen_class_fig(E, S, out, num, fn, ylabel, name, text):
    rows = _gen_class_stat(E, S, fn)
    if not rows:
        return []
    f, ax = ps.figure("page", 2.4)
    for m in sorted({r[0] for r in rows}, key=E.methods.index):
        rr = [r for r in rows if r[0] == m]
        ax.plot([r[1] for r in rr], [r[2] for r in rr], label=m, markevery=max(1, len(rr) // 12), **ps.style(m))
    ax.set_xlabel("Class (ordered from most to least frequent)")
    ax.set_ylabel(ylabel)
    ax.legend()
    p = ps.save(f, out, f"fig_{num:02d}_{name}", rows, ["method", "class", "value"])
    return [(p, cap.simple(num, E, S, text))]


@fig(80, "D")
def f80(E, S, out):
    return _gen_class_fig(E, S, out, 80, rp.mmd_rbf, "MMD$^2$", "per_class_mmd",
                          "Per-class squared maximum mean discrepancy (MMD; Gretton et al., JMLR 2012) between the generated features and the real "
                          "validation features of the class (Gaussian kernel, bandwidth = median distance; at most 300 "
                          "of each; 0 = same distribution); methods with a generator, mean over seeds.")


@fig(81, "D")
def f81(E, S, out):
    return _gen_class_fig(E, S, out, 81, rp.mean_mismatch, "Mean mismatch (SD units)", "per_class_mean_mismatch",
                          "Per-class distance between the mean of the generated features and the mean of the real "
                          "validation features, in units of the real within-class standard deviation; methods with a "
                          "generator, mean over seeds.")


@fig(82, "D")
def f82(E, S, out):
    rows = []
    for m in E.methods:
        v = [r["active_units"] / r["latent_dim"] * 100 for r in E.of(m, S) if r.get("active_units") is not None
             and r.get("latent_dim")]
        if v:
            rows.append([m, float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0])
    return _bars(E, S, out, 82, rows, "Active latent units (%)", "active_latent_units",
                 "Active latent units of the generator: share of latent dimensions whose posterior mean varies over "
                 "the validation features by more than 0.01 (Burda et al., ICLR 2016); methods with a variational "
                 "generator, mean over seeds.", width="column")


def _embeddings(E, m, S):
    """First seed's saved test embeddings of a method's first head architecture: (H, y, names) or None."""
    rs = E.of(m, S)
    d = E.pred(rs[0]) if rs else None
    if d is None or "y" not in d.files:
        return None
    keys = sorted(k for k in d.files if k.startswith("embed_"))
    if not keys:
        return None
    return d[keys[0]].astype(np.float64), d["y"], rs[0].get("class_names")


@fig(83, "C")
def f83(E, S, out):
    """Side by side, the reference and ours on the same test images with the same t-SNE settings (the layout of
    the feature-space figures of federated long-tail papers, e.g. the FedLTLib benchmark), tail classes emphasised."""
    ms_ = [m for m in (REF, OURS) if m in E.methods]
    got = [(m, _embeddings(E, m, S)) for m in ms_]
    got = [(m, e) for m, e in got if e is not None]
    if len(got) < 2:
        return []
    f, axes = ps.figure("page", 3.4, ncols=2)
    f.subplots_adjust(wspace=0.08)
    rows, texts = [], []
    C = int(max(e[1].max() for _, e in got)) + 1
    for i, (ax, (m, (H, y, names))) in enumerate(zip(axes, got)):
        Z = _tsne(H)
        _scatter_classes(ax, Z, y, C, emphasise_tail=True)
        ax.set_xlabel("t-SNE 1")
        if i == 0:
            ax.set_ylabel("t-SNE 2")
        ok = rp.knn_correct(H, y)
        tail = np.isin(y, np.arange(C)[class_groups(C)[2]])
        texts.append(f"({chr(97 + i)}) {cap.mname(m)}: {100 * ok.mean():.1f}% (tail {100 * ok[tail].mean():.1f}%)")
        rows += [[m, float(a), float(b), int(c)] for (a, b), c in zip(Z, y)]
    _class_legend(axes[-1], C, names=got[-1][1][2])
    ps.panel_labels(axes, y=-0.1)
    p = ps.save(f, out, "fig_83_tsne_comparison", rows, ["method", "tsne1", "tsne2", "class"], raster=True)
    return [(p, cap.simple(83, E, S, "t-SNE of the test-set embeddings of the first head architecture (same 2,000 "
                                     "test images, first seed), coloured by true class (numbered from the most to the "
                                     "least frequent; the tail classes, the rarest 40%, drawn larger and opaque). "
                                     "Leave-one-out 10-nearest-neighbour accuracy of the embeddings: "
                                     + "; ".join(texts) + "." + _colour_note(C) + TSNE_NOTE))]


@fig(84, "C")
def f84(E, S, out):
    """Robustness of the map to its one free parameter: the same embeddings at three perplexities."""
    e = _embeddings(E, OURS, S)
    if e is None:
        return []
    H, y, names = e
    C = int(y.max()) + 1
    perps = (10, 30, 100)
    f, axes = ps.figure("page", 2.6, ncols=3)
    f.subplots_adjust(wspace=0.08)
    rows = []
    for ax, pp in zip(axes, perps):
        Z = _tsne(H, perplexity=pp)
        _scatter_classes(ax, Z, y, C, emphasise_tail=True)
        ax.set_xlabel("t-SNE 1")
        rows += [[pp, float(a), float(b), int(c)] for (a, b), c in zip(Z, y)]
    axes[0].set_ylabel("t-SNE 2")
    _class_legend(axes[-1], C)
    ps.panel_labels(axes, y=-0.1)
    p = ps.save(f, out, "fig_84_tsne_perplexities", rows, ["perplexity", "tsne1", "tsne2", "class"], raster=True)
    return [(p, cap.simple(84, E, S, f"t-SNE of {cap.mname(OURS)}'s test-set embeddings (first head architecture, same "
                                     "2,000 test images, first seed) at perplexity (a) 10, (b) 30 (the value of the "
                                     "other maps) and (c) 100: structure that appears at every perplexity is a "
                                     "property of the embeddings, not of the map (Wattenberg et al., Distill 2016). "
                                     "Tail classes (rarest 40%) drawn larger and opaque." + _colour_note(C)))]


def _umap(X, seed=0):
    """UMAP (McInnes, Healy and Melville, 2018) with its default 15 neighbours and minimum distance 0.1, on the same 50
    principal components as the t-SNE maps; one thread so that the seed makes it reproducible."""
    import umap
    from sklearn.decomposition import PCA
    X = X.astype(np.float32)
    if X.shape[1] > 50:
        X = PCA(50, random_state=seed).fit_transform(X)
    return umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=seed, n_jobs=1).fit_transform(X)


@fig(85, "C")
def f85(E, S, out):
    """UMAP of the same embeddings as the t-SNE maps: GeFL-F and FedECS-LT side by side when both are in the
    experiment, FedECS-LT alone otherwise."""
    got = [(m, _embeddings(E, m, S)) for m in (REF, OURS) if m in E.methods]
    got = [(m, e) for m, e in got if e is not None]
    if not got:
        return []
    two = len(got) == 2
    f, axes = ps.figure("page" if two else "column", 3.4 if two else 3.3, ncols=len(got))
    axes = list(np.ravel(axes)) if two else [axes]
    if two:
        f.subplots_adjust(wspace=0.08)
    rows = []
    C = int(max(e[1].max() for _, e in got)) + 1
    for i, (ax, (m, (H, y, names))) in enumerate(zip(axes, got)):
        Z = _umap(H)
        _scatter_classes(ax, Z, y, C, emphasise_tail=True)
        ax.set_xlabel("UMAP 1")
        if i == 0:
            ax.set_ylabel("UMAP 2")
        rows += [[m, float(a), float(b), int(c)] for (a, b), c in zip(Z, y)]
    _class_legend(axes[-1], C, names=got[-1][1][2])
    if two:
        ps.panel_labels(axes, y=-0.1)
    p = ps.save(f, out, "fig_85_umap", rows, ["method", "umap1", "umap2", "class"], raster=True)
    who = " and ".join(f"({chr(97 + i)}) {cap.mname(m)}" for i, (m, _) in enumerate(got)) if two else cap.mname(got[0][0])
    return [(p, cap.simple(85, E, S, f"UMAP (McInnes, Healy and Melville, 2018; 15 neighbours, minimum distance 0.1, on "
                                     "the same 50 principal components) of the test-set embeddings of the first head "
                                     f"architecture of {who} (same 2,000 test images as the t-SNE maps, first seed), "
                                     "coloured by true class (numbered from the most to the least frequent; the tail "
                                     "classes, the rarest 40%, drawn larger and opaque). UMAP keeps more of the "
                                     "arrangement of the clusters than t-SNE, but distances between clusters remain "
                                     "qualitative." + _colour_note(C)))]


def _arch(K):
    blk = max(K // 10, 1)
    return [min(k // blk, 9) for k in range(K)]


# ---------------------------------------------------------------- driver
def make_all(res_dir, out_dir):
    """Every figure of the catalogue for every setting of the experiment. Returns (figures, not_applicable), where
    figures = [(number, group, pdf path, caption)] and not_applicable = [(number, group, reason)]."""
    E = Exp(res_dir)
    figs, na = [], []
    ps.LAYOUT_PROBLEMS.clear()
    for S in E.settings:
        sub = os.path.join(out_dir, cap.setting_slug(S))
        for num, group, fn in sorted(FIGS):
            try:
                made = fn(E, S, sub)
            except Exception as e:  # a failing figure is reported in the report, never hides the others
                na.append((num, group, f"error: {e!r}"[:300], S))
                continue
            if made:
                figs += [(num, group, p, c) for p, c in made]
            else:
                na.append((num, group, cap.not_applicable(num), S))
    return figs, na, E


In [ ]:
%%writefile /kaggle/working/fedecs_lt/report.py
"""report.pdf of one experiment: every table and every figure the experiment produced, with captions, plus the run
list, failures and a file index. A completeness check fails the notebook if any figure file or any logged scalar
metric is missing from the PDF.

    python report.py --results RESULTS_DIR --title "..." --out report.pdf
"""
import argparse
import datetime
import glob
import json
import os
import platform
import re

import numpy as np

import plot_style as ps
import plots
import captions as cap
from captions import proto_note, setting_slug, setting_text
from settings import CITE, DATASETS

MAIN = [("final_bal", "Bal. acc."), ("final_f1", "Macro-F1"), ("final_precision", "Precision"), ("final_mcc", "MCC"),
        ("final_gmean", "G-mean"), ("final_tail", "Tail rec."), ("final_worst", "Worst rec."),
        ("final_auc_pr", "AUC-PR")]
SECOND = [("final_acc", "Accuracy"), ("final_kappa", "Kappa"), ("final_top5", "Top-5"), ("final_auc_roc", "AUC-ROC"),
          ("final_head", "Head rec."), ("final_medium", "Med. rec."), ("final_rec_std", "Recall SD")]
CLIENTS = [("final_worst_arch", "Worst-arch. bal."), ("final_local", "Own-mix acc."),
           ("final_own_mix_sd", "Own-mix SD"), ("final_own_mix_min", "Own-mix min.")]
CALIB = [("final_ece", "ECE"), ("final_cw_ece", "Class-wise ECE"), ("final_nll", "NLL"), ("final_brier", "Brier")]
READOUT = [("er_bal", "Bal. acc."), ("er_f1", "Macro-F1"), ("er_tail", "Tail rec."), ("er_worst", "Worst rec."),
           ("er_mcc", "MCC"), ("er_gmean", "G-mean")]
VAL = [("val_final_bal", "Val. bal. acc."), ("val_er_bal", "Val. bal. acc., exact readout")]
SYSTEM = [("comm_mb_per_client", "Comm./client (MB)"), ("time_s", "Time (s)"), ("fe_time_s", "Stage I (s)"),
          ("gen_time_s", "Stage II (s)"), ("gpu_peak_mb", "Peak GPU (MB)"), ("fwd_mflops", "MFLOPs/img"),
          ("train_gflops_round", "Train. GFLOPs/round")]
NOT_PCT = {"final_nll", "final_brier", "comm_mb_per_client", "time_s", "fe_time_s", "gen_time_s", "gpu_peak_mb",
           "class_sep_F", "fwd_mflops", "train_gflops_round"}


REFS = {
    "FedECS-LT": ("This work", "Our implementation (Flower)"),
    "GeFL-F": ("Kang et al., IEEE TMC 2025", "Re-implementation of the authors' Algorithm 3, checked against the "
                                             "earlier experiments"),
    "GeFL-F + LA": ("Ablation: GeFL-F with logit adjustment (Menon et al., ICLR 2021) and decoupled weight decay "
                    "(Loshchilov and Hutter, ICLR 2019)", "Our implementation"),
    "GeFL-F w/o generator": ("Ablation: GeFL-F without its generator (heads trained on real features only)",
                             "Our implementation"),
    "Balanced Softmax": ("Ren et al., NeurIPS 2020 (each client's head trained with its own class prior; equal to "
                         "logit adjustment, Menon et al., ICLR 2021, with tau = 1)", "Our implementation (loss only)"),
    "FedLC": ("Zhang et al., ICML 2022", "Re-implemented from the paper (loss only)"),
    "FedRS": ("Li and Zhan, KDD 2021", "Re-implemented from the paper (loss only)"),
    "GeFL-F w/ Gaussian generator + LA": ("Ablation: GeFL-F with its generator replaced by class Gaussians from the "
                                          "exact class statistics, with logit adjustment", "Our implementation"),
    "FedDF": ("Lin et al., NeurIPS 2020", "Re-implementation (server ensemble distillation on generated features)"),
    "FedDF + LA": ("Ablation: FedDF (Lin et al., NeurIPS 2020) with logit adjustment", "Re-implementation"),
    "Local": ("Lower bound: each client trains its own model alone, without federation", "Our implementation"),
    "FedECS-LT (200 extractor rounds)": ("This work, with the feature extractor trained for 200 rounds (diagnosis run)",
                               "Our implementation (Flower)"),
}


# (runs.jsonl key, name in the caption, shown in %): every per-class metric (metrics.PER_CLASS), heads as trained,
# then the exact readout's per-class recall
PER_CLASS_TABLES = [
    ("final_recall_c", "recall (%)", True), ("final_precision_c", "precision (%)", True),
    ("final_f1_c", "F1 score (%)", True), ("final_ap_c", "average precision, AUC-PR (%)", True),
    ("final_auc_c", "AUC-ROC, class against the rest (%)", True), ("final_mcc_c", "MCC, class against the rest (%)", True),
    ("final_ece_c", "calibration error of p(c|x) (%)", True), ("final_nll_c", "NLL on the images of the class (nats)", False),
    ("final_brier_c", "Brier score on the images of the class", False),
    ("final_top5_c", "top-5 recall (%)", True), ("final_pred_share_c", "share of all predictions (%)", True),
    ("er_recall_c", "recall with the exact class-balanced readout (%)", True)]


def method_ref(m):
    if m in REFS:
        return REFS[m]
    if m.startswith("FedECS-LT"):
        return ("This work (ablation / variant)", "Our implementation (Flower)")
    base = m.split(" (")[0]
    return (CITE.get(base, "see the experiment description"), "see the method's source field")


def roman(n):
    """IEEE table numbers."""
    out = ""
    for v, r in ((1000, "M"), (900, "CM"), (500, "D"), (400, "CD"), (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
                 (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I")):
        while n >= v:
            out, n = out + r, n - v
    return out


def sub_super(text):
    """Captions write subscripts and superscripts as _{...} and ^{...} (escaped text in, reportlab markup out)."""
    return re.sub(r"\^\{([^}]*)\}", r"<super>\1</super>", re.sub(r"_\{([^}]*)\}", r"<sub>\1</sub>", text))


def fmt(mu, sd, key, n=2):
    """mean ± SD of n values; the SD is shown only for two or more (one run has no spread)."""
    if np.isnan(mu):
        return "-"
    spread = n > 1 and not np.isnan(sd)
    if key in NOT_PCT:
        if abs(mu) >= 1000:   # times in seconds, bytes: whole numbers, no exponent
            return f"{mu:,.0f} ± {sd:,.0f}" if spread and sd > 0 else f"{mu:,.0f}"
        return f"{mu:.3g} ± {sd:.2g}" if spread and sd > 0 else f"{mu:.3g}"
    return f"{100 * mu:.1f} ± {100 * sd:.1f}" if spread else f"{100 * mu:.1f}"


def build(res_dir, title, out_pdf, fig_dir):
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_JUSTIFY
    from reportlab.lib.pagesizes import letter
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import inch
    from reportlab.platypus import (Image, KeepTogether, PageBreak, Paragraph, SimpleDocTemplate, Spacer, Table,
                                    TableStyle)
    from xml.sax.saxutils import escape

    figs, na, E = plots.make_all(res_dir, fig_dir)
    ss = getSampleStyleSheet()
    body = ParagraphStyle("b", parent=ss["BodyText"], fontName="Times-Roman", fontSize=9, leading=11, alignment=TA_JUSTIFY)
    small = ParagraphStyle("s", parent=body, fontSize=7.5, leading=9)
    capst = ParagraphStyle("c", parent=body, fontSize=8, leading=10)
    h1 = ParagraphStyle("h1", parent=ss["Heading1"], fontName="Times-Bold", fontSize=14, spaceAfter=6)
    h2 = ParagraphStyle("h2", parent=ss["Heading2"], fontName="Times-Bold", fontSize=11, spaceAfter=4)
    cell = ParagraphStyle("cell", parent=body, fontSize=6.5, leading=7.5, alignment=0)
    P = lambda t, st=body: Paragraph(escape(str(t)), st)  # noqa: E731
    story = []
    used_keys = set()

    def table(rows, header, widths=None, bold_best=None):
        data = [[Paragraph(f"<b>{sub_super(escape(h))}</b>", cell) for h in header]]
        for r in rows:
            data.append([Paragraph(escape(str(x)), cell) for x in r])
        t = Table(data, colWidths=widths, repeatRows=1)
        st = [("LINEABOVE", (0, 0), (-1, 0), 0.8, colors.black), ("LINEBELOW", (0, 0), (-1, 0), 0.5, colors.black),
              ("LINEBELOW", (0, -1), (-1, -1), 0.8, colors.black), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
              ("TOPPADDING", (0, 0), (-1, -1), 1), ("BOTTOMPADDING", (0, 0), (-1, -1), 1)]
        for (i, j) in (bold_best or []):
            st.append(("BACKGROUND", (j, i + 1), (j, i + 1), colors.HexColor("#E6E6E6")))
        t.setStyle(TableStyle(st))
        return t

    tab_no = [0]
    table_errors = []
    caption_problems = []

    def tcap(text):
        tab_no[0] += 1
        n = roman(tab_no[0])
        caption_problems.extend(f"Table {n}: {p_}" for p_ in cap.lint(text, figure=False))
        return Paragraph(f"<b>TABLE {n}</b><br/>{sub_super(escape(text))}", ParagraphStyle("tc", parent=capst,
                                                                                          alignment=1))

    def metric_table(S, keys, caption):
        rows, cells = [], []
        for i, m in enumerate(E.methods):
            r = [m]
            vals = []
            for k, _ in keys:
                mu, sd, n = plots.metric(E, m, k, S)
                used_keys.add(k)
                r.append(fmt(mu, sd, k, n))
                vals.append(mu)
            if all(np.isnan(v) for v in vals):
                continue
            rows.append(r)
            cells.append(vals)
        if not rows:
            return []
        best = []
        A = np.array(cells, dtype=float)
        for j, (k, _) in enumerate(keys):
            col = A[:, j]
            if np.all(np.isnan(col)):
                continue
            lower_better = k in ("final_ece", "final_cw_ece", "final_nll", "final_brier", "final_rec_std",
                                 "comm_mb_per_client", "time_s", "fe_time_s", "gen_time_s", "gpu_peak_mb",
                                 "fwd_mflops", "train_gflops_round", "final_own_mix_sd")
            b = np.nanmin(col) if lower_better else np.nanmax(col)
            best += [(i, j + 1) for i in np.where(np.isclose(col, b, rtol=0, atol=1e-9))[0]]   # ties: all shaded
        w = [1.55 * inch] + [(7.0 - 1.55) * inch / len(keys)] * len(keys)
        return [tcap(caption), Spacer(1, 3), table(rows, ["Method"] + [n for _, n in keys], w, best), Spacer(1, 10)]

    def paired_table(S, st):
        """FedECS-LT minus each method in balanced accuracy, paired over the common seeds: mean, 95% t interval,
        paired t-test p and Holm-adjusted p over the methods."""
        from scipy import stats
        import representation as rp
        ours = {r["seed"]: r for r in E.of(plots.OURS, S)}
        rows_ = []
        for m in E.methods:
            if m == plots.OURS:
                continue
            other = {r["seed"]: r for r in E.of(m, S)}
            d = [100 * (ours[x]["final_bal"] - other[x]["final_bal"]) for x in sorted(set(ours) & set(other))
                 if ours[x].get("final_bal") is not None and other[x].get("final_bal") is not None]
            if len(d) >= 2:
                sd = float(np.std(d, ddof=1))
                half = float(stats.t.ppf(0.975, len(d) - 1) * sd / np.sqrt(len(d)))
                pv = float(stats.ttest_1samp(d, 0.0).pvalue) if sd > 0 else (0.0 if np.mean(d) != 0 else 1.0)
                rows_.append([m, len(d), float(np.mean(d)), half, pv])
        if not ours:
            return []
        if not rows_:
            return [P("Paired tests: FedECS-LT and the other methods share fewer than two seeds in this experiment, "
                      "so no paired test is computed (the merged experiment computes them over seeds 3-5).", small),
                    Spacer(1, 8)]
        adj = rp.holm([r[4] for r in rows_])
        body = [[r[0], r[1], f"{r[2]:+.1f}", f"[{r[2] - r[3]:+.1f}, {r[2] + r[3]:+.1f}]", f"{r[4]:.3g}", f"{a:.3g}"]
                for r, a in zip(rows_, adj)]
        return [tcap("Balanced accuracy of FedECS-LT minus each method (points, test set, heads as trained), paired "
                     "over the common seeds: mean, 95% t confidence interval, two-sided paired t-test p-value and "
                     "Holm-adjusted p-value over the methods. " + st), Spacer(1, 3),
                table(body, ["Method", "Seeds", "Difference", "95% CI", "p", "p (Holm)"],
                      [1.9 * inch, 0.6 * inch, 1.0 * inch, 1.4 * inch, 1.0 * inch, 1.1 * inch]), Spacer(1, 10)]

    def rounds_table(S, st):
        """Rounds each method needs to reach a target balanced accuracy: the final balanced accuracy of the
        reference (GeFL-F, or FedAvg in the one-shared-model setting)."""
        shared = DATASETS.get(S[0], {}).get("track") == 2
        ref = "FedAvg" if shared else plots.REF
        tgt = plots.metric(E, ref, "final_bal", S)[0]
        if np.isnan(tgt):
            return []
        body = []
        for m in E.methods:
            first = []
            for r in E.of(m, S):
                series = plots._global_series(E, r) if shared else \
                    [(e["round"], e["post"]) for e in E.log(r, "stage3_round") if e.get("post")]
                if not series:      # no rounds (Local trains once, alone)
                    continue
                # the per-round logs keep 4 decimals: a round reaches the target within that rounding
                hit = [rd for rd, p in series if p.get("bal") is not None and p["bal"] >= tgt - 5e-5]
                first.append(min(hit) if hit else None)
            if not first:
                continue
            got = [x for x in first if x is not None]
            body.append([m, f"{np.mean(got):.0f}" if got else "not reached", f"{len(got)} of {len(first)}"])
        if not body:
            return []
        stage = "communication rounds of the one shared model (FedECS-LT: extractor rounds, then head rounds)" \
            if shared else "head-training rounds (Stage III, or the method's own rounds)"
        return [tcap(f"Rounds to the target balanced accuracy {100 * tgt:.1f}% (the final balanced accuracy of {ref}): "
                     f"first evaluated round, in {stage}, at which a method's test balanced accuracy reaches it; mean "
                     "over the seeds that reach it. " + st), Spacer(1, 3),
                table(body, ["Method", "Rounds", "Seeds reaching it"], [2.4 * inch, 1.6 * inch, 1.6 * inch]),
                Spacer(1, 10)]

    def representation_table(S, st):
        import representation as rp
        gen_mmd = plots._gen_class_stat(E, S, rp.mmd_rbf)
        gen_mis = plots._gen_class_stat(E, S, rp.mean_mismatch)
        body = []
        for m in E.methods:
            col = plots._collapse(E, m, S)
            mmd = [x[2] for x in gen_mmd if x[0] == m]
            mis = [x[2] for x in gen_mis if x[0] == m]
            au = [r["active_units"] / r["latent_dim"] for r in E.of(m, S) if r.get("active_units") is not None
                  and r.get("latent_dim")]

            def mean_of(k):
                v = [x[k] for x in col if x.get(k) is not None]
                return f"{np.mean(v):.3f}" if v else "-"
            def pct_of(k):
                v = [x[k] for x in col if x.get(k) is not None]
                return f"{100 * np.mean(v):.1f}" if v else "-"
            row = [m, pct_of("knn"), pct_of("knn_tail"), mean_of("nc2_equinorm"), mean_of("nc2_equiangle"), mean_of("nc3"),
                   (f"{np.mean([x['minority_angle'] for x in col if x.get('minority_angle') is not None]):.1f}"
                    if any(x.get("minority_angle") is not None for x in col) else "-"),
                   f"{100 * np.mean(au):.0f}" if au else "-",
                   f"{np.nanmean(mmd):.3f}" if mmd and not np.all(np.isnan(mmd)) else "-",
                   f"{np.nanmean(mis):.2f}" if mis and not np.all(np.isnan(mis)) else "-"]
            if any(x != "-" for x in row[1:]):
                body.append(row)
        if not body:
            return []
        return [tcap("Representation and generator: leave-one-out 10-nearest-neighbour accuracy of the test embeddings "
                     "(%, all classes and the tail classes), NC2 equinorm and equiangularity and NC3 self-duality of "
                     "the test "
                     "embeddings and last layers (lower = closer to neural collapse), mean angle between the tail "
                     "classes' classifier vectors (degrees; small = minority collapse), active latent units of the "
                     "generator (%), and the mean over classes of the squared MMD and of the mean mismatch (in "
                     "within-class standard deviations) between generated and real validation features; mean over "
                     "architectures and seeds. " + st), Spacer(1, 3),
                table(body, ["Method", "10-NN acc.", "10-NN tail", "NC2 norm", "NC2 angle", "NC3", "Tail angle",
                             "Active units", "MMD^{2}", "Mean mism."], [1.55 * inch] + [(7.0 - 1.55) * inch / 9] * 9),
                Spacer(1, 10)]

    # ---------------------------------------------------------------- cover
    done = len(E.runs)
    fails = []
    fp = os.path.join(res_dir, "failures.jsonl")
    if os.path.exists(fp):
        fails = [json.loads(line) for line in open(fp) if line.strip()]
    done_ids = {r["run_id"] for r in E.runs}
    lost = sorted({f["run_id"] for f in fails} - done_ids)        # failed and not recovered by the retry
    recovered = sorted({f["run_id"] for f in fails} & done_ids)
    story += [Paragraph(escape(title), h1),
              P(f"Report generated {datetime.datetime.utcnow():%Y-%m-%d %H:%M} UTC. {done} runs completed"
                + (f" ({len(recovered)} of them on a second attempt)" if recovered else "")
                + f", {len(lost)} failed. Framework: Flower simulation (one server, one client process per simulation); "
                  "every client trains on its own data only."
                + (" The long-tail baselines of the one-shared-model setting run in their authors' own code "
                   "(LongTailLib, FedLTLib benchmark) on the same client split (Table II)."
                   if any("LongTailLib" in str(r.get("framework", "")) for r in E.runs) else "")),
              Spacer(1, 6)]
    aud = None
    if os.path.exists(os.path.join(res_dir, "audit.json")):
        aud = json.load(open(os.path.join(res_dir, "audit.json")))
        story += [P("Logging audit: " + (
            f"every run has every metric for every class on every evaluation ({aud['checked']['runs']} runs, "
            f"{aud['checked']['round_evaluations']} per-round evaluations checked)." if aud["n_problems"] == 0 else
            f"{aud['n_problems']} problems found; see the appendix 'Logging audit'.")), Spacer(1, 6)]
    n_err = sum(1 for _, _, reason, _ in na if str(reason).startswith("error:"))
    if n_err:
        story += [P(f"Figure errors: {n_err} figures could not be drawn; Appendix A gives the error of each."),
                  Spacer(1, 6)]
    cover_at = len(story)   # table errors are known only after the tables: their note is inserted here
    try:
        import torch
        tv, gpu = torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"
    except Exception:          # a merged report (merge_results.py) can be built on a computer without PyTorch
        tv, gpu = "not installed", "CPU only"
    try:
        import flwr
        fv = flwr.__version__
    except Exception:
        fv = "unknown"
    env = [["Python", platform.python_version()], ["PyTorch", tv], ["Flower", fv], ["GPU", gpu],
           ["Code version", os.environ.get("FEDECS_CODE_VERSION", "see results/code_version.txt")]]
    merged = os.environ.get("FEDECS_MERGED")   # set by merge_results.py
    env_cap = ("Software and hardware of the computer that joined the experiment's parts and drew this report; the "
               "runs themselves ran on Kaggle GPUs (each part's own report gives its software and GPU)."
               if merged else "Software and hardware.")
    story += [tcap(env_cap), Spacer(1, 3), table(env, ["Item", "Value"], [1.6 * inch, 5.4 * inch]),
              Spacer(1, 8)]
    meth = []
    for m in E.methods:
        r = E.of(m)[0]
        ref, src = method_ref(m)
        meth.append([m, ref, r["method"].get("source", src)])
    story += [tcap("Methods in this experiment, their reference and the origin of their code."), Spacer(1, 3),
              table(meth, ["Method", "Reference", "Code"], [1.9 * inch, 2.3 * inch, 2.8 * inch]), Spacer(1, 8)]
    for S in E.settings:
        story.append(P("Setting: " + setting_text(S, sorted({r['seed'] for r in E.runs if plots.setting_key(r) == S}))))
    acc = os.path.join(res_dir, "acceptance.json")
    v = None
    if os.path.exists(acc):
        try:
            v = json.load(open(acc))
        except ValueError as e:
            story.append(P(f"The acceptance check's output could not be read ({escape(str(e))})."))

    def pct(d):
        return "-" if d.get("new") is None else (f"{100 * d['new']:.1f} / {100 * d['old_seed3']:.1f} "
                                                f"(±{100 * d['tolerance']:.1f})")
    if v is not None:
        rows = []
        for c in v["checks"]:
            if "final_bal" not in c:
                rows.append([c["check"], c["verdict"], "-", "-", "-", "-"])
                continue
            rows.append([c["check"], c["verdict"], "yes" if c["counts_identical"] else "NO",
                         pct(c["final_bal"]), pct(c["er_bal"]),
                         f"{c['F']['new']:.3f} / {c['F']['old_seed3']:.3f}" if "F" in c else "-"])
        story += [Spacer(1, 8), tcap(f"Acceptance of the Flower implementation against the earlier experiments "
                                     f"(seed 3): overall {v['overall']}. Balanced accuracy new / old (tolerance), "
                                     "as trained and with the exact readout; class separation F new / old."),
                  Spacer(1, 3), table(rows, ["Check", "Verdict", "Same splits", "Bal. acc.", "Bal. acc. (ER)", "F"],
                                      [1.6 * inch, 0.6 * inch, 0.7 * inch, 1.4 * inch, 1.4 * inch, 1.3 * inch])]
    story.append(PageBreak())

    # ---------------------------------------------------------------- tables
    story.append(Paragraph("1. Results", h1))
    for S in E.settings:
        st = setting_text(S, sorted({r['seed'] for r in E.runs if plots.setting_key(r) == S}))
        story.append(Paragraph(escape(st), h2))
        # the classes of this setting, in the order every per-class table and figure uses
        r0 = next((r for r in E.runs if plots.setting_key(r) == S and r.get("class_order")), None)
        if r0 is not None:
            C = len(r0["class_order"])
            ntr = np.array(r0.get("client_counts") or [[0] * C]).sum(0)
            nte = r0.get("test_class_counts") or [None] * C   # images of each class in the test set
            names = r0.get("class_names") or [None] * C
            rows = [[str(c), str(o), escape(names[c] or "-"), str(int(ntr[c])), "-" if nte[c] is None else str(int(nte[c]))]
                    for c, o in enumerate(r0["class_order"])]
            story += [tcap("Classes in the order used by every per-class table and figure (0 = most frequent in the "
                           "federation's training data): original label in the dataset's files, name, training images "
                           "held by all clients together, test images. " + st), Spacer(1, 3),
                      table(rows, ["Class", "Original label", "Name", "Training images", "Test images"],
                            [0.6 * inch, 1.0 * inch, 3.4 * inch, 1.0 * inch, 1.0 * inch]), Spacer(1, 10)]
        seeds = sorted({r['seed'] for r in E.runs if plots.setting_key(r) == S})
        one = lambda t: cap.one_seed(t, seeds)  # noqa: E731
        shared = DATASETS.get(S[0], {}).get("track") == 2      # Track 2: one model shared by all clients
        trained = "model as trained" if shared else "heads as trained"
        story += metric_table(S, MAIN, one(f"Main metrics (%, test set, {trained}; mean ± standard deviation over "
                                           "seeds; shaded = best). Bal. acc.: mean of the class recalls; Macro-F1 "
                                           "and Precision: means over the classes (precision 0 for a class that is "
                                           "never predicted); G-mean: geometric mean of the class recalls, each "
                                           "floored at 1%; Tail rec.: mean recall of the rarest 40% of the classes; "
                                           "Worst rec.: lowest class recall; AUC-PR: macro average precision. " + st))
        story += metric_table(S, READOUT, ("The same model" if shared else "The same heads") + " with the exact "
                              "class-balanced readout (last layer recomputed from secure sums of every client's "
                              "embeddings). " + st)
        story += metric_table(S, SECOND, f"Further metrics (%, test set, {trained}). Top-5: share of the test images "
                                         "whose class is among the five most probable; AUC-ROC: one class against the "
                                         "rest, macro average; Head rec. and Med. rec.: mean recall of the most "
                                         "frequent 30% and of the next 30% of the classes; Recall SD: standard "
                                         "deviation of the class recalls. " + st)
        if not shared:   # one shared model: no architectures, every client evaluates the same model
            story += metric_table(S, CLIENTS, f"Fairness across clients (%, test set, {trained}): balanced accuracy of "
                                  "the weakest head architecture, and the accuracy re-weighted to each architecture's "
                                  "own clients' class mix (own-mix accuracy; mean, standard deviation and minimum over "
                                  "the architectures, or over the clients' own models). " + st)
        story += metric_table(S, CALIB, "Calibration (ECE and class-wise ECE in %, NLL in nats, Brier score). ECE: 15 "
                              "equal-width bins of the highest probability; class-wise ECE: the same for p(c|x) of "
                              "every class, averaged (Kull et al., NeurIPS 2019); NLL with probabilities floored at "
                              "10^{-6}; Brier score summed over the classes (0 to 2)."
                              + proto_note([m for m in E.methods if E.of(m, S)]) + " " + st)
        story += metric_table(S, VAL, "Balanced accuracy on the validation set (images no client holds), used to select "
                                      "settings without the test set. " + st)
        story += metric_table(S, SYSTEM, "Cost: communication per client over the whole training, wall-clock time, "
                                         "peak GPU memory on one GPU, forward FLOPs per image (extractor and the mean "
                                         "head, or the whole model) and training FLOPs of one round over all clients "
                                         "(3 x forward; the head stage for the methods on the frozen extractor, every "
                                         "round of the one shared model otherwise). " + st)
        for fn in (paired_table, rounds_table, representation_table):
            try:
                story += fn(S, st)
            except Exception as e:   # a failing table is reported, never stops the report
                table_errors.append(f"{fn.__name__} ({setting_slug(S)}): {e!r}"[:400])
        # per-seed table
        rows = []
        for m in E.methods:
            for r in sorted(E.of(m, S), key=lambda r: r["seed"]):
                rows.append([m, r["seed"], fmt(r.get("final_bal", np.nan) or np.nan, 0, "final_bal", 1),
                             fmt(r.get("er_bal", np.nan) or np.nan, 0, "er_bal", 1),
                             fmt(r.get("final_tail", np.nan) or np.nan, 0, "final_tail", 1),
                             fmt(r.get("final_f1", np.nan) or np.nan, 0, "final_f1", 1), f"{r.get('time_s', 0):.0f}"])
        if rows:
            story += [tcap("Every run (one row per method and seed): balanced accuracy as trained and with the exact "
                           "readout, tail recall, macro-F1 (%), time (s). " + st), Spacer(1, 3),
                      table(rows, ["Method", "Seed", "Bal.", "Bal. (ER)", "Tail", "F1", "Time"],
                            [1.9 * inch, 0.5 * inch] + [0.9 * inch] * 5), Spacer(1, 10)]
        # per-class tables: every metric that has a per-class form, for every class (blocks of 20 classes)
        for key, name, pct in PER_CLASS_TABLES:
            recs = []
            for m in E.methods:
                v = [r.get(key) for r in E.of(m, S) if r.get(key)]
                if v:
                    mu = np.nanmean(np.array(v, dtype=float), 0) * (100 if pct else 1)
                    recs.append((m, mu))
            if not recs:
                continue
            if key == "final_pred_share_c":   # reference row: each class's true share of the test set
                cm = next((np.array(r["final_confusion"], dtype=float) for r in E.of(recs[0][0], S)
                           if r.get("final_confusion")), None)
                if cm is not None:
                    recs.append(("Test set (true share)", cm.sum(1) / cm.sum() * 100))
            C = len(recs[0][1])
            for c0 in range(0, C, 20):
                cols = list(range(c0, min(C, c0 + 20)))
                def cell_(x):   # no "-0" for a value that rounds to zero
                    return "-" if np.isnan(x) else (f"{int(round(x))}" if pct else f"{x:.2f}".replace("-0.00", "0.00"))
                rows = [[m] + [cell_(mu[c]) for c in cols]
                        for m, mu in recs]
                part = "" if C <= 20 else f" Classes {cols[0]}-{cols[-1]} of {C}."
                note = proto_note([m for m, _ in recs]) if key in ("final_ece_c", "final_nll_c", "final_brier_c") else ""
                story += [tcap(one(f"Per-class {name} (test set, mean over seeds); class 0 is the most frequent."
                                   f"{part}{note} " + st)), Spacer(1, 3),
                          table(rows, ["Method"] + [str(c) for c in cols],
                                [1.55 * inch] + [(7.0 - 1.55) * inch / len(cols)] * len(cols)), Spacer(1, 10)]
    story.append(PageBreak())

    # ---------------------------------------------------------------- figures
    story.append(Paragraph("2. Figures", h1))
    story.append(P("Every figure is drawn at its printed IEEE size (one column 3.5 in, two columns 7.16 in, Times "
                   "8 pt); shown here as previews. The vector PDF and the plotted data (CSV) of each are in "
                   "results.zip, folder figures/.", small))
    included = set()
    for i, (num, group, pdf, caption) in enumerate(figs, 1):
        png = pdf[:-4] + ".png"
        from PIL import Image as PILImage
        with PILImage.open(png) as im:
            w, h = im.size
        width_in = min(7.0, w / 200.0)
        img = Image(png, width=width_in * inch, height=width_in * inch * h / w)
        story.append(KeepTogether([img, Spacer(1, 3), Paragraph(f"<b>Fig. {i}.</b> {sub_super(escape(caption))}", capst),
                                   Spacer(1, 12)]))
        included.add(os.path.abspath(pdf))
        caption_problems.extend(f"Fig. {i} (catalogue {num}): {p_}" for p_ in cap.lint(caption))
    story.append(PageBreak())

    # ---------------------------------------------------------------- appendix
    story.append(Paragraph("Appendix A. Figures not applicable to this experiment", h1))
    rows = [[n, g, reason, setting_slug(S)] for n, g, reason, S in na]
    caption_problems.extend(f"not-applicable reason of figure {n}: {p_}" for n, g, reason, S in na
                            if not str(reason).startswith("error:") for p_ in cap.lint(str(reason), figure=False))
    if rows:
        story += [table(rows, ["Catalogue no.", "Group", "Reason", "Setting"],
                        [0.9 * inch, 0.6 * inch, 4.0 * inch, 1.5 * inch]), Spacer(1, 8)]
    story.append(Paragraph("Appendix B. Figure sources", h1))
    rows = [[i, n, os.path.relpath(p, os.path.dirname(fig_dir))] for i, (n, g, p, c) in enumerate(figs, 1)]
    story += [table(rows, ["Fig.", "Catalogue no.", "File"], [0.5 * inch, 0.9 * inch, 5.6 * inch]), Spacer(1, 8)]
    story.append(Paragraph("Appendix C. All logged values of every run", h1))
    for r in E.runs:
        scal = sorted((k, v) for k, v in r.items() if isinstance(v, (int, float)) and not isinstance(v, bool))
        used_keys.update(k for k, _ in scal)
        rows = [[k, f"{v:.6g}" if isinstance(v, float) else str(v)] for k, v in scal]
        vec = sorted(k for k, v in r.items() if isinstance(v, (list, dict)))
        story += [Paragraph(escape(r["run_id"]), h2), table(rows, ["Quantity", "Value"], [3.0 * inch, 4.0 * inch]),
                  P("Vector-valued fields (in runs.jsonl): " + ", ".join(vec), small), Spacer(1, 6)]
    if fails:
        story.append(Paragraph("Appendix D. Failed runs", h1))
        story += [table([[f["run_id"] + (" (completed on retry)" if f["run_id"] in done_ids else ""),
                          f["error"][:600]] for f in fails], ["Run", "Error"], [2.0 * inch, 5.0 * inch])]
    if aud is not None and aud["n_problems"]:
        story.append(Paragraph("Appendix E. Logging audit", h1))
        story += [P(f"{aud['n_problems']} problems (at most 200 listed)."),
                  table([[escape(p_)] for p_ in aud["problems"]], ["Problem"], [7.0 * inch])]

    # ---------------------------------------------------------------- completeness check
    all_pdfs = {os.path.abspath(p) for p in glob.glob(os.path.join(fig_dir, "**", "*.pdf"), recursive=True)}
    missing = sorted(all_pdfs - included)
    if missing:
        raise RuntimeError(f"report incomplete: {len(missing)} figure files not in the PDF: {missing[:5]}")

    def page(canvas, doc):
        canvas.saveState()
        canvas.setFont("Times-Roman", 8)
        canvas.drawRightString(letter[0] - 0.75 * inch, 0.5 * inch, f"{title} - page {doc.page}")
        canvas.restoreState()
    layout = [f"{n}: {p_}" for n, p_ in ps.LAYOUT_PROBLEMS]
    if layout or caption_problems:   # presentation checks: on the cover, and listed in an appendix
        story.insert(cover_at, P(f"Presentation checks: {len(layout)} figure-layout and {len(caption_problems)} "
                                 "caption problems; Appendix G lists them."))
        story.append(Paragraph("Appendix G. Presentation checks", h1))
        story += [table([[escape(t)] for t in (layout + caption_problems)[:300]], ["Problem"], [7.0 * inch])]
    if table_errors:   # on the cover, and listed in an appendix
        story.insert(cover_at, P(f"Table errors: {len(table_errors)} tables could not be computed; Appendix F lists "
                                 "them."))
        story.append(Paragraph("Appendix F. Table errors", h1))
        story += [table([[escape(t)] for t in table_errors], ["Error"], [7.0 * inch])]
    doc = SimpleDocTemplate(out_pdf, pagesize=letter, leftMargin=0.75 * inch, rightMargin=0.75 * inch,
                            topMargin=0.7 * inch, bottomMargin=0.8 * inch, title=title)
    doc.build(story, onFirstPage=page, onLaterPages=page)
    return dict(figures=len(figs), not_applicable=len(na), runs=len(E.runs), failures=len(lost),
                figure_errors=sum(1 for _, _, reason, _ in na if str(reason).startswith("error:")),
                table_errors=len(table_errors), layout_problems=len(ps.LAYOUT_PROBLEMS),
                caption_problems=len(caption_problems))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", required=True)
    ap.add_argument("--title", required=True)
    ap.add_argument("--out", required=True)
    a = ap.parse_args()
    info = build(a.results, a.title, a.out, os.path.join(a.results, "figures"))
    print(json.dumps(info))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/fedecs_lt/representation.py
"""Representation, generator-quality and testing statistics computed from a run's saved outputs (test embeddings,
last layers, real and generated validation features); used by plots.py and report.py.

Neural collapse (Papyan, Han and Donoho, PNAS 2020): with class means m_c of the embeddings, their mean m_G and the
centred means M_c = m_c - m_G,
  NC2 equinorm      = std_c ||M_c|| / mean_c ||M_c||                         (0 at collapse)
  NC2 equiangularity = std over class pairs of cos(M_a, M_b)                 (0 at collapse)
  NC2 maximal angle  = mean over pairs of |cos(M_a, M_b) + 1 / (C - 1)|      (0 for a simplex ETF)
  NC3 self-duality   = || W / ||W||_F - M / ||M||_F ||_F^2                   (0 when the classifier aligns with M)
Minority collapse (Fang et al., PNAS 2021): the mean pairwise angle between the classifier vectors of the tail
classes (small = the rare classes' classifiers collapse onto each other).
Linear CKA (Kornblith et al., ICML 2019) between two architectures' embeddings of the same images.
MMD: unbiased squared maximum mean discrepancy with a Gaussian kernel, bandwidth = median pairwise distance.
Holm (1979) step-down adjustment of p-values.
"""
import numpy as np


def class_means(H, y, C):
    present = [c for c in range(C) if (y == c).any()]
    return np.stack([H[y == c].mean(0) for c in present]), present


def nc1(H, y, C, tol=1e-8):
    """NC1 = tr(S_W S_B^+) / C, with S_B^+ the pseudo-inverse over the between-class directions whose eigenvalue is
    above tol x the largest (S_B has rank at most C - 1; inverting its numerically zero eigenvalues, as a plain
    pinv of a covariance in many dimensions does, gives meaningless or even negative values). Always >= 0."""
    H = np.asarray(H, dtype=np.float64)
    mus, present = class_means(H, y, C)
    mg = H.mean(0)
    B = (mus - mg) / np.sqrt(len(mus))             # S_B = B^T B
    U, sv, _ = np.linalg.svd(B.T, full_matrices=False)
    lam = sv ** 2
    keep = lam > tol * lam.max() if len(lam) and lam.max() > 0 else np.zeros(len(lam), bool)
    if not keep.any():
        return None
    U, lam = U[:, keep], lam[keep]
    tot = 0.0
    for c in present:
        X = H[y == c] - mus[present.index(c)]
        if len(X) > 1:
            P = X @ U                                 # within-class scatter along the between-class directions
            tot += ((P ** 2).sum(0) / len(X) / lam).sum() * (len(X) / len(H))
    return float(tot / len(present))


def nc2(H, y, C):
    mus, _ = class_means(H.astype(np.float64), y, C)
    M = mus - mus.mean(0)
    norms = np.linalg.norm(M, axis=1)
    U = M / np.maximum(norms[:, None], 1e-12)
    cos = U @ U.T
    iu = np.triu_indices(len(M), 1)
    k = len(M)
    return dict(nc2_equinorm=float(norms.std() / max(norms.mean(), 1e-12)),
                nc2_equiangle=float(cos[iu].std()),
                nc2_maxangle=float(np.abs(cos[iu] + 1.0 / (k - 1)).mean()) if k > 1 else 0.0)


def nc3(H, y, C, W):
    mus, present = class_means(H.astype(np.float64), y, C)
    M = mus - mus.mean(0)
    Wp = np.asarray(W, dtype=np.float64)[present]
    if Wp.shape[1] != M.shape[1]:
        return None
    return float(np.linalg.norm(Wp / np.linalg.norm(Wp) - M / np.linalg.norm(M)) ** 2)


def minority_angle(W, classes):
    """Mean pairwise angle (degrees) between the classifier vectors of the given classes."""
    V = np.asarray(W, dtype=np.float64)[list(classes)]
    if len(V) < 2:
        return None
    U = V / np.maximum(np.linalg.norm(V, axis=1, keepdims=True), 1e-12)
    cos = np.clip(U @ U.T, -1.0, 1.0)
    iu = np.triu_indices(len(V), 1)
    return float(np.degrees(np.arccos(cos[iu])).mean())


def linear_cka(X, Y):
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    X = X - X.mean(0)
    Y = Y - Y.mean(0)
    xy = np.linalg.norm(X.T @ Y) ** 2
    xx = np.linalg.norm(X.T @ X)
    yy = np.linalg.norm(Y.T @ Y)
    return float(xy / max(xx * yy, 1e-300))


def mmd_rbf(X, Y, bandwidth=None):
    """Unbiased MMD^2 between samples X and Y (rows), Gaussian kernel exp(-d^2 / (2 b^2))."""
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    Z = np.concatenate([X, Y])
    sq = (Z * Z).sum(1)
    D2 = np.maximum(sq[:, None] + sq[None, :] - 2 * Z @ Z.T, 0.0)
    if bandwidth is None:
        iu = np.triu_indices(len(Z), 1)
        bandwidth = np.sqrt(np.median(D2[iu])) if len(iu[0]) else 1.0
    K = np.exp(-D2 / (2 * max(bandwidth, 1e-12) ** 2))
    n, m = len(X), len(Y)
    if n < 2 or m < 2:
        return None
    kxx = (K[:n, :n].sum() - np.trace(K[:n, :n])) / (n * (n - 1))
    kyy = (K[n:, n:].sum() - np.trace(K[n:, n:])) / (m * (m - 1))
    kxy = K[:n, n:].mean()
    return float(kxx + kyy - 2 * kxy)


def mean_mismatch(Xg, Xr):
    """Distance between the generated and the real class mean, in units of the real within-class spread
    (square root of the mean per-dimension variance)."""
    Xg = np.asarray(Xg, dtype=np.float64)
    Xr = np.asarray(Xr, dtype=np.float64)
    spread = np.sqrt(Xr.var(0).mean()) if len(Xr) > 1 else np.nan
    return float(np.linalg.norm(Xg.mean(0) - Xr.mean(0)) / np.sqrt(Xr.shape[1]) / max(spread, 1e-12))


def holm(p):
    """Holm-adjusted p-values (same order as p)."""
    p = np.asarray(p, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    run = 0.0
    for i, j in enumerate(order):
        run = max(run, (len(p) - i) * p[j])
        adj[j] = min(1.0, run)
    return adj.tolist()


def knn_correct(H, y, k=10):
    """Leave-one-out k-nearest-neighbour classification in the embedding space (Euclidean, majority vote, ties to the
    nearest neighbour's class): a quantitative companion of a t-SNE map, computed on the embeddings themselves, not
    on the map (t-SNE can make unclustered data look clustered). Returns one bool per image."""
    H = np.asarray(H, dtype=np.float64)
    y = np.asarray(y)
    sq = (H * H).sum(1)
    D = sq[:, None] + sq[None, :] - 2 * H @ H.T
    np.fill_diagonal(D, np.inf)
    nn = np.argsort(D, 1)[:, :k]
    lab = y[nn]
    out = np.empty(len(y), bool)
    for i in range(len(y)):
        vals, cnt = np.unique(lab[i], return_counts=True)
        best = vals[cnt == cnt.max()]
        out[i] = (best[0] if len(best) == 1 else lab[i][np.isin(lab[i], best)][0]) == y[i]
    return out


In [ ]:
%%writefile /kaggle/working/fedecs_lt/runner.py
"""Runs experiments: one Flower simulation per run, two runs per GPU in parallel worker processes, resumable
(runs already in runs.jsonl are skipped), stops starting new runs at the deadline.

    python runner.py --exp NAME --runs runs.json --out /kaggle/working/results --gpus 2 --per_gpu 2
"""
import argparse
import json
import os
import shutil
import subprocess
import sys
import threading
import time

HERE = os.path.dirname(os.path.abspath(__file__))

# Methods of the GeFL-F pipeline family (server.run_pipeline). Baselines with their own protocol are in baselines/.
PIPELINE = {
    "FedECS-LT": dict(pipeline="gefl", stats=True, secure=True, readout=True,
                      gen=dict(type="gate", opt="adam", wd=1e-3, beta=0.999),
                      head=dict(ts=10, tr=5, loss="la", tau=1.0, mc=True, csl=0.5)),
    "GeFL-F": dict(pipeline="gefl", gen=dict(type="cvae", opt="adam", wd=1e-3, agg="flat"),
                   head=dict(ts=1, tr=5, loss="ce")),
    "GeFL-F + LA": dict(pipeline="gefl", gen=dict(type="cvae", opt="adamw", wd=1e-3, agg="flat"),
                                head=dict(ts=1, tr=5, loss="la", tau=1.0)),
    "GeFL-F w/o generator": dict(pipeline="gefl", gen=None, head=dict(ts=0, tr=5, loss="ce")),
    "Balanced Softmax": dict(pipeline="gefl", gen=None, head=dict(ts=0, tr=5, loss="la", tau=1.0)),
    "FedLC": dict(pipeline="gefl", gen=None, head=dict(ts=0, tr=5, loss="lc", tau=1.0)),
    "FedRS": dict(pipeline="gefl", gen=None, head=dict(ts=0, tr=5, loss="rs")),
    "GeFL-F w/ Gaussian generator + LA": dict(pipeline="gefl", stats=False, gen=dict(type="gauss"),
                                    head=dict(ts=1, tr=5, loss="la", tau=1.0)),
    "FedDF": dict(pipeline="gefl", gen=dict(type="cvae", opt="adam", wd=1e-3, agg="flat"),
                  head=dict(ts=1, tr=5, loss="ce", sed=20, sed_beta=1.0, sed_lr=0.05, sed_bs=128)),
    "FedDF + LA": dict(pipeline="gefl", gen=dict(type="cvae", opt="adam", wd=1e-3, agg="flat"),
                       head=dict(ts=1, tr=5, loss="la", tau=1.0, sed=20, sed_beta=1.0, sed_lr=0.05, sed_bs=128)),
    "Local": dict(pipeline="local", head=dict(ts=0, tr=5, loss="ce")),
}
# Track 2 diagnosis (user, 24 Oct): FedECS-LT with its extractor trained for all 200 rounds of the baselines (the run
# sets rounds T_FE = 200; generator and heads as FedECS-LT, so 300 rounds in total, reported as such)
PIPELINE["FedECS-LT (extractor 200 rounds)"] = dict(PIPELINE["FedECS-LT"])
# Component ablation (paper, Table "Components of FedECS-LT"; user, 27 Oct 2026): one component changed at a time,
# everything else as FedECS-LT. The two generator variants replace the gate: class rows averaged by CCWA, or class
# rows averaged flat (GeFL-F's generator), both with the exact statistics, moment correction and the same heads.
_HEAD = PIPELINE["FedECS-LT"]["head"]
COMPONENTS = {
    "FedECS-LT w/ class rows (CCWA)": dict(gen=dict(type="cvae", opt="adam", wd=1e-3, agg="ccwa", beta=0.999)),
    "FedECS-LT w/ class rows (flat averaging)": dict(gen=dict(type="cvae", opt="adam", wd=1e-3, agg="flat")),
    "FedECS-LT w/o moment correction": dict(head=dict(_HEAD, mc=False)),
    "FedECS-LT w/o consensus soft labels": dict(head=dict(_HEAD, csl=0.0)),
    "FedECS-LT w/o logit adjustment": dict(head=dict(_HEAD, loss="ce")),
    "FedECS-LT w/ one synthetic epoch": dict(head=dict(_HEAD, ts=1)),
}
for _name, _change in COMPONENTS.items():
    PIPELINE[_name] = dict(PIPELINE["FedECS-LT"], **_change)


def method_spec(label):
    if label in PIPELINE:
        return PIPELINE[label]
    from baselines import BASELINES
    return BASELINES[label]


def run_id(r):
    extra = ("|tp" if r.get("tail_perm") else "") + (f"|p{r['pfrac']}" if r.get("pfrac", 1.0) < 1.0 else "")
    return f"{r['label']}|{r['ds']}|IF{r['IF']}|a{r['alpha']}|K{r['K']}|s{r['seed']}" + extra


def run_one(R):
    """One run = one Flower simulation (Ray backend, one persistent client process on this process's GPU); Track-2
    baselines run the authors' LongTailLib code instead (track2.py)."""
    if R["method"].get("pipeline") == "longtaillib":
        sys.path.insert(0, HERE)
        import track2
        return track2.run_for_suite(R)
    import torch
    from flwr.simulation import run_simulation
    sys.path.insert(0, HERE)
    os.environ["PYTHONPATH"] = HERE + os.pathsep + os.environ.get("PYTHONPATH", "")
    os.environ.setdefault("RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO", "0")
    import client
    import server
    server.RUN.clear()
    server.RUN.update(R)
    gpu = 1.0 if torch.cuda.is_available() else 0.0
    backend = {"client_resources": {"num_cpus": 1, "num_gpus": gpu},
               "init_args": {"num_cpus": 1, "num_gpus": int(gpu), "include_dashboard": False,
                             "object_store_memory": int(R.get("ray_object_store_gb", 2) * 2 ** 30),
                             "logging_level": "ERROR", "log_to_driver": True}}
    run_simulation(server_app=server.app, client_app=client.app, num_supernodes=R["K"], backend_config=backend)
    if "result" not in server.RUN:
        raise RuntimeError(f"run {R['run_id']} produced no result")
    return server.RUN["result"]


def done_ids(out_dir):
    p = os.path.join(out_dir, "runs.jsonl")
    if not os.path.exists(p):
        return set()
    out = set()
    for line in open(p):
        try:
            out.add(json.loads(line)["run_id"])
        except Exception:
            pass
    return out


def worker_main(args):
    """One worker: takes the next run of the list that no other worker has claimed (a shared queue, so long runs do
    not leave other workers idle) and runs it; runs are ordered by dataset and seed, so runs sharing cached stages
    are close together."""
    runs = json.load(open(args.runs))
    claims = os.path.join(os.path.dirname(os.path.abspath(args.runs)), "claims")
    os.makedirs(claims, exist_ok=True)
    deadline = float(os.environ.get("FEDECS_DEADLINE", "inf"))
    # pass 0: the queue; pass 1: every run that failed (out of memory while two heavy runs overlapped, a download
    # error) is tried once more after the queue is empty
    for attempt in (0, 1):
        for r in runs:
            if time.time() > deadline:
                print("deadline reached: no new run started", flush=True)
                return
            R = dict(r)
            R.setdefault("method", method_spec(R["label"]))
            R["run_id"] = run_id(R)
            if R["run_id"] in done_ids(R["out_dir"]):
                if attempt == 0:
                    print("already done:", R["run_id"], flush=True)
                continue
            if attempt == 1 and R["run_id"] not in failed_ids(R["out_dir"]):
                continue
            safe = "".join(ch if ch.isalnum() or ch in "-_.=" else "_" for ch in R["run_id"])
            try:   # atomic claim: exactly one worker gets each run (and each retry)
                os.close(os.open(os.path.join(claims, ("retry_" if attempt else "") + safe),
                                 os.O_CREAT | os.O_EXCL | os.O_WRONLY))
            except FileExistsError:
                continue
            execute(R, attempt)


def failed_ids(out_dir):
    path = os.path.join(out_dir, "failures.jsonl")
    if not os.path.exists(path):
        return set()
    out = set()
    for line in open(path):
        try:
            out.add(json.loads(line)["run_id"])
        except Exception:
            pass
    return out


def needs_whole_gpu(R):
    """FedECS-LT with one shared model keeps 32,768-dimensional features of every image on the GPU: about 13 GB of a
    T4's 14.6 GB (Kaggle, 27 Oct 2026), so it ran out of memory next to any other run."""
    from settings import DATASETS
    return R["label"].startswith("FedECS-LT") and DATASETS.get(R["ds"], {}).get("track") == 2


def _alive(pid):
    """A waiting run's process still exists (a process killed outright leaves its .want file behind, which would
    otherwise block every shared run on that GPU until the deadline)."""
    try:
        os.kill(int(pid), 0)
        return True
    except PermissionError:
        return True
    except (ValueError, ProcessLookupError):
        return False


class GpuShare:
    """The GPU of this worker (CUDA_VISIBLE_DEVICES): runs that need all of it take it exclusively, the others share
    it. A run waiting for exclusive use keeps new shared runs from starting there, so it is not starved. Only the
    scheduling changes, never a run's computation."""

    def __init__(self, work_dir, alone):
        dev = os.environ.get("CUDA_VISIBLE_DEVICES", "cpu")
        self.path = os.path.join(work_dir, f"gpu_{dev}.lock")
        self.want = f"{self.path}.want.{os.getpid()}"   # one per waiting run
        self.alone = alone

    def __enter__(self):
        import fcntl
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        self.f = open(self.path, "a")
        if self.alone:
            open(self.want, "a").close()
            print("waiting for the whole GPU", flush=True)
            fcntl.flock(self.f, fcntl.LOCK_EX)
        else:
            import glob
            while True:
                waiting = [w for w in glob.glob(self.path + ".want.*") if _alive(w.rsplit(".", 1)[-1])]
                if not waiting:
                    break
                time.sleep(5)
            fcntl.flock(self.f, fcntl.LOCK_SH)
        return self

    def __exit__(self, *exc):
        import fcntl
        if self.alone:
            try:
                os.remove(self.want)
            except FileNotFoundError:
                pass
        fcntl.flock(self.f, fcntl.LOCK_UN)
        self.f.close()


def execute(R, attempt=0):
    t0 = time.time()
    print(("retry " if attempt else "") + "start", R["run_id"], flush=True)
    try:
        with GpuShare(os.path.dirname(os.path.abspath(R["out_dir"])), needs_whole_gpu(R)):
            res = run_one(R)
        print(f"done {R['run_id']}  bal={res.get('final_bal', float('nan')):.4f}  "
              f"er_bal={(res.get('er_bal') or float('nan')):.4f}  {time.time() - t0:.0f}s", flush=True)
    except Exception as e:  # record the failure, release this process's stage locks, continue with the next run
        import glob
        import traceback
        traceback.print_exc()
        os.makedirs(R["out_dir"], exist_ok=True)
        with open(os.path.join(R["out_dir"], "failures.jsonl"), "a") as f:
            f.write(json.dumps(dict(run_id=R["run_id"], attempt=attempt + 1, error=repr(e)[:4000],
                                    time_s=time.time() - t0)) + "\n")
        for lk in glob.glob(os.path.join(R["cache_dir"], "*", "*.lock")):
            try:
                if open(lk).read().strip() == str(os.getpid()):
                    os.remove(lk)
            except OSError:
                pass
    import gc   # the server side of every run lives in this worker process: free its memory before the next run
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    if os.environ.get("FEDECS_PACK_OUT"):   # protect finished work against a timeout
        try:
            import pack
            pack.quick(os.path.dirname(R["out_dir"]), os.environ["FEDECS_NAME"], os.environ["FEDECS_PACK_OUT"])
        except Exception as e:
            print("pack failed:", repr(e), flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", required=True, help="JSON list of runs")
    ap.add_argument("--gpus", type=int, default=0)
    ap.add_argument("--per_gpu", type=int, default=2)
    ap.add_argument("--worker", type=int, default=None)
    ap.add_argument("--n_workers", type=int, default=1)
    ap.add_argument("--log_dir", default=None)
    args = ap.parse_args()
    if args.worker is not None:
        return worker_main(args)
    n = max(1, args.gpus * args.per_gpu) if args.gpus else max(1, args.per_gpu)
    # a hand-edited run cell ('--gpus 1 --per_gpu 1' in NB10_main_cifar100_part1of2, Kaggle, 26 Oct 2026) ran every
    # run one after another on one of the two GPUs: say how many workers start, and warn when GPUs stay unused
    try:
        import torch
        found = torch.cuda.device_count()
    except Exception:
        found = 0
    print(f"runner: {n} worker(s) on {args.gpus or 'no'} GPU(s), {args.per_gpu} run(s) per GPU; GPUs found: {found}",
          flush=True)
    if found > args.gpus:
        print(f"WARNING: {found - args.gpus} GPU(s) will stay idle: the run cell asks for --gpus {args.gpus}. "
              "Use the notebook as downloaded from the repository (never edit its run cell).", flush=True)
    # a new start of the queue: claims of an earlier, interrupted start are void (finished runs are skipped anyway)
    shutil.rmtree(os.path.join(os.path.dirname(os.path.abspath(args.runs)), "claims"), ignore_errors=True)
    procs, threads = [], []
    for w in range(n):
        env = dict(os.environ)
        if args.gpus:
            env["CUDA_VISIBLE_DEVICES"] = str(w % args.gpus)
        # less fragmentation of GPU memory when two runs share a GPU (allocation only; the computation is unchanged)
        env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
        cmd = [sys.executable, os.path.abspath(__file__), "--runs", args.runs, "--worker", str(w), "--n_workers", str(n)]
        log_dir = args.log_dir or os.path.dirname(args.runs)
        os.makedirs(log_dir, exist_ok=True)
        log = open(os.path.join(log_dir, f"worker{w}.log"), "a")
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=HERE)

        def stream(p=p, w=w, log=log):
            for line in p.stdout:
                log.write(line)
                log.flush()
                if line.startswith(("start", "retry", "done", "already", "deadline", "Traceback", "RuntimeError")):
                    print(f"[w{w}] {line}", end="", flush=True)
        t = threading.Thread(target=stream, daemon=True)
        t.start()
        procs.append(p)
        threads.append(t)
    for p in procs:
        p.wait()
    for t in threads:
        t.join(timeout=5)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/fedecs_lt/secure_sum.py
"""Secure summation with Flower's SecAgg+ (Bell et al., CCS 2020), and the plain sum used by baseline protocols.

The server learns only the sum over the participating clients. Flower's SecAgg+ workflow returns a weighted mean of
quantised vectors in [-clip, clip]; every client therefore sends its contribution divided by a public bound (with
num_examples = 1) and the server multiplies the mean by (number of clients x bound). Values outside the bound are
clipped and counted (reported in the run log). Quantisation uses 30 bits (settings.SECAGG); the round-off is below
1e-7 of the bound per client.
"""
import numpy as np
from flwr.app import ArrayRecord, ConfigRecord, Message
from flwr.common import Code, FitIns, FitRes, Status, ndarrays_to_parameters, parameters_to_ndarrays
from flwr.compat.common import recorddict_compat as compat
from flwr.server import ServerConfig
from flwr.server.compat import LegacyContext
from flwr.server.compat.grid_client_proxy import GridClientProxy
from flwr.server.strategy import Strategy
from flwr.server.workflow import SecAggPlusWorkflow
from flwr.server.workflow.constant import MAIN_CONFIGS_RECORD, MAIN_PARAMS_RECORD, Key as WK

from fl_common import jdump
from settings import SECAGG


class _SumStrategy(Strategy):
    """Legacy-strategy shim for SecAggPlusWorkflow: sends one config to every node, keeps the unmasked mean."""

    def __init__(self, cfg):
        self.cfg, self.result = cfg, None

    def initialize_parameters(self, client_manager):
        return None

    def configure_fit(self, server_round, parameters, client_manager):
        return [(c, FitIns(ndarrays_to_parameters([np.zeros(1)]), {"j": jdump(self.cfg)}))
                for c in client_manager.all().values()]

    def aggregate_fit(self, server_round, results, failures):
        if failures:
            raise RuntimeError(f"secure sum: {len(failures)} client failures: {failures[:1]}")
        self.result = parameters_to_ndarrays(results[0][1].parameters)
        return None, {}

    def configure_evaluate(self, *a):
        return []

    def aggregate_evaluate(self, *a):
        return None, {}

    def evaluate(self, *a):
        return None


def secure_sum(grid, context, node_ids, cfg, bounds, round_no=1):
    """Sum over node_ids of the arrays each client's contribution function returns for cfg; bounds: one public bound
    per array (client divides by it). Returns float64 numpy arrays."""
    node_ids = list(node_ids)
    n = len(node_ids)
    if n < 3:  # SecAgg+ needs at least three parties; with fewer, a sum reveals the summands anyway
        return None
    strat = _SumStrategy(dict(cfg, bounds=[float(b) for b in bounds]))
    lc = LegacyContext(context, config=ServerConfig(num_rounds=1), strategy=strat)
    for nid in node_ids:
        lc.client_manager.register(GridClientProxy(nid, grid, context.run_id))
    lc.state.config_records[MAIN_CONFIGS_RECORD] = ConfigRecord({WK.CURRENT_ROUND: int(round_no)})
    lc.state.array_records[MAIN_PARAMS_RECORD] = ArrayRecord([np.zeros(1)])
    wf = SecAggPlusWorkflow(num_shares=n, reconstruction_threshold=max(2, n // 2 + 1),
                            max_weight=SECAGG["max_weight"], clipping_range=SECAGG["clipping_range"],
                            quantization_range=SECAGG["quantization_range"], modulus_range=SECAGG["modulus_range"])
    wf(grid, lc)
    if strat.result is None:
        raise RuntimeError(f"secure sum of {cfg.get('what')} failed (a client raised an error: see the log above)")
    return [a.astype(np.float64) * n * b for a, b in zip(strat.result, bounds)]


def client_secsum(msg, contrib_fn):
    """Client side of a secure sum (registered with secaggplus_mod): scale by the public bounds, clip, reply in the
    legacy fit format the mod expects. contrib_fn(cfg) -> list of numpy arrays."""
    fitins = compat.recorddict_to_fitins(msg.content, keep_input=True)
    import json
    cfg = json.loads(fitins.config["j"])
    arrays = contrib_fn(cfg)
    lim = SECAGG["clipping_range"]
    out, clipped = [], 0
    for a, b in zip(arrays, cfg["bounds"]):
        s = np.asarray(a, dtype=np.float64) / b
        clipped += int((np.abs(s) > lim).sum())
        out.append(np.clip(s, -lim, lim))
    res = FitRes(status=Status(Code.OK, ""), parameters=ndarrays_to_parameters(out), num_examples=1,
                 metrics={"clipped": clipped})
    return Message(compat.fitres_to_recorddict(res, keep_input=True), reply_to=msg)


In [ ]:
%%writefile /kaggle/working/fedecs_lt/server.py
"""Flower ServerApp: orchestrates one run of the GeFL-F pipeline family (GeFL-F, its variants, the controls and
FedECS-LT) and the baselines that share it. The run is described by the module-level dict RUN (set by runner.py).

Stage I    the feature extractor (FE) is averaged over all clients, each head within its architecture; then frozen.
Release    (FedECS-LT) per-class count, sum h, sum ||h||^2 by secure aggregation -> exact means, spreads, gate F.
Stage II   the feature generator: averaged (FedAvg); with the class-row branch, rows by class-count weighted
           averaging (CCWA, secure sums); with the anchor branch, conditioned on the exact class means.
Stage III  heads trained on generated features (moment-corrected, consensus soft labels for FedECS-LT), then real
           ones (logit-adjusted for FedECS-LT); averaged within each architecture.
Readout    exact class-balanced readout from secure sums of every client's embeddings (reported separately).
Clients evaluate every model the server sends (before and after aggregation); the server collects.
"""
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from flwr.app import Context, Message
from flwr.serverapp import Grid, ServerApp

import fedecs
import local_train as lt
from data import Federation, active_clients
from fl_common import Averager, arrays_of, cfg_of, key_of, nbytes, pack, seed_all
from flops import forward_flops
from metrics import PER_CLASS, log_view, summarize
from models import (GaussFeatureGen, PaperHeader, make_fe, make_generator, make_head, make_headers, n_params,
                    sd_clone)
from monitor import Monitor
from secure_sum import secure_sum
from settings import DATASETS, EVAL_EVERY, FEDECS, HEADER_WIDTHS

DEV = "cuda" if torch.cuda.is_available() else "cpu"
RUN = {}
app = ServerApp()


class Fed:
    """Server-side view of the federation and the messaging helpers (with byte accounting)."""

    def __init__(self, grid, context, R, mon):
        self.grid, self.context, self.R, self.mon = grid, context, R, mon
        self.spec = DATASETS[R["ds"]]
        t0 = time.time()
        while len(list(grid.get_node_ids())) < R["K"]:
            if time.time() - t0 > 600:
                raise RuntimeError("clients did not register")
            time.sleep(0.2)
        nodes = sorted(grid.get_node_ids())
        if DEV == "cuda":   # a new run: the server's GPU memory peak is measured from here
            torch.cuda.reset_peak_memory_stats()
        rep = self.ask("query.hello", [(n, pack(self.base())) for n in nodes], count=False)
        self.nid = {cfg_of(m.content)["pid"]: m.metadata.src_node_id for m in rep}
        self.pid = {v: k for k, v in self.nid.items()}
        self.fed = Federation(R["ds"], R["data_root"], R["IF"], R["alpha"], R["K"], R["seed"], DEV,
                              tail_perm=R.get("tail_perm", False), frac_scale=R.get("frac_scale", 1.0), clients=False)
        self.C, self.K, self.arch, self.groups = self.fed.C, R["K"], self.fed.arch, self.fed.groups
        self.N = int(sum(self.fed.sizes))

    def base(self, **kw):
        R = self.R
        d = dict(ds=R["ds"], IF=R["IF"], alpha=R["alpha"], K=R["K"], seed=R["seed"], tail_perm=R.get("tail_perm", False),
                 frac_scale=R.get("frac_scale", 1.0), data_root=R["data_root"], run_id=R["run_id"])
        d.update(kw)
        return d

    def ask(self, mtype, items, count=True, account=True):
        """Send (node, content) pairs, return replies sorted by client id; raise on any client error."""
        msgs = [Message(content=c, dst_node_id=n, message_type=mtype) for n, c in items]
        down = sum(nbytes(c) for _, c in items)
        rep = list(self.grid.send_and_receive(msgs))
        for m in rep:
            if m.has_error():
                raise RuntimeError(f"{mtype}: client error {m.error.reason[:2000]}")
        if count and account:
            self.mon.count(mtype, down, sum(nbytes(m.content) for m in rep), len(items))
        if hasattr(self, "pid"):
            rep.sort(key=lambda m: self.pid[m.metadata.src_node_id])
        return rep

    def to(self, ks):
        return [self.nid[k] for k in ks]

    def ssum(self, cfg, bounds, clients, round_no=1, secure=True):
        """Sum over clients of their contributions: SecAgg+ when secure (FedECS-LT releases), else plain."""
        n_el = None
        if secure and len(clients) >= 3:
            out = secure_sum(self.grid, self.context, self.to(clients), dict(self.base(), **cfg), bounds, round_no)
            n_el = sum(a.size for a in out)
            self.mon.count("secsum." + cfg["what"], 0, 8 * n_el * len(clients), len(clients))
            return [torch.from_numpy(a).to(DEV) for a in out]
        rep = self.ask("query.sums", [(self.nid[k], pack(dict(self.base(), **cfg))) for k in clients],
                       account=False)
        arrs = None
        for m in rep:
            a = arrays_of(m.content, "s", DEV)
            a = [a[str(i)].double() for i in range(len(a))]
            arrs = a if arrs is None else [x + y for x, y in zip(arrs, a)]
        self.mon.count("sums." + cfg["what"], 0, 8 * sum(int(x.numel()) for x in arrs) * len(clients), len(clients))
        return arrs


# ---------------------------------------------------------------- helpers
def heads_rec(hd_sd, groups):
    return {f"{g}|{k}": v for g in groups for k, v in hd_sd[g].items()}


def eval_heads(S, hd_sd, feat_key, splits=("test",), probs=False, embed=False, lastlayer=None, tag=""):
    """Clients evaluate, the server collects: the first client evaluates every architecture's head."""
    named = dict(heads=heads_rec(hd_sd, S.groups))
    if lastlayer:
        named["lastlayer"] = lastlayer
    rec = pack(S.base(fe_key=feat_key, splits=list(splits), probs=probs, embed=embed), **named)
    rep = S.ask("evaluate.heads", [(S.nid[0], rec)], account=False)
    out = cfg_of(rep[0].content)
    pr = arrays_of(rep[0].content, "probs")
    if pr is not None:
        out["test"]["probs"] = {g: v.numpy() for g, v in pr.items()}
    return out


def flat(prefix, summ):
    keep = ("acc", "bal", "precision", "f1", "mcc", "kappa", "gmean", "worst", "worst_arch", "head", "medium", "tail",
            "rec_std", "auc_roc", "auc_pr", "top5", "ece", "cw_ece", "nll", "brier", "local", "ens_acc", "ens_bal")
    out = {f"{prefix}_{k}": summ[k] for k in keep if k in summ}
    for k in PER_CLASS:
        if k in summ:
            out[f"{prefix}_{k}"] = [round(float(v), 4) for v in summ[k]]
    if "confusion" in summ:
        out[f"{prefix}_confusion"] = summ["confusion"]
    return out


def cache_path(R, kind, key):
    d = os.path.join(R["cache_dir"], kind)
    os.makedirs(d, exist_ok=True)
    return os.path.join(d, key + ".pt")


def claim(path, wait_s=8 * 3600):
    """Parallel workers sharing a stage: the first claims it (lock file holding its process id) and trains it; the
    others wait for its result instead of training the same stage twice. A lock whose process has died is removed.
    Returns True if this process must train the stage."""
    lock = path + ".lock"
    t0 = time.time()
    while True:
        if os.path.exists(path):
            return False
        try:
            fd = os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.write(fd, str(os.getpid()).encode())
            os.close(fd)
            return True
        except FileExistsError:
            try:
                pid = int(open(lock).read() or 0)
                os.kill(pid, 0)
                alive = True
            except (ValueError, ProcessLookupError, FileNotFoundError):
                alive = False
            except PermissionError:
                alive = True
            if not alive or time.time() - t0 > wait_s:
                try:
                    os.remove(lock)
                except FileNotFoundError:
                    pass
                continue
            time.sleep(5)


def release(path):
    try:
        os.remove(path + ".lock")
    except FileNotFoundError:
        pass


# ---------------------------------------------------------------- stage I
def _bytes_delta(before, after):
    return {k: [a - b for a, b in zip(v, before.get(k, [0, 0, 0]))] for k, v in after.items()
            if v != before.get(k)}


def _add_bytes(mon, delta):
    """A stage loaded from cache still counts its communication (it was paid once per run)."""
    for k, (d, u, n) in (delta or {}).items():
        mon.count(k, d, u, n)


def stage_one(S, T_FE, la=False):
    R, spec, mon = S.R, S.spec, S.mon
    key = key_of("stage1", R["ds"], R["IF"], R["alpha"], R["K"], R["seed"], R.get("tail_perm", False),
                 R.get("frac_scale", 1.0), R.get("pfrac", 1.0), T_FE, la)
    path = cache_path(R, "stage1", key)
    if not claim(path):
        c = torch.load(path, weights_only=False)
        _add_bytes(mon, c.get("bytes"))
        mon.log("stage1_cached", key=key, time_s=c["time_s"])
        for rec in c["history"]:
            mon.log("stage1_round", **rec)
        return key, c
    mon.start("stage1")
    b0 = {k: list(v) for k, v in mon.bytes.items()}
    t0 = time.time()
    torch.manual_seed(R["seed"])   # same initial weights as core.py (set_seed(seed) before PaperFE, build_headers)
    np.random.seed(R["seed"])
    fe = make_fe(spec)
    headers = make_headers(spec, fe.out_ch, S.C, "cpu")
    fe_sd = sd_clone(fe)
    hd_sd = [sd_clone(h) for h in headers]
    history = []
    for r in range(T_FE):
        act = active_clients(S.K, R.get("pfrac", 1.0), R["seed"], "i", r)
        ev = r == 0 or (r + 1) % EVAL_EVERY == 0 or r == T_FE - 1
        items = [(S.nid[k], pack(S.base(round=r, la=la, eval=ev), fe=fe_sd, head=hd_sd[S.arch[k]])) for k in act]
        rep = S.ask("train.stage1", items)
        fe_avg, hd_avg = Averager(), {g: Averager() for g in S.groups}
        locs, infos = [], []
        for m in rep:
            k = S.pid[m.metadata.src_node_id]
            f_sd, h_sd = arrays_of(m.content, "fe"), arrays_of(m.content, "head")
            fe_avg.add(f_sd)
            hd_avg[S.arch[k]].add(h_sd)
            locs.append(f_sd)
            infos.append((k, cfg_of(m.content)))
        new_fe = fe_avg.result()
        # drift: distance of each client's FE from the new average (client drift), update norm of the average
        drift = [float(sum((l_[n] - new_fe[n]).float().pow(2).sum() for n in new_fe if new_fe[n].is_floating_point()) ** 0.5)
                 for l_ in locs]
        upd = float(sum((new_fe[n] - fe_sd[n]).float().pow(2).sum() for n in new_fe if new_fe[n].is_floating_point()) ** 0.5)
        bn_div = [float(torch.stack([l_[n].float() for l_ in locs]).var(0).mean()) for n in new_fe if "running_var" in n]
        fe_sd = new_fe
        for g in S.groups:
            if hd_avg[g].n:
                hd_sd[g] = hd_avg[g].result()
        rec = dict(round=r + 1, clients=act, loss={str(k): i["loss"] for k, i in infos}, drift=drift, update_norm=upd,
                   bn_divergence=bn_div, time={str(k): i["time"] for k, i in infos})
        if ev:
            rec["pre"] = {str(k): i["pre"] for k, i in infos if "pre" in i}
            post = S.ask("evaluate.stage1", [(S.nid[0], pack(S.base(), fe=fe_sd, heads=heads_rec(hd_sd, S.groups)))],
                         account=False)
            rec["post"] = log_view(cfg_of(post[0].content)["summary"])
        mon.log("stage1_round", **rec)
        history.append(rec)
    c = dict(fe_sd=fe_sd, hd_sd=hd_sd, history=history, time_s=time.time() - t0, bytes=_bytes_delta(b0, mon.bytes))
    torch.save(c, path + ".part")
    os.replace(path + ".part", path)
    release(path)
    mon.stop("stage1")
    return key, c


def train_referee(S, fe_sd):
    """Server-side referee on the validation set (public server data), as core.train_referee: CNN-5 with BN,
    Adam 1e-3, 40 epochs, batch 64. Used only to measure the generator's per-class fidelity."""
    spec = S.spec
    fe = make_fe(spec).to(DEV)
    fe.load_state_dict(fe_sd)
    fe.eval()
    with torch.no_grad():
        X = torch.cat([fe(S.fed.x_val[s:s + 4000]) for s in range(0, len(S.fed.y_val), 4000)])
    Y = S.fed.y_val
    seed_all(S.R["seed"], "referee")
    ref = PaperHeader(HEADER_WIDTHS[4], X.shape[1], True, S.C).to(DEV)
    opt = torch.optim.Adam(ref.parameters(), lr=1e-3)
    for _ in range(40):
        perm = torch.randperm(len(Y), device=DEV)
        for s in range(0, len(Y), 64):
            b = perm[s:s + 64]
            loss = F.cross_entropy(ref(X[b]), Y[b])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
    ref.eval()
    return ref, X


# ---------------------------------------------------------------- statistics release
def release_stats(S, feat_key, Xval):
    """Secure sum of (n_c, sum h, sum ||h||^2); public bounds from the federation size and the validation features."""
    N = S.N
    fmax = float(Xval.abs().max()) * 4 + 1e-6
    smax = float(Xval.flatten(1).pow(2).sum(1).max()) * 4 + 1e-6
    bounds = [N / 4.0, N * fmax / 4.0, N * smax / 4.0]
    n, S1, S2 = S.ssum(dict(what="stats", fe_key=feat_key), bounds, list(range(S.K)), secure=True)
    n = n.round()
    return n, S1, S2


# ---------------------------------------------------------------- stage II
def stage_two(S, gspec, feat_key, shape, T_KA, ref, proto=None, stats=None):
    R, mon = S.R, S.mon
    key = key_of("stage2", feat_key, gspec, T_KA, R.get("pfrac", 1.0),
                 None if proto is None else float(proto.double().sum()))
    path = cache_path(R, "stage2", key)
    if gspec["type"] == "gauss":
        n, Ssum, M = S.ssum(dict(what="gram", fe_key=feat_key),
                            [S.N / 4.0, S.N * 1e3, S.N * 1e6], list(range(S.K)), secure=False)
        mu, chol = fedecs.gaussian_from_sums(n, Ssum, M, shape)
        G = GaussFeatureGen(mu, chol, shape).to(DEV)
        return key, dict(g_sd=dict(mu=mu, chol=chol), G=G, history=[], time_s=0.0)
    seed_all(R["seed"], "gen_init")
    torch.manual_seed(R["seed"] + 1000)   # core.py: set_seed(seed + 1000) before make_gen
    G = make_generator(gspec["type"], shape[0], S.spec["latent"], S.C, proto)
    if not claim(path):
        c = torch.load(path, weights_only=False)
        G.load_state_dict(c["g_sd"])
        G = G.to(DEV)
        _add_bytes(mon, c.get("bytes"))
        mon.log("stage2_cached", key=key, time_s=c["time_s"])
        for rec in c["history"]:
            mon.log("stage2_round", **rec)
        c["G"] = G
        return key, c
    mon.start("stage2")
    b0 = {k: list(v) for k, v in mon.bytes.items()}
    t0 = time.time()
    g_sd = sd_clone(G)
    G = G.to(DEV)
    ccwa = gspec.get("agg") == "ccwa"
    history = []
    ref_rec = {k: v for k, v in ref.state_dict().items()}
    rnorm = lambda sd: sd["label_proj"].float().norm(dim=1).tolist() if "label_proj" in sd else None  # noqa: E731
    for r in range(T_KA):
        act = active_clients(S.K, R.get("pfrac", 1.0), R["seed"], "ii", r)
        ev = r == 0 or (r + 1) % EVAL_EVERY == 0 or r == T_KA - 1
        cfg = S.base(round=r, fe_key=feat_key, gen=gspec, gen_key=key, send_rows=not ccwa)
        extra = dict(ref=ref_rec) if ev else {}
        items = [(S.nid[k], pack(dict(cfg, eval=ev), g=g_sd, **extra)) for k in act]
        rep = S.ask("train.gen", items)
        avg, infos = Averager(), []
        for m in rep:
            k = S.pid[m.metadata.src_node_id]
            avg.add(arrays_of(m.content, "g"))
            infos.append((k, cfg_of(m.content)))
        new = avg.result()
        if ccwa:
            bound_rows = float(g_sd["label_proj"].abs().max()) * 20 + 1.0
            rs, ws = S.ssum(dict(what="rows", gen_key=key, beta=gspec.get("beta", FEDECS["ccwa_beta"])),
                            [1000.0 * bound_rows / 4.0, 1000.0 / 4.0], act, round_no=r + 1, secure=True)
            new["label_proj"] = fedecs.ccwa_rows(rs.float(), ws.float(), g_sd["label_proj"].to(DEV)).cpu()
        g_sd = {k: v.cpu() for k, v in new.items()}
        rec = dict(round=r + 1, clients=act, loss={str(k): i["loss"] for k, i in infos},
                   row_norm_global=rnorm(g_sd), time={str(k): i["time"] for k, i in infos})
        if any("row_norm" in i for _, i in infos):
            rec["row_norm_local"] = {str(k): i["row_norm"] for k, i in infos if "row_norm" in i}
            rec["row_v"] = {str(k): i["row_v"] for k, i in infos if "row_v" in i}
        if ev and any("fidelity_pre" in i for _, i in infos):
            rec["fidelity_pre"] = {str(k): i["fidelity_pre"] for k, i in infos if "fidelity_pre" in i}
        if ev:
            copies = {"post": g_sd}
            ep = pack(S.base(round=r, fe_key=feat_key, gen=gspec, copies=list(copies)),
                      ref=ref_rec, **{"g_" + k: v for k, v in copies.items()})
            rec["fidelity_post"] = cfg_of(S.ask("evaluate.gen", [(S.nid[0], ep)], account=False)[0].content)["post"]
        mon.log("stage2_round", **rec)
        history.append(rec)
    G.load_state_dict(g_sd)
    c = dict(g_sd=g_sd, history=history, time_s=time.time() - t0, bytes=_bytes_delta(b0, mon.bytes))
    torch.save(c, path + ".part")
    os.replace(path + ".part", path)
    release(path)
    c["G"] = G
    mon.stop("stage2")
    return key, c


@torch.no_grad()
def fidelity(ref, G, C, n=300, seed=0):
    seed_all(seed, "fidelity")
    G.eval()
    out = []
    for c in range(C):
        y = torch.full((n,), c, device=DEV, dtype=torch.long)
        out.append(float((ref(G.sample(y)).argmax(1) == c).float().mean()))
    return out


@torch.no_grad()
def spread_ratio(G, mu, V, C, n=300, seed=0):
    """Generated within-class spread over the exact real spread (1 = realistic)."""
    seed_all(seed, "spread")
    G.eval()
    out = []
    for c in range(C):
        g = G.sample(torch.full((n,), c, device=DEV, dtype=torch.long)).flatten(1)
        out.append(float((g - g.mean(0)).pow(2).sum(1).mean() / V[c].clamp(min=1e-12)))
    return out


# ---------------------------------------------------------------- stage III
def stage_three(S, hs, feat_key, hd_sd, T_TN, sampler, sampler_key, gen_cfg, secure, res):
    R, mon, C = S.R, S.mon, S.C
    mon.start("stage3")
    hd_sd = [dict(s) for s in hd_sd]
    csl = hs.get("csl", 0.0) > 0 and sampler is not None and hs["ts"] > 0
    P, offs = None, None
    if csl:
        P, offs, _ = fedecs.csl_pool_size(S.fed.sizes, hs["ts"], S.spec["B"], S.K, len(S.groups))
    history = []
    G = sampler
    for r in range(T_TN):
        act = active_clients(S.K, R.get("pfrac", 1.0), R["seed"], "iii", r)
        ev = r == 0 or (r + 1) % EVAL_EVERY == 0 or r == T_TN - 1
        cfg = S.base(round=r, fe_key=feat_key, head=hs, sampler_key=sampler_key if sampler is not None else None,
                     eval=ev, C=C, csl_beta=hs.get("csl", 0.0), round_offset=getattr(S, "round_offset", 0))
        cons = None
        rec = dict(round=r + 1, clients=act)
        if csl:
            pool_seed = int(key_of(R["seed"], "pool", r), 16) % (2 ** 31 - 1)
            cfg.update(pool_seed=pool_seed, P=P)
            hk = key_of(R["run_id"], "heads", r)
            S.ask("query.heads", [(S.nid[k], pack(dict(heads_key=hk), heads=heads_rec(hd_sd, S.groups))) for k in act])
            g_act = sorted({S.arch[k] for k in act})
            n_arch = {g: sum(1 for k in act if S.arch[k] == g) for g in g_act}
            w = {str(k): 1.0 / (len(g_act) * n_arch[S.arch[k]]) for k in act}
            out = S.ssum(dict(cfg, what="consensus", heads_key=hk, w=w), [1.0 / 4.0], act, round_no=r + 1,
                         secure=secure)
            cons = out[0].float().view(P, C)
            ent = float(-(cons.clamp(min=1e-12).log() * cons).sum(1).mean())
            rec.update(consensus_entropy=ent, consensus_confidence=float(cons.max(1).values.mean()))
        items = []
        for k in act:
            c = dict(cfg)
            named = dict(head=hd_sd[S.arch[k]])
            if csl:
                nsyn = hs["ts"] * ((S.fed.sizes[k] // S.spec["B"]) if S.fed.sizes[k] >= S.spec["B"] else 1) * \
                    (S.spec["B"] if S.fed.sizes[k] >= S.spec["B"] else S.fed.sizes[k])
                c["pool_off"] = offs[k]
                named["cons"] = dict(t=cons[offs[k]:offs[k] + nsyn].float())
            items.append((S.nid[k], pack(c, **named)))
        rep = S.ask("train.head", items)
        hd_avg, infos = {g: Averager() for g in S.groups}, []
        for m in rep:
            k = S.pid[m.metadata.src_node_id]
            hd_avg[S.arch[k]].add(arrays_of(m.content, "head"))
            infos.append((k, cfg_of(m.content)))
        for g in S.groups:
            if hd_avg[g].n:
                hd_sd[g] = hd_avg[g].result()
        agree = [i["consensus_agreement"] for _, i in infos if i.get("consensus_agreement") is not None]
        if agree:
            rec["consensus_agreement"] = float(np.mean(agree))
        if hs.get("sed") and G is not None:
            hd_sd = feddf_distill(S, hs, G, hd_sd, r)
        rec.update(syn_loss={str(k): i["syn_loss"] for k, i in infos}, real_loss={str(k): i["real_loss"] for k, i in infos},
                   time={str(k): i["time"] for k, i in infos},
                   probe={str(k): i["probe"] for k, i in infos if i.get("probe")},
                   fc_norm={str(g): hd_sd[g]["fc.weight"].float().norm(dim=1).tolist() for g in S.groups})
        if ev:
            rec["pre"] = {str(k): i["pre"] for k, i in infos if "pre" in i}
            post = eval_heads(S, hd_sd, feat_key)["test"]
            rec["post"] = log_view(post["summary"])
            rec["post_pred_counts"] = np.array(post["summary"]["confusion"]).sum(0).tolist()
        mon.log("stage3_round", **rec)
        history.append(rec)
    mon.stop("stage3")
    return hd_sd, history


def feddf_distill(S, hs, G, hd_sd, r):
    """FedDF-style server ensemble distillation on generated features (as core.stage_iii 'sed')."""
    C = S.C
    seed_all(S.R["seed"], "feddf", r)
    in_ch = hd_sd[S.groups[0]]["features.0.weight"].shape[1]
    headers = make_headers(S.spec, in_ch, C, DEV)
    n_sed = hs["sed"] * hs.get("sed_bs", 128)
    sy = torch.randint(0, C, (n_sed,), device=DEV)
    with torch.no_grad():
        sx = torch.cat([G.sample(sy[s:s + 4096]) for s in range(0, n_sed, 4096)])
        cons = torch.zeros(n_sed, C, device=DEV)
        for g in S.groups:
            headers[g].load_state_dict(hd_sd[g])
            headers[g].eval()
            cons += torch.cat([headers[g](sx[s:s + 4096]).softmax(1) for s in range(0, n_sed, 4096)])
        cons /= len(S.groups)
        tgt = (1 - hs.get("sed_beta", 1.0)) * F.one_hot(sy, C).float() + hs.get("sed_beta", 1.0) * cons
    out = list(hd_sd)
    for g in S.groups:
        h = headers[g]
        h.train()
        opt = torch.optim.SGD(h.parameters(), lr=hs.get("sed_lr", 0.05), momentum=0.0)
        ok = True
        bs = hs.get("sed_bs", 128)
        for s in range(0, n_sed, bs):
            loss = -(tgt[s:s + bs] * h(sx[s:s + bs]).log_softmax(1)).sum(1).mean()
            if not torch.isfinite(loss):
                ok = False
                break
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        if ok:
            new_sd = {k: v.detach().cpu().clone() for k, v in h.state_dict().items()}
            for k, v in hd_sd[g].items():  # BN running statistics stay those of real features
                if "running_" in k or "num_batches" in k:
                    new_sd[k] = v
            out[g] = new_sd
    return out


# ---------------------------------------------------------------- exact readout
def exact_readout(S, hd_sd, feat_key, Xval, secure, gamma=0.1):
    """Per architecture: secure sums of every client's (n_c, sum e, sum e e^T) through that head's body ->
    class-balanced LDA last layer. Bounds from the validation embeddings (public server data)."""
    hk = key_of(S.R["run_id"], "heads", "readout")
    S.ask("query.heads", [(S.nid[k], pack(dict(heads_key=hk), heads=heads_rec(hd_sd, S.groups))) for k in range(S.K)])
    bounds = []
    in_ch = Xval.shape[1]
    for g in S.groups:
        h = make_head(S.spec, g, in_ch, S.C).to(DEV)
        h.load_state_dict(hd_sd[g])
        h.eval()
        with torch.no_grad():
            E = h.embed(Xval)
        emax = float(E.abs().max()) * 4 + 1e-6
        bounds += [S.N / 4.0, S.N * emax / 4.0, S.N * emax * emax / 4.0]
    out = S.ssum(dict(what="readout", heads_key=hk, groups=S.groups, fe_key=feat_key), bounds, list(range(S.K)),
                 secure=secure)
    last = {}
    for i, g in enumerate(S.groups):
        n, Ssum, M = out[3 * i], out[3 * i + 1], out[3 * i + 2]
        W, b = fedecs.lda_from_sums(n.round(), Ssum, M, gamma)
        last[f"{g}|W"], last[f"{g}|b"] = W.cpu(), b.cpu()
    return last


# ---------------------------------------------------------------- representation and cost measurements
def active_units(G, X, y, thr=0.01):
    """Active latent units of a VAE generator (Burda et al., ICLR 2016): latent dimensions whose posterior mean
    varies over the validation features by more than thr. None for generators without an encoder."""
    if G is None or not hasattr(G, "encode"):
        return {}
    modes = [(m, m.training) for m in G.modules()]
    G.eval()
    with torch.no_grad():
        mus = torch.cat([G.encode(X[s:s + 2048].float(), y[s:s + 2048])[0] for s in range(0, len(y), 2048)])
    for m, t in modes:
        m.training = t
    var = mus.float().var(0)
    return dict(active_units=int((var > thr).sum()), latent_dim=int(var.numel()), latent_var=var.tolist())


class _Sampler(torch.nn.Module):
    def __init__(self, G):
        super().__init__()
        self.G = G

    def forward(self, y):
        return self.G.sample(y)


def model_flops(S, shape, G, hs, pfrac=1.0):
    """Forward FLOPs per image of the extractor, of every head architecture and of one generated feature, and the
    expected training FLOPs of one head-stage round over all clients (3 x forward: forward and backward pass;
    real epochs plus the synthetic pass; pfrac of the clients per round)."""
    fe = make_fe(S.spec).to(DEV)
    f_fe = forward_flops(fe, torch.zeros(2, S.spec["in_ch"], 32, 32, device=DEV))
    headers = make_headers(S.spec, shape[0], S.C, DEV)
    f_head = {str(g): forward_flops(headers[g], torch.zeros(2, *shape, device=DEV)) for g in S.groups}
    f_gen = forward_flops(_Sampler(G), torch.zeros(2, dtype=torch.long, device=DEV)) if G is not None else 0.0
    per_round = 0.0
    for k in range(S.K):
        n = int(S.fed.sizes[k])
        nsyn = lt.n_syn(n, S.spec["B"], hs["ts"]) if hs.get("ts", 0) > 0 and G is not None else 0
        per_round += 3 * f_head[str(S.arch[k])] * (hs["tr"] * n + nsyn)
    return dict(flops_fe=f_fe, flops_head=f_head, flops_gen=f_gen,
                head_stage_gflops_per_round=pfrac * per_round / 1e9)


# ---------------------------------------------------------------- one run
def run_pipeline(grid, context, R):
    mon = Monitor(os.path.join(R["out_dir"], "logs"), R["run_id"], tensorboard=R.get("tensorboard", True))
    t0 = time.time()
    S = Fed(grid, context, R, mon)
    m = R["method"]
    spec = S.spec
    T = dict(T_FE=spec["T_FE"], T_KA=spec["T_KA"], T_TN=spec["T_TN"])
    T.update(R.get("rounds") or {})
    res = dict(class_order=S.fed.class_order, class_names=S.fed.class_names,
               test_class_counts=np.bincount(S.fed.y_te.cpu().numpy(), minlength=S.C).tolist(),
               run_id=R["run_id"], label=R["label"], dataset=R["ds"], IF=R["IF"], alpha=R["alpha"], K=R["K"],
               seed=R["seed"], pfrac=R.get("pfrac", 1.0), tail_perm=R.get("tail_perm", False), method=m,
               n_train=S.N, min_client=int(min(S.fed.sizes)), client_counts=S.fed.counts.astype(int).tolist(),
               rounds=T, framework="flower")
    mon.log("run_start", **{k: v for k, v in res.items() if k != "client_counts"})
    if m["pipeline"] == "local":
        return run_local(S, R, T, res, mon, t0)
    feat_key, s1 = stage_one(S, T["T_FE"], la=m.get("fe_la", False))
    res["fe_time_s"] = s1["time_s"]
    res["stage1_final"] = s1["history"][-1].get("post")
    S.ask("query.features", [(S.nid[k], pack(S.base(fe_key=feat_key), fe=s1["fe_sd"])) for k in range(S.K)])
    ref, Xval = train_referee(S, s1["fe_sd"])
    shape = tuple(Xval.shape[1:])
    secure = m.get("secure", False)
    stats = None
    if m.get("stats"):
        n, S1, S2 = release_stats(S, feat_key, Xval)
        stats = (n, S1, S2)
        res["class_sep_F"] = fedecs.separation_F(n, S1, S2)
        _, mu, V = fedecs.moments_from_sums(n, S1, S2)
        mon.log("stats_release", F=res["class_sep_F"], n=n.tolist())
    G, gen_cfg, sampler_key = None, None, None
    gspec = m.get("gen")
    if gspec is not None:
        gspec = dict(gspec)
        if gspec["type"] == "gate":
            rows = res["class_sep_F"] < FEDECS["gate_F"]
            if R.get("force_branch") in ("rows", "anchor"):   # only for tests of the branch not chosen by the gate
                rows = R["force_branch"] == "rows"
            gspec.update(dict(type="cvae", agg="ccwa") if rows else dict(type="pcvae"))
            gspec.pop("gate", None)
            res["gate_branch"] = "rows" if rows else "anchor"
        proto = None
        if gspec["type"] == "pcvae":
            proto = mu.float().view(S.C, *shape)
        key2, s2 = stage_two(S, gspec, feat_key, shape, T["T_KA"], ref, proto=proto, stats=stats)
        res["gen_time_s"] = s2["time_s"]
        G = s2["G"]
        res["generator_params"] = n_params(G) if gspec["type"] != "gauss" else 0
        if "label_proj" in s2["g_sd"]:
            rn = s2["g_sd"]["label_proj"].float().norm(dim=1)
            res["cond_norms_end"] = rn.tolist()
        res["fidelity"] = fidelity(ref, G, S.C, seed=R["seed"])
        res.update(active_units(G, Xval, S.fed.y_val))
        sampler = G
        mc_arr = None
        if m["head"].get("mc"):
            sampler = fedecs.MomentCalibratedGen(G, mu.to(DEV), V.to(DEV), n=FEDECS["mc_samples"],
                                                 clip=FEDECS["mc_clip"], seed=R["seed"])
            mc_arr = dict(mu=sampler.mu, m=sampler.m, scale=sampler.scale)
            res["mc_scale"] = sampler.scale.tolist()
            res["fidelity_mc"] = fidelity(ref, sampler, S.C, seed=R["seed"])
        if stats is not None:
            res["spread_ratio"] = spread_ratio(sampler, mu.to(DEV), V.to(DEV), S.C, seed=R["seed"])
        sampler_key = key_of(key2, "mc" if mc_arr else "plain")
        gen_cfg = gspec
        lg = dict(sampler_key=sampler_key, gen=gspec)
        S.ask("query.load_gen", [(S.nid[k], pack(S.base(fe_key=feat_key, **lg), g=s2["g_sd"],
                                                 **({"mc": mc_arr} if mc_arr else {}))) for k in range(S.K)])
        G = sampler
    hs = dict(m["head"])
    if G is None:
        hs["ts"] = 0
    if spec.get("local_epochs"):   # Track 2 protocol: local epochs of the benchmark
        hs["tr"] = spec["local_epochs"]
    S.round_offset = T["T_FE"]
    hd_sd, h_hist = stage_three(S, hs, feat_key, s1["hd_sd"], T["T_TN"], G, sampler_key, gen_cfg, secure, res)
    res["head_time_s"] = sum(h.get("time", {}).get(str(0), 0) for h in h_hist)
    # final evaluation: test (with probabilities and embeddings for the plots) and validation
    fin = eval_heads(S, hd_sd, feat_key, splits=("test", "val"), probs=True, embed=True)
    res.update(flat("final", fin["test"]["summary"]))
    res.update(flat("val_final", fin["val"]["summary"]))
    res["final_per_arch"] = {g: {k: v for k, v in mm.items() if k in ("acc", "bal", "f1", "tail", "worst", "local")}
                             for g, mm in fin["test"]["per_arch"].items()}
    # exact readout (reported with and without)
    if m.get("readout", True):
        last = exact_readout(S, hd_sd, feat_key, Xval, secure)
        er = eval_heads(S, hd_sd, feat_key, splits=("test", "val"), lastlayer=last, probs=True, embed=False)
        res.update(flat("er", er["test"]["summary"]))
        res.update(flat("val_er", er["val"]["summary"]))
        er_probs = er["test"].get("probs")
    else:
        er_probs, last = None, None
    res.update(model_flops(S, shape, s2["G"] if gspec is not None and gspec["type"] != "gauss" else None, hs,
                           R.get("pfrac", 1.0)))
    res["time_s"] = time.time() - t0
    res["stage_time_s"] = dict(mon.stage_t)
    res["bytes"] = {k: dict(down=v[0], up=v[1], messages=v[2]) for k, v in mon.bytes.items()}
    res["comm_mb_per_client"] = sum(v[0] + v[1] for v in mon.bytes.values()) / 2 ** 20 / S.K
    res.update(gpu_peaks(S))
    extra = dict(y_full=S.fed.y_te.cpu().numpy())
    # last layers (classifier vectors) of the heads as trained and of the exact readout: neural-collapse NC2 / NC3
    # and the minority-collapse angle are computed from them with the test embeddings
    for g in S.groups:
        extra[f"fcw_{g}"] = hd_sd[g]["fc.weight"].float().cpu().numpy()
        extra[f"fcb_{g}"] = hd_sd[g]["fc.bias"].float().cpu().numpy()
        if last is not None and f"{g}|W" in last:
            extra[f"erw_{g}"] = last[f"{g}|W"].float().cpu().numpy()
    nv = min(2000, len(Xval))
    sub = np.random.RandomState(0).choice(len(Xval), nv, replace=False)
    extra.update(val_x=Xval[sub].flatten(1).cpu().numpy().astype(np.float16), val_y=S.fed.y_val[sub].cpu().numpy())
    if G is not None:
        seed_all(R["seed"], "gen_samples")
        gy = torch.arange(S.C, device=DEV).repeat_interleave(max(10, 1000 // S.C))
        with torch.no_grad():
            gx = torch.cat([G.sample(gy[i:i + 4096]) for i in range(0, len(gy), 4096)])
        extra.update(gen_x=gx.flatten(1).cpu().numpy().astype(np.float16), gen_y=gy.cpu().numpy())
    save_run(R, res, s1, hd_sd, G, ref, fin, er_probs, last, s2 if gspec is not None else None, extra)
    mon.log("run_end", time_s=res["time_s"])
    mon.close()
    return res


def run_local(S, R, T, res, mon, t0):
    """Local (no federation): every client trains its own extractor and head for the federated stages' epochs combined."""
    spec = S.spec
    torch.manual_seed(R["seed"])
    fe = make_fe(spec)
    headers = make_headers(spec, fe.out_ch, S.C, "cpu")
    epochs = 5 * T["T_FE"] + 5 * T["T_TN"]
    rep = S.ask("train.local", [(S.nid[k], pack(S.base(epochs=epochs), fe=sd_clone(fe),
                                                head=sd_clone(headers[S.arch[k]]))) for k in range(S.K)])
    per_client = {str(S.pid[m.metadata.src_node_id]): cfg_of(m.content)["metrics"] for m in rep}
    per_client_val = {str(S.pid[m.metadata.src_node_id]): cfg_of(m.content)["val_metrics"] for m in rep}
    # every scalar and every per-class metric, averaged over the clients (each client's own model)
    res.update(flat("final", summarize(per_client)))
    res.update(flat("val_final", summarize(per_client_val)))
    res["local_per_client"] = {k: log_view(v) for k, v in per_client.items()}
    res["time_s"] = time.time() - t0
    save_run(R, res, None, None, None, None, None, None, None, None)
    mon.log("run_end", time_s=res["time_s"])
    mon.close()
    return res


def gpu_peaks(S):
    """Peak GPU memory of the run (MB): the server process and the client process(es) share the worker's GPU, so
    their sum is an upper bound of what the run needed (CUDA's own context, ~0.3-0.5 GB per process, comes on top)."""
    rep = S.ask("query.gpu", [(S.nid[k], pack(S.base())) for k in range(S.K)], account=False)   # (also on CPU)
    procs = {}
    for m in rep:
        o = cfg_of(m.content)
        procs[o["pidproc"]] = max(procs.get(o["pidproc"], 0), o["peak_mb"])
    if DEV != "cuda":
        return {}
    server = round(torch.cuda.max_memory_allocated() / 2 ** 20)
    client = sum(procs.values())
    return dict(gpu_peak_mb=server + client, gpu_peak_server_mb=server, gpu_peak_clients_mb=client)


def save_run(R, res, s1, hd_sd, G, ref, fin, er_probs, last, s2, extra=None):
    out = os.path.join(R["out_dir"], "runs.jsonl")
    os.makedirs(R["out_dir"], exist_ok=True)
    with open(out, "a") as f:
        f.write(json.dumps(res) + "\n")
    if fin is not None:
        pdir = os.path.join(R["out_dir"], "predictions")
        os.makedirs(pdir, exist_ok=True)
        safe = "".join(ch if ch.isalnum() or ch in "-_.=" else "_" for ch in R["run_id"])
        np.savez_compressed(os.path.join(pdir, safe + ".npz"),
                            y=np.array(fin["test"]["embed_y"]),
                            **{f"probs_{g}": np.array(p, dtype=np.float16) for g, p in fin["test"]["probs"].items()},
                            **{f"embed_{g}": np.array(e, dtype=np.float16) for g, e in fin["test"]["embed"].items()},
                            **({f"er_probs_{g}": np.array(p, dtype=np.float16) for g, p in er_probs.items()}
                               if er_probs else {}), **(extra or {}))
    if R.get("models_dir") and (s1 is not None or hd_sd is not None):
        mdir = os.path.join(R["models_dir"])
        os.makedirs(mdir, exist_ok=True)
        safe = "".join(ch if ch.isalnum() or ch in "-_.=" else "_" for ch in R["run_id"])
        torch.save(dict(fe=s1["fe_sd"] if s1 is not None else None, heads=hd_sd, gen=(s2["g_sd"] if s2 is not None else None),
                        referee=ref.state_dict() if ref is not None else None, readout=last, method=R["method"],
                        run_id=R["run_id"]), os.path.join(mdir, safe + ".pt"))


@app.main()
def main(grid: Grid, context: Context):
    R = RUN
    if R["method"].get("pipeline") in ("gefl", "local"):
        RUN["result"] = run_pipeline(grid, context, R)
    else:
        from baselines import run_baseline
        RUN["result"] = run_baseline(grid, context, R)


In [ ]:
%%writefile /kaggle/working/fedecs_lt/settings.py
"""Fixed settings of the FedECS-LT experiment suite: datasets, rounds, seeds and the method registry.

Values for the five tier-1 datasets are copied from experiments/core.py (DATASETS, HEADER_WIDTHS), which produced every
earlier result, so new runs pair with old ones (same partitions for the same seeds). Every method carries its citation
and the source of its code ("ours", "paste": licence verified, code included with credit, "fetch": no licence, the
authors' repository is downloaded at run time at a pinned commit).
"""

SEEDS = [3, 4, 5]
EVAL_EVERY = 5            # rounds between evaluations (pre- and post-aggregation)
DEADLINE_H = 11.0         # a notebook starts no new run after this many hours (Kaggle limit 12 h)

# ---------------------------------------------------------------- datasets
# in_ch, normalisation, BN in the heads, FE variant, fraction of the training set used, batch size, rounds per stage
# (T_FE extractor, T_KA generator, T_TN heads), generator latent size, augmentation, number of classes, imbalance:
#   "IF": long tail made by subsampling (imbalance factor from the run), "natural": the dataset's own class sizes.
DATASETS = {
    "mnist": dict(name="MNIST", in_ch=1, norm="none", header_bn=False, fe="mnist", frac=0.1, B=64,
                  T_FE=20, T_KA=100, T_TN=50, latent=16, augment=False, C=10, imbalance="IF", tier=1),
    "fmnist": dict(name="Fashion-MNIST", in_ch=1, norm="half", header_bn=True, fe="mnist", frac=0.1, B=64,
                   T_FE=20, T_KA=100, T_TN=50, latent=16, augment=False, C=10, imbalance="IF", tier=1),
    "svhn": dict(name="SVHN", in_ch=3, norm=((0.4376821, 0.4437697, 0.47280442), (0.19803012, 0.20101562, 0.19703614)),
                 header_bn=True, fe="mnist", frac=0.1, B=64, T_FE=50, T_KA=100, T_TN=100, latent=16, augment=False,
                 C=10, imbalance="IF", tier=1),
    "cifar10": dict(name="CIFAR-10", in_ch=3, norm=((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                    header_bn=False, fe="cifar10", frac=0.5, B=128, T_FE=50, T_KA=200, T_TN=100, latent=50,
                    augment=True, C=10, imbalance="IF", tier=1),
    "cifar100": dict(name="CIFAR-100", in_ch=3, norm=((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
                     header_bn=False, fe="cifar10", frac=0.22, B=128, T_FE=50, T_KA=200, T_TN=100, latent=50,
                     augment=True, C=100, imbalance="IF", tier=1),
    # Tier 2: natural imbalance, training sets capped at 20,000 images (frac = min(1, 20000 / N)); 28x28 images are
    # resized to 32x32 as MNIST is; grey images use MNIST's extractor and rounds, colour images CIFAR-10's.
    "dermamnist": dict(name="DermaMNIST", in_ch=3, norm="half", header_bn=True, fe="cifar10", frac=None, B=64,
                       T_FE=50, T_KA=100, T_TN=100, latent=16, augment=False, C=7, imbalance="natural", tier=2),
    "organamnist": dict(name="OrganAMNIST", in_ch=1, norm="half", header_bn=True, fe="mnist", frac=None, B=64,
                        T_FE=20, T_KA=100, T_TN=50, latent=16, augment=False, C=11, imbalance="natural", tier=2),
    "bloodmnist": dict(name="BloodMNIST", in_ch=3, norm="half", header_bn=True, fe="cifar10", frac=None, B=64,
                       T_FE=50, T_KA=100, T_TN=100, latent=16, augment=False, C=8, imbalance="natural", tier=2),
    "pathmnist": dict(name="PathMNIST", in_ch=3, norm="half", header_bn=True, fe="cifar10", frac=None, B=64,
                      T_FE=50, T_KA=100, T_TN=100, latent=16, augment=False, C=9, imbalance="natural", tier=2),
    "gtsrb": dict(name="GTSRB", in_ch=3, norm="half", header_bn=True, fe="cifar10", frac=None, B=64,
                  T_FE=50, T_KA=100, T_TN=100, latent=16, augment=False, C=43, imbalance="natural", tier=2),
    "emnist": dict(name="EMNIST (ByClass)", in_ch=1, norm="half", header_bn=True, fe="mnist", frac=None, B=64,
                   T_FE=20, T_KA=100, T_TN=50, latent=16, augment=False, C=62, imbalance="natural", tier=2),
    "fer2013": dict(name="FER-2013", in_ch=1, norm="half", header_bn=True, fe="mnist", frac=None, B=64,
                    T_FE=50, T_KA=100, T_TN=100, latent=16, augment=False, C=7, imbalance="natural", tier=2),
    # Track 2 (one shared model, FedLTLib / CReFF protocol): standard CIFAR-LT profile, 20 clients, 40% per round,
    # ResNet-8 split after its second stage (frozen extractor) for FedECS-LT; SGD 0.1, momentum 0.9, learning-rate
    # decay 0.99 per round, 10 local epochs, batch 32, as the baselines in LongTailLib. T_FE + T_TN = 200 rounds, the
    # baselines' total; the generator's rounds (T_KA) are extra and reported.
    "cifar10_t2": dict(name="CIFAR-10", raw="cifar10", in_ch=3, norm=((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
                       header_bn=True, fe="resnet8", heads="resnet8", frac=None, B=32, T_FE=100, T_KA=100, T_TN=100,
                       latent=50, augment=True, C=10, imbalance="IF", partition="lt_standard", single_arch=True,
                       track=2, local_epochs=10, opt=dict(lr=0.1, momentum=0.9, decay=0.99), tier=1),
    "cifar100_t2": dict(name="CIFAR-100", raw="cifar100", in_ch=3, norm=((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
                        header_bn=True, fe="resnet8", heads="resnet8", frac=None, B=32, T_FE=100, T_KA=100, T_TN=100,
                        latent=50, augment=True, C=100, imbalance="IF", partition="lt_standard", single_arch=True,
                        track=2, local_epochs=10, opt=dict(lr=0.1, momentum=0.9, decay=0.99), tier=1),
}
CAP_IMAGES = 20000
HEADER_WIDTHS = [(16,), (16, 32), (20, 40), (10, 20), (16, 32, 64), (20, 40, 80),
                 (10, 20, 40), (16, 32, 64, 128), (20, 40, 80, 100), (10, 20, 40, 80)]
N_ARCH = 10


def class_groups(C):
    """Head / medium / tail classes by frequency rank (first 30%, next 30%, last 40%), as in all earlier results."""
    a, b = int(round(0.3 * C)), int(round(0.6 * C))
    return slice(0, a), slice(a, b), slice(b, C)


# ---------------------------------------------------------------- our method's settings (action plan section 12)
FEDECS = dict(
    gate_F=0.05,          # F below this: class-row generator with class-count weighted averaging (CCWA)
    ccwa_beta=0.999,      # E(n) = (1 - beta^n) / (1 - beta)
    mc_samples=500,       # generator samples per class to measure its moments (moment correction)
    mc_clip=(0.1, 10.0),  # bounds of the spread ratio
    csl_beta=0.5,         # consensus soft label = (1 - beta) one-hot + beta consensus
    la_tau=1.0,           # logit adjustment strength
    ts=10,                # synthetic epochs per round (T_s), all datasets
    tr=5,                 # real epochs per round
    er_gamma=0.1,         # shrinkage of the exact readout's within-class covariance
)

# Secure aggregation (Flower SecAgg+): 30-bit quantisation (Flower's default 22 bits loses precision on sums), modulus
# 2^40, every vector divided by a public bound before quantisation so that it lies inside the clipping range +-8.
SECAGG = dict(quantization_range=2 ** 30, modulus_range=2 ** 40, clipping_range=8.0, max_weight=1.0)

# ---------------------------------------------------------------- methods
# kind: "pipeline" (runs the GeFL-F three-stage pipeline with the given options) or "baseline" (an adapter in
# baselines/). Every method of the main comparison; "rank" A = must have, B = second priority, C = control.
CITE = {
    "GeFL-F": "Kang et al., IEEE TMC 2025",
    "FedProto": "Tan et al., AAAI 2022", "FedTGP": "Zhang et al., AAAI 2024", "FedGen": "Zhu et al., ICML 2021",
    "FedDF": "Lin et al., NeurIPS 2020", "CReFF": "Shang et al., IJCAI 2022", "CCVR": "Luo et al., NeurIPS 2021",
    "FedETF": "Li et al., ICCV 2023", "FedLoGe": "Xiao et al., ICLR 2024", "Fed-GraB": "Xiao et al., NeurIPS 2023",
    "FedYoYo": "Yan et al., ICCV 2025", "FedLC": "Zhang et al., ICML 2022", "FedRS": "Li and Zhan, KDD 2021",
    "RUCR": "Huang et al., IEEE TIFS 2024", "FedGH": "Yi et al., ACM MM 2023", "LG-FedAvg": "Liang et al., 2020",
    "FedNH": "Dai et al., AAAI 2023", "FedVLS": "Guo et al., AAAI 2025", "FedMRL": "Yi et al., NeurIPS 2024",
    "FedAvg": "McMahan et al., AISTATS 2017", "FedProx": "Li et al., MLSys 2020", "MOON": "Li et al., CVPR 2021",
    "CLIP2FL": "Shi et al., AAAI 2024", "Balanced Softmax": "Ren et al., NeurIPS 2020",
    "LA": "Menon et al., ICLR 2021",
}


In [ ]:
%%writefile /kaggle/working/fedecs_lt/track2.py
"""Track 2 (one shared model; the FedLTLib benchmark protocol): the baselines run the authors' LongTailLib code
unchanged (Lin and Wang, FedLTLib, arXiv 2609.15625; https://github.com/gzhu-hcai/LongTailLib, MIT licence stated in
its README; downloaded at run time at a pinned commit, never stored in this repository), on this suite's client splits.

  prepare()  writes our split of CIFAR-10/100-LT into LongTailLib's dataset format (per-client train / test files and
             the balanced global test set)
  driver     runs LongTailLib's main.py with its default FedLTLib protocol (20 clients, 40% per round, 200 rounds,
             10 local epochs, batch 32, SGD 0.1 with momentum 0.9 and decay 0.99 per round), ResNet-8 for every
             method; keeps the server object, then re-runs the library's own global-test evaluation while recording
             the scores it computes, so that every metric of this suite is computed from exactly the predictions the
             library itself uses (checked: the recorded predictions give the library's own accuracy).

    python track2.py prepare DS IF SEED LT_DIR DATA_ROOT
    python track2.py run ALGO DS IF SEED LT_DIR OUT_DIR [ROUNDS]
"""
import json
import os
import subprocess
import sys
import time

import numpy as np

LT_REPO = "https://github.com/gzhu-hcai/LongTailLib.git"
LT_COMMIT = "414167807aa244819bde33988ae36d07edfed645"
ALGOS = {"FedAvg": "FedAvg", "FedProx": "FedProx", "MOON": "MOON", "FedLC": "FedLC", "CCVR": "CCVR",
         "CReFF": "CReFF", "FedETF": "FedETF", "Fed-GraB": "FedGraB", "FedLoGe": "FedLoGe", "FedYoYo": "FedYoYo",
         "RUCR": "RUCR", "CLIP2FL": "CLIP2FL"}
K2, ALPHA2, JR2 = 20, 0.5, 0.4


def fetch(lt_dir):
    """The authors' code at the pinned commit (downloaded once per notebook)."""
    if not os.path.isdir(os.path.join(lt_dir, ".git")):
        subprocess.run(["git", "clone", "-q", LT_REPO, lt_dir], check=True)
    subprocess.run(["git", "-C", lt_dir, "fetch", "-q", "--depth", "50", "origin", LT_COMMIT], check=False)
    subprocess.run(["git", "-C", lt_dir, "checkout", "-q", LT_COMMIT], check=True)
    return lt_dir


def folder(ds, IF, seed):
    return f"{'Cifar100' if ds == 'cifar100' else 'Cifar10'}-IF{int(round(1 / IF))}-a{ALPHA2}-K{K2}-s{seed}"


def lt_partition(y, C, K, IF, alpha, seed, min_size=10):
    """Standard CIFAR-LT profile n_c = n_max IF^(c/(C-1)) (Cao et al., 2019; CReFF), after holding out the
    validation images (200 per class; 20 for C = 100), then a Dirichlet(alpha) split over K clients (same rule and
    random calls as data.make_partition)."""
    rng = np.random.RandomState(seed)
    by_class = [rng.permutation(np.where(y == c)[0]) for c in range(C)]
    nh = min(200, 2000 // C)
    heldout = np.concatenate([bc[:nh] for bc in by_class])
    n_max = min(len(bc) for bc in by_class) - nh
    want = [int(n_max * IF ** (c / (C - 1))) for c in range(C)]
    pools = [bc[nh:nh + want[c]] for c, bc in enumerate(by_class)]
    for _ in range(100):
        parts = [[] for _ in range(K)]
        for c in range(C):
            prop = rng.dirichlet([alpha] * K)
            cuts = (np.cumsum(prop) * len(pools[c])).astype(int)[:-1]
            for k, chunk in enumerate(np.split(pools[c], cuts)):
                parts[k].extend(chunk.tolist())
        if min(len(p) for p in parts) >= min_size:
            break
    while min(len(p) for p in parts) < min_size:
        small, big = int(np.argmin([len(p) for p in parts])), int(np.argmax([len(p) for p in parts]))
        parts[small].append(parts[big].pop(rng.randint(len(parts[big]))))
    parts = [rng.permutation(p) for p in parts]
    counts = np.array([[int((y[p] == c).sum()) for c in range(C)] for p in parts])
    return parts, heldout, counts


def raw_cifar(ds, data_root):
    """uint8 NHWC images and labels (the same Hugging Face copies data.py uses)."""
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from data import _raw
    xtr, ytr, xte, yte = _raw(ds, data_root)
    return (xtr.permute(0, 2, 3, 1).numpy(), ytr.numpy(), xte.permute(0, 2, 3, 1).numpy(), yte.numpy())


def prepare(ds, IF, seed, lt_dir, data_root):
    C = 100 if ds == "cifar100" else 10
    xtr, ytr, xte, yte = raw_cifar(ds, data_root)
    parts, heldout, counts = lt_partition(ytr, C, K2, IF, ALPHA2, seed)
    root = os.path.join(lt_dir, "dataset", folder(ds, IF, seed))
    os.makedirs(os.path.join(root, "train"), exist_ok=True)
    os.makedirs(os.path.join(root, "test"), exist_ok=True)
    rng = np.random.RandomState(10_000 + seed)
    for k, p in enumerate(parts):
        np.savez_compressed(os.path.join(root, "train", f"{k}.npz"), data=dict(x=xtr[p], y=ytr[p].astype(np.int64)))
        t = rng.choice(len(yte), 200, replace=False)  # small local test slice (the library averages client tests)
        np.savez_compressed(os.path.join(root, "test", f"{k}.npz"), data=dict(x=xte[t], y=yte[t].astype(np.int64)))
    np.savez_compressed(os.path.join(root, "global_test.npz"), data=dict(x=xte, y=yte.astype(np.int64)))
    json.dump(dict(num_clients=K2, num_classes=C, non_iid=True, balance=False, partition="dir", alpha=ALPHA2,
                   longtail=True, imbalance_factor=int(round(1 / IF)), seed=seed, source="FedECS-LT suite split"),
              open(os.path.join(root, "config.json"), "w"))
    np.savez_compressed(os.path.join(root, "suite_split.npz"), counts=counts, heldout=heldout)
    return root, counts


DRIVER = r'''
import json, os, runpy, sys, time
import numpy as np, torch
sys.path.insert(0, os.getcwd())
import flcore.servers.serverbase as sb
CAP = {}
def _wrap(cls):
    if "train" in cls.__dict__ and not getattr(cls.__dict__["train"], "_fedecs", False):
        orig = cls.__dict__["train"]
        def train(self, *a, **k):
            CAP["server"] = self
            return orig(self, *a, **k)
        train._fedecs = True
        cls.train = train
args, out = json.loads(sys.argv[1]), sys.argv[2]
# every global evaluation the library makes (every eval_gap rounds): record the scores it ranks and compute every
# per-class metric with the suite's metrics.py (wrappers chain, so the final re-evaluation below still records)
import importlib.util
sys.path.append(os.environ["FEDECS_SRC"])
_spec = importlib.util.spec_from_file_location("fedecs_metrics", os.path.join(os.environ["FEDECS_SRC"], "metrics.py"))
FM = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(FM)
_name = args[args.index("-data") + 1]
Y = torch.from_numpy(np.load(os.path.join("..", "dataset", _name, "global_test.npz"), allow_pickle=True)["data"]
                     .item()["y"]).long()
C_ = int(args[args.index("-ncl") + 1])
PER, DEPTH, ROUND = [], [0], [0]
def _wrap_round(cls):
    """Counts completed rounds (one aggregation per round; a subclass calling super() counts once)."""
    orig = cls.__dict__["aggregate_parameters"]
    if getattr(orig, "_fedecs", False):
        return
    def f(self, *a, **k):
        DEPTH[0] += 1
        try:
            return orig(self, *a, **k)
        finally:
            DEPTH[0] -= 1
            if DEPTH[0] == 0:
                ROUND[0] += 1
    f._fedecs = True
    setattr(cls, "aggregate_parameters", f)
def _wrap_eval(cls, name):
    orig = cls.__dict__[name]
    if getattr(orig, "_fedecs", False):
        return
    def f(self, *a, **k):
        if DEPTH[0]:
            return orig(self, *a, **k)
        rec, pmax, ptmax = [], torch.max, torch.Tensor.max
        def m1(x, *aa, **kk):
            if torch.is_tensor(x) and x.dim() == 2 and x.is_floating_point() and (aa[:1] == (1,) or kk.get("dim") == 1):
                rec.append(x.detach().float().cpu())
            return pmax(x, *aa, **kk)
        def m2(t, *aa, **kk):
            if t.dim() == 2 and t.is_floating_point() and (aa[:1] == (1,) or kk.get("dim") == 1):
                rec.append(t.detach().float().cpu())
            return ptmax(t, *aa, **kk)
        DEPTH[0] += 1
        torch.max, torch.Tensor.max = m1, m2
        try:
            res = orig(self, *a, **k)
        finally:
            torch.max, torch.Tensor.max = pmax, ptmax
            DEPTH[0] -= 1
        try:
            lg = torch.cat(rec)[-len(Y):] if rec else None
            if lg is not None and len(lg) == len(Y) and lg.shape[1] == C_:
                PER.append(dict(round=ROUND[0], post=FM.log_view(FM.class_metrics(lg, Y, C_))))
        except Exception as e:
            print("per-class evaluation not recorded:", repr(e))
        return res
    f._fedecs = True
    setattr(cls, name, f)
import importlib, pkgutil, flcore.servers as fs
for m in pkgutil.iter_modules(fs.__path__):
    mod = importlib.import_module("flcore.servers." + m.name)
    for v in list(vars(mod).values()):
        if isinstance(v, type) and issubclass(v, sb.Server):
            _wrap(v)
            for _n in ("compute_global_test_accuracy", "_evaluate_global", "_globaltest"):
                if _n in v.__dict__:
                    _wrap_eval(v, _n)
            if "aggregate_parameters" in v.__dict__:
                _wrap_round(v)
sys.argv = ["main.py"] + args
t0 = time.time()
runpy.run_path("main.py", run_name="__main__")
srv = CAP["server"]
# Fed-GraB keeps its trained classifier in srv.g_classifier and evaluates with it (_globaltest); the global model's
# own head is never trained there, so the generic evaluation below would score an untrained head (27 Oct 2026:
# 1-13% instead of the library's own 28-51%). Copy the trained classifier into the head before evaluating.
if hasattr(srv, "g_classifier") and hasattr(srv.global_model, "head"):
    srv.global_model.head.load_state_dict(srv.g_classifier.state_dict())
# re-run the library's own global evaluation while recording the scores it ranks
REC, ACC = [], None
_max, _tmax = torch.max, torch.Tensor.max
def rmax(x, *a, **k):
    if torch.is_tensor(x) and x.dim() == 2 and x.is_floating_point() and (a[:1] == (1,) or k.get("dim") == 1):
        REC.append(x.detach().float().cpu())
    return _max(x, *a, **k)
def rtmax(self, *a, **k):
    if self.dim() == 2 and self.is_floating_point() and (a[:1] == (1,) or k.get("dim") == 1):
        REC.append(self.detach().float().cpu())
    return _tmax(self, *a, **k)
torch.max, torch.Tensor.max = rmax, rtmax
with torch.no_grad():
    if hasattr(srv, "_evaluate_global"):
        res = srv._evaluate_global()
    else:
        res = srv.compute_global_test_accuracy()
torch.max, torch.Tensor.max = _max, _tmax
lib_acc = res[0] if isinstance(res, tuple) else res
n_test = len(srv.global_testloader.dataset)
logits = torch.cat(REC)[-n_test:] if REC else None
def _num_list(v):
    """The library's per-round history lists (rs_*) that hold numbers; some methods (Fed-GraB) also keep lists of
    dicts there, which are skipped (the per-class metrics above cover every evaluation)."""
    out = []
    for x in v:
        if torch.is_tensor(x) and x.numel() == 1:
            x = x.item()
        if not isinstance(x, (int, float, np.integer, np.floating)) or isinstance(x, bool):
            return None
        out.append(float(x))
    return out
hist = {}
for k in dir(srv):
    if k.startswith("rs_") and isinstance(getattr(srv, k), list):
        _l = _num_list(getattr(srv, k))
        if _l is not None:
            hist[k] = _l
emb = None
try:
    srv.global_model.eval()
    feats = []
    with torch.no_grad():
        for i, (xb, yb) in enumerate(srv.global_testloader):
            o = srv.global_model(xb.to(srv.device))
            if isinstance(o, tuple):
                feats.append(o[0].flatten(1).float().cpu())
            if sum(len(f) for f in feats) >= 2000:
                break
    emb = torch.cat(feats)[:2000].numpy() if feats else None
except Exception as e:
    print("embeddings not available:", repr(e))
np.savez_compressed(out, logits=logits.numpy() if logits is not None else np.zeros(0), lib_acc=np.array(
    [lib_acc if lib_acc is not None else np.nan]), time_s=np.array([time.time() - t0]),
    embed=emb if emb is not None else np.zeros(0),
    gpu_peak_mb=np.array([torch.cuda.max_memory_allocated() / 2 ** 20 if torch.cuda.is_available() else 0.0]))
json.dump(hist, open(out + ".hist.json", "w"))
json.dump(PER, open(out + ".perclass.json", "w"))
'''


def run(algo, ds, IF, seed, lt_dir, out_dir, rounds=200, label=None, data_root=None, ds_key=None):
    """One LongTailLib run on our split; writes a suite-format record into out_dir/runs.jsonl."""
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from class_names import CLASS_NAMES
    from metrics import PER_CLASS, class_metrics, summarize
    import torch
    C = 100 if ds == "cifar100" else 10
    name = folder(ds, IF, seed)
    sysdir = os.path.join(lt_dir, "system")
    drv = os.path.join(sysdir, "_suite_driver.py")
    open(drv, "w").write(DRIVER)
    os.makedirs(os.path.join(out_dir, "predictions"), exist_ok=True)
    label = label or algo
    ds_key = ds_key or ds
    rid = f"{label}|{ds_key}|IF{IF}|a{ALPHA2}|K{K2}|s{seed}|p{JR2}"
    safe = "".join(ch if ch.isalnum() or ch in "-_.=" else "_" for ch in rid)
    tmp = os.path.join(out_dir, "predictions", safe + "_lib.npz")
    args = ["-data", name, "-algo", ALGOS[algo], "-m", "ResNet8", "-gr", str(rounds), "-ncl", str(C), "-nc", str(K2),
            "-jr", str(JR2), "-seed", str(seed), "-did", "0", "-dev", "cuda" if torch.cuda.is_available() else "cpu"]
    # evaluate every 5th round (LongTailLib's default is every round): training is unchanged, the final model is
    # evaluated by the driver below; saves about 15% of a run
    args += ["-eg", "5"]
    args += os.environ.get("FEDECS_LT_TEST_ARGS", "").split()   # CPU tests only (e.g. "-ls 1"); unset on Kaggle
    t0 = time.time()
    os.makedirs(os.path.join(out_dir, "logs"), exist_ok=True)
    log = open(os.path.join(out_dir, "logs", safe + "_longtaillib.log"), "w")
    p = subprocess.run([sys.executable, drv, json.dumps(args), tmp], cwd=sysdir, stdout=log, stderr=subprocess.STDOUT,
                       env=dict(os.environ, FEDECS_SRC=os.path.dirname(os.path.abspath(__file__))))
    log.close()
    if p.returncode != 0:
        raise RuntimeError(f"LongTailLib {algo} failed (see logs/{safe}_longtaillib.log)")
    d = np.load(tmp)
    root = os.path.join(lt_dir, "dataset", name)
    y = np.load(os.path.join(root, "global_test.npz"), allow_pickle=True)["data"].item()["y"]
    logits = torch.from_numpy(d["logits"]).float()
    m = class_metrics(logits, torch.from_numpy(y).long(), C, keep_probs=True)
    probs = m.pop("probs")
    lib_acc = float(d["lib_acc"][0])
    summ = summarize({0: m})
    rec = dict(run_id=rid, label=label, dataset=ds_key, IF=IF, alpha=ALPHA2,
               K=K2, seed=seed, pfrac=JR2, tail_perm=False, track=2, framework="LongTailLib (authors' code)",
               method=dict(pipeline="longtaillib", algo=ALGOS[algo], model="ResNet8", commit=LT_COMMIT,
                           source="LongTailLib (FedLTLib benchmark), authors' code unchanged, downloaded at run time"),
               client_counts=np.load(os.path.join(root, "suite_split.npz"))["counts"].tolist(),
               class_order=list(range(C)), class_names=CLASS_NAMES.get(ds),
               test_class_counts=np.bincount(y, minlength=C).tolist(),
               fedgrab_eval_fixed=True,   # Fed-GraB scored with its trained classifier (see DRIVER)
               library_accuracy=lib_acc, accuracy_check=abs(m["acc"] - lib_acc) < 1e-6 if lib_acc == lib_acc else None,
               time_s=time.time() - t0, rounds=dict(T=rounds))
    # FLOPs of the ResNet-8 forward pass per image (this suite's ResNet-8, the architecture of LongTailLib's) and the
    # expected training FLOPs of one round (3 x forward; the library's local epochs over its batches of 32; JR2 of
    # the clients per round)
    from flops import forward_flops
    from models import ResNet8FE, ResNet8Head
    net = torch.nn.Sequential(ResNet8FE(3), ResNet8Head(128, C))
    f_fwd = forward_flops(net, torch.zeros(2, 3, 32, 32))
    ls = int(args[args.index("-ls") + 1]) if "-ls" in args else 10
    n_k = np.array(rec["client_counts"]).sum(1)
    rec.update(flops_model=f_fwd,
               train_gflops_per_round=JR2 * float(sum(3 * f_fwd * ls * (n // 32) * 32 for n in n_k)) / 1e9)
    if "gpu_peak_mb" in d.files and d["gpu_peak_mb"][0] > 0:   # the library's process (a fresh one per run)
        rec["gpu_peak_mb"] = round(float(d["gpu_peak_mb"][0]))
    for k, v in summ.items():
        if isinstance(v, (int, float)):
            rec[f"final_{k}"] = v
        elif k in PER_CLASS:
            rec[f"final_{k}"] = [round(float(x), 4) for x in v]
        elif k == "confusion":
            rec["final_confusion"] = v
    hist = json.load(open(tmp + ".hist.json"))
    rec["history_library"] = hist
    # every evaluation of the library (every 5th round, and the final model): every metric for every class, as the
    # suite's per-round logs (event "round_eval", round = rounds of training completed)
    per = json.load(open(tmp + ".perclass.json")) if os.path.exists(tmp + ".perclass.json") else []
    with open(os.path.join(out_dir, "logs", safe + ".jsonl"), "w") as f:
        start = {k: rec[k] for k in ("run_id", "label", "dataset", "IF", "alpha", "K", "seed", "pfrac", "tail_perm",
                                     "framework", "method", "rounds", "class_order", "class_names",
                                     "test_class_counts") if k in rec}   # the same configuration as every run_start
        f.write(json.dumps(dict(t=time.time(), event="run_start", **start)) + "\n")
        for e in per:
            f.write(json.dumps(dict(t=time.time(), event="round_eval", round=e["round"], post=e["post"])) + "\n")
        f.write(json.dumps(dict(t=time.time(), event="run_end", time_s=rec["time_s"])) + "\n")
    rec["evaluations_logged"] = len(per)
    np.savez_compressed(os.path.join(out_dir, "predictions", safe + ".npz"), y_full=y, probs_0=probs.astype(np.float16),
                        **({"embed_0": d["embed"].astype(np.float16), "y": y[:len(d["embed"])]}
                           if d["embed"].size else {}))
    for f_ in (tmp, tmp + ".hist.json", tmp + ".perclass.json"):   # temporary files of the driver
        if os.path.exists(f_):
            os.remove(f_)
    with open(os.path.join(out_dir, "runs.jsonl"), "a") as f:
        f.write(json.dumps(rec) + "\n")
    return rec


def run_for_suite(R):
    """Runner entry: R["ds"] is a Track-2 dataset key (cifar10_t2 / cifar100_t2). Fetches the library and prepares the
    split once per notebook (lock file), then runs the method."""
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    from settings import DATASETS
    raw = DATASETS[R["ds"]]["raw"]
    lt_dir = os.path.join(R["cache_dir"], "LongTailLib")
    lock = lt_dir + ".lock"
    while True:
        try:
            os.close(os.open(lock, os.O_CREAT | os.O_EXCL))
            break
        except FileExistsError:
            time.sleep(5)
    try:
        fetch(lt_dir)
        if not os.path.exists(os.path.join(lt_dir, "dataset", folder(raw, R["IF"], R["seed"]), "global_test.npz")):
            prepare(raw, R["IF"], R["seed"], lt_dir, R["data_root"])
    finally:
        os.remove(lock)
    rounds = (R.get("rounds") or {}).get("T", 200)
    return run(R["method"]["name"], raw, R["IF"], R["seed"], lt_dir, R["out_dir"], rounds=rounds, label=R["label"],
               ds_key=R["ds"])


if __name__ == "__main__":
    cmd = sys.argv[1]
    if cmd == "prepare":
        ds, IF, seed, lt_dir, data_root = sys.argv[2], float(sys.argv[3]), int(sys.argv[4]), sys.argv[5], sys.argv[6]
        fetch(lt_dir)
        print(prepare(ds, IF, seed, lt_dir, data_root)[0])
    elif cmd == "run":
        algo, ds, IF, seed, lt_dir, out = sys.argv[2], sys.argv[3], float(sys.argv[4]), int(sys.argv[5]), sys.argv[6], \
            sys.argv[7]
        rounds = int(sys.argv[8]) if len(sys.argv) > 8 else 200
        r = run(algo, ds, IF, seed, lt_dir, out, rounds)
        print(r["run_id"], "bal", round(r["final_bal"], 4), "library acc", r["library_accuracy"], "check",
              r["accuracy_check"])


In [ ]:
# Run list and code version
import json, os, time
os.makedirs('/kaggle/working/fedecs_lt', exist_ok=True)
RUNS = json.loads(r"""[
 {
  "label": "FedGH",
  "ds": "gtsrb",
  "seed": 3,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "LG-FedAvg",
  "ds": "gtsrb",
  "seed": 3,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "FedMRL",
  "ds": "gtsrb",
  "seed": 3,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "FedGH",
  "ds": "gtsrb",
  "seed": 4,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "LG-FedAvg",
  "ds": "gtsrb",
  "seed": 4,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "FedMRL",
  "ds": "gtsrb",
  "seed": 4,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "FedGH",
  "ds": "gtsrb",
  "seed": 5,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "LG-FedAvg",
  "ds": "gtsrb",
  "seed": 5,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 },
 {
  "label": "FedMRL",
  "ds": "gtsrb",
  "seed": 5,
  "IF": null,
  "alpha": 0.5,
  "K": 10,
  "data_root": "/tmp/fedecs/data",
  "out_dir": "/tmp/fedecs/work/results",
  "cache_dir": "/tmp/fedecs/cache",
  "models_dir": "/tmp/fedecs/work/models",
  "tensorboard": true
 }
]""")
for d in ('/tmp/fedecs/data', '/tmp/fedecs/cache', '/tmp/fedecs/work/results', '/tmp/fedecs/work/models'):
    os.makedirs(d, exist_ok=True)
json.dump(RUNS, open('/tmp/fedecs/work/runs.json', 'w'))
open('/tmp/fedecs/work/results/code_version.txt', 'w').write('da5efa7b5089 / sources sha256 b6e96f10a3ad230b')
print(len(RUNS), 'runs')

In [ ]:
# Run everything: 2 GPUs x 2 run(s) per GPU; resumable; no new run after 11 h
import subprocess, os, sys, time
env = dict(os.environ, FEDECS_DEADLINE=str(time.time() + 11 * 3600), FEDECS_PACK_OUT='/kaggle/working',
           FEDECS_NAME='NB10_main_gtsrb_part3of3', FEDECS_CODE_VERSION=open('/tmp/fedecs/work/results/code_version.txt').read(),
           PYTHONPATH='/kaggle/working/fedecs_lt')
p = subprocess.Popen([sys.executable, '/kaggle/working/fedecs_lt/runner.py', '--runs', '/tmp/fedecs/work/runs.json',
                      '--gpus', '2', '--per_gpu', '2', '--log_dir', '/tmp/fedecs/work/results/logs'], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, cwd='/kaggle/working/fedecs_lt')
for line in p.stdout:
    print(line, end='')
p.wait()
print('finished with code', p.returncode)

In [ ]:
# Report and the three output files
import subprocess, sys, os
r = subprocess.run([sys.executable, '/kaggle/working/fedecs_lt/pack.py', '/tmp/fedecs/work', 'NB10_main_gtsrb_part3of3',
                    'Main comparison on GTSRB (natural imbalance), 10 clients, Dirichlet 0.5 (part 3 of 3)', '/kaggle/working'], cwd='/kaggle/working/fedecs_lt',
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
for f in sorted(os.listdir('/kaggle/working')):
    if f.startswith('NB10_main_gtsrb_part3of3'):
        print(f, round(os.path.getsize('/kaggle/working/' + f) / 2**20, 1), 'MB')